### 1. Environment Setup and Existing Dataset Verification

This section establishes the Google Colab environment for implementing an RF-DETR-based pedestrian detection system under low-light conditions. The experiment initially investigates the existing pedestrian dataset used in previous object detection activities to determine whether its images and bounding-box annotations can be reused for RF-DETR training and evaluation.

The dataset discovery process identifies image files, annotation formats, existing dataset configurations, and the available GPU resources. The findings will guide subsequent conversion to COCO format, preparation of independent training and evaluation partitions, and integration with the official RF-DETR implementation.

In [ ]:
# Import Path for handling folders and file locations.
# This allows the notebook to identify existing dataset resources without changing them.
from pathlib import Path

# Import operating system utilities for inspecting dataset directories.
# Directory traversal will be used to locate relevant images, annotation files, and configuration files.
import os

# Import pandas for checking existing dataset records.
# These records may identify images and annotation files from previous object detection activities.
import pandas as pd

# Import PyTorch for checking GPU availability.
# RF-DETR fine-tuning is computationally demanding, so a CUDA-capable GPU is strongly preferred.
import torch

# Import the Google Drive utility for accessing previously saved project files.
# The new experiment will reuse the original dataset rather than recreating it unnecessarily.
from google.colab import drive

# Mount Google Drive in the Colab runtime.
# This provides access to the existing Person_Detection project and its saved annotations.
drive.mount("/content/drive")

# Define the main Google Drive directory used for the previous pedestrian detection activities.
# The RF-DETR notebook will inspect this directory for reusable data and results.
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Person_Detection"
)

# Confirm that the previous project directory exists.
# If the directory has moved, the location must be corrected before proceeding.
assert PROJECT_DIR.is_dir(), (
    f"Previous project folder not found: {PROJECT_DIR}"
)

# Create a separate project directory for the RF-DETR final project.
# All new files will be saved here without overwriting previous YOLOv11 experiments.
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"

# Define dedicated folders for dataset preparation, model training, evaluation, and reporting.
# Keeping these outputs separate will make final submission and GitHub organization easier.
DATASET_DIR = RFDETR_DIR / "dataset"
MODEL_DIR = RFDETR_DIR / "trained_models"
RESULTS_DIR = RFDETR_DIR / "evaluation_results"
FIGURES_DIR = RFDETR_DIR / "figures"
REPORT_DIR = RFDETR_DIR / "final_report"
REPOSITORY_DIR = RFDETR_DIR / "github_repository"

# Create the working directories required for the new project.
# Existing data in the Person_Detection project will remain unchanged.
for folder in [
    DATASET_DIR,
    MODEL_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    REPORT_DIR,
    REPOSITORY_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Select a CUDA device when one is available.
# Training should not begin until the available hardware and its memory have been inspected.
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Display the available PyTorch and GPU information.
# This will help us select an appropriate RF-DETR model size and training batch size.
print("RF-DETR ENVIRONMENT VERIFICATION")
print("=" * 65)
print("PyTorch Version:", torch.__version__)
print("Computing Device:", DEVICE)
print("CUDA Available:", torch.cuda.is_available())

# Report additional GPU information when a CUDA device is present.
# GPU memory is especially important for transformer-based object detection training.
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        )
    )
else:
    print("GPU acceleration is unavailable.")
    print("Enable a GPU under Runtime > Change runtime type.")

# Define the most relevant locations from the previous pedestrian detection project.
# These folders may contain original annotations, corrected labels, or a previously exported dataset.
candidate_directories = [
    PROJECT_DIR,
    PROJECT_DIR / "Second_Model_YOLO11",
    PROJECT_DIR / "CO1_3_Ground_Truth_Validation",
    PROJECT_DIR / "CO1_3_Ground_Truth_Validation" / "evaluation_results",
    PROJECT_DIR / "preprocessed_images"
]

# Display which candidate directories are present.
# This helps identify where the original and processed pedestrian images are stored.
print("\nEXISTING PROJECT DIRECTORIES")
print("=" * 65)

for folder in candidate_directories:
    print(
        "[FOUND]" if folder.is_dir() else "[MISSING]",
        folder
    )

# Define the previously used CO1.3 ground truth review inventory.
# The inventory may provide image paths and bounding-box label locations for dataset reconstruction.
INVENTORY_CSV = (
    PROJECT_DIR
    / "CO1_3_Ground_Truth_Validation"
    / "ground_truth_review_inventory.csv"
)

# Define the corresponding manually verified image records.
# These records can identify the lighting conditions and source files previously examined.
REVIEW_CSV = (
    PROJECT_DIR
    / "CO1_3_Ground_Truth_Validation"
    / "ground_truth_manual_verification.csv"
)

# Load the available review inventory without changing the source files.
# The output will reveal whether image and annotation paths are already documented.
if INVENTORY_CSV.is_file():
    inventory = pd.read_csv(
        INVENTORY_CSV,
        keep_default_na=False
    )

    print("\nPREVIOUS ANNOTATION INVENTORY")
    print("=" * 65)
    print("Inventory Records:", len(inventory))
    print("Available Columns:", inventory.columns.tolist())

    # Print the number of accessible files referenced by important path columns.
    # This distinguishes documented paths from source files that are actually still available.
    for column in [
        "Image_Path",
        "Original_Label_Path",
        "Working_Label_Path"
    ]:
        if column in inventory.columns:
            available = inventory[column].astype(str).apply(
                lambda path: bool(path.strip()) and Path(path).is_file()
            ).sum()

            print(
                f"{column}: {available}/{len(inventory)} files found"
            )
else:
    print("\nNo previous annotation inventory was found.")

# Check whether the manually verified image record is available.
# This provides another source of documented image locations from the earlier experiments.
if REVIEW_CSV.is_file():
    reviews = pd.read_csv(
        REVIEW_CSV,
        keep_default_na=False
    )

    print("\nPREVIOUS IMAGE REVIEW RECORDS")
    print("=" * 65)
    print("Review Records:", len(reviews))
    print("Available Columns:", reviews.columns.tolist())

    # Display lighting-condition counts when the corresponding column exists.
    # These counts help distinguish lighting variants from independent source photographs.
    if "Condition" in reviews.columns:
        print(reviews["Condition"].value_counts().to_string())
else:
    print("\nNo previous image review CSV was found.")

# Define the image and annotation filename extensions to search for.
# COCO JSON, YOLO TXT, and YAML configuration files are all potentially relevant.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
ANNOTATION_EXTENSIONS = {".json", ".yaml", ".yml"}

# Prepare a list of dataset-related files discovered within the original project.
# The new RF-DETR output folder is excluded to avoid counting newly generated files.
discovered_files = []

# Walk through the previous project directory to locate existing dataset resources.
# Only file locations and extensions are collected at this stage, not the image contents.
for root, directories, filenames in os.walk(PROJECT_DIR):

    # Exclude the newly created RF-DETR project directory.
    # This ensures that dataset discovery examines only the previous experiment's resources.
    directories[:] = [
        name
        for name in directories
        if Path(root, name) != RFDETR_DIR
    ]

    # Inspect each filename for supported image or dataset annotation formats.
    # The collected records will be used to identify a suitable source dataset.
    for filename in filenames:
        file_path = Path(root) / filename
        suffix = file_path.suffix.lower()

        if suffix in IMAGE_EXTENSIONS:
            file_type = "Image"
        elif suffix == ".txt":
            file_type = "Text/YOLO Candidate"
        elif suffix in ANNOTATION_EXTENSIONS:
            file_type = "Annotation/Config Candidate"
        else:
            continue

        # Store each discovered file's type and location.
        # The original files will not be copied or modified during this inspection.
        discovered_files.append({
            "File_Type": file_type,
            "File_Name": filename,
            "File_Path": str(file_path),
            "Parent_Folder": str(file_path.parent)
        })

# Convert the discovered file records into a structured table.
# This allows image and annotation counts to be summarized without manually browsing Drive.
discovery_df = pd.DataFrame(
    discovered_files,
    columns=[
        "File_Type",
        "File_Name",
        "File_Path",
        "Parent_Folder"
    ]
)

# Display the total number of discovered images and candidate annotation files.
# These counts will help us determine whether the existing dataset is suitable for training.
print("\nDATASET RESOURCE DISCOVERY")
print("=" * 65)
print("Total Relevant Files:", len(discovery_df))

if not discovery_df.empty:
    print(
        discovery_df["File_Type"]
        .value_counts()
        .to_string()
    )

    # Display folders containing the largest number of image files.
    # This helps identify complete original datasets instead of small preview-image folders.
    image_folders = (
        discovery_df[
            discovery_df["File_Type"] == "Image"
        ]
        .groupby("Parent_Folder")
        .size()
        .sort_values(ascending=False)
        .head(12)
    )

    print("\nTOP IMAGE FOLDERS")
    print("=" * 65)
    print(image_folders.to_string())

    # Display relevant COCO and dataset configuration candidates.
    # These filenames may already describe training, validation, and test splits.
    configurations = discovery_df[
        discovery_df["File_Name"].str.lower().str.contains(
            r"coco|data\.ya?ml|dataset\.ya?ml|annotations",
            regex=True,
            na=False
        )
    ].head(25)

    print("\nCOCO / DATASET CONFIGURATION CANDIDATES")
    print("=" * 65)

    if configurations.empty:
        print("No matching configuration filenames found.")
    else:
        for path in configurations["File_Path"]:
            print(path)

# Save the complete discovery inventory for the final project documentation.
# This record will help trace which original files were available before preprocessing.
DISCOVERY_CSV = (
    DATASET_DIR
    / "cell1_existing_dataset_discovery.csv"
)

discovery_df.to_csv(
    DISCOVERY_CSV,
    index=False
)

# Display the new project location and the saved discovery file.
# The next cell will use these results to identify suitable annotated images for COCO preparation.
print("\nRF-DETR PROJECT LOCATIONS")
print("=" * 65)
print("Project Directory:", RFDETR_DIR)
print("Dataset Directory:", DATASET_DIR)
print("Discovery Inventory:", DISCOVERY_CSV)

# Confirm that environment initialization and dataset discovery completed.
# No images or annotations have been altered during this first step.
print("\nCELL 1 COMPLETED SUCCESSFULLY!")

### 2. Dataset Selection and Bounding-Box Annotation Verification

This section evaluates the suitability of the existing low-light pedestrian dataset for the RF-DETR object detection experiment. The previously reviewed ground-truth annotations are examined to identify valid bounding boxes, confirm image-label correspondence, and determine whether corrected annotations are available for images that required manual verification.

Each annotation is checked for valid YOLO-format class identifiers and normalized bounding-box coordinates. The verification process also identifies the original source frames and lighting conditions, allowing the subsequent dataset preparation stage to prevent related image variants from being distributed across training, validation, and testing partitions.

The results are saved as a verified dataset manifest and an annotation quality report. These records provide an auditable foundation for converting the annotations to COCO format and training the RF-DETR model.

In [ ]:

# Import the libraries required for image and bounding-box validation.
# This cell inspects existing files and records their condition without modifying the original annotations.
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Define the previous pedestrian detection project and the new RF-DETR output folders.
# These paths follow the directory structure established in Cell 1.
PROJECT_DIR = Path("/content/drive/MyDrive/Person_Detection")
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"
DATASET_DIR = RFDETR_DIR / "dataset"
FIGURES_DIR = RFDETR_DIR / "figures"

# Create the new output directories when necessary.
# Dataset audit records and example figures will be saved separately from the original YOLO files.
DATASET_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Identify the manually reviewed annotation inventory from CO1.3.
# This file records original and working label paths associated with the evaluated images.
GT_DIR = PROJECT_DIR / "CO1_3_Ground_Truth_Validation"

INVENTORY_CSV = GT_DIR / "ground_truth_review_inventory.csv"
REVIEW_CSV = GT_DIR / "ground_truth_manual_verification.csv"

# Confirm that both required source records exist before selecting any annotations.
# We will not invent ground-truth paths or silently substitute model predictions for labels.
assert INVENTORY_CSV.is_file(), (
    f"Missing annotation inventory: {INVENTORY_CSV}"
)
assert REVIEW_CSV.is_file(), (
    f"Missing manual review records: {REVIEW_CSV}"
)

# Load the two CO1.3 records without interpreting empty paths as numerical values.
# This preserves filenames and status values needed to select the correct bounding-box labels.
inventory = pd.read_csv(INVENTORY_CSV, keep_default_na=False)
reviews = pd.read_csv(REVIEW_CSV, keep_default_na=False)

# Display the column structures before combining the existing records.
# These outputs help identify differences if an earlier notebook saved a different CSV version.
print("ORIGINAL DATASET RECORDS")
print("=" * 65)
print("Inventory records:", len(inventory))
print("Review records:", len(reviews))
print("Inventory columns:", inventory.columns.tolist())
print("Review columns:", reviews.columns.tolist())

# Define the required fields for the annotation inventory and review records.
# Every image needs an original label path, a working label path, and a final review status.
inventory_columns = [
    "Condition",
    "Image",
    "Image_Path",
    "Original_Label_Path",
    "Working_Label_Path"
]

review_columns = [
    "Condition",
    "Image",
    "Review_Status"
]

# Verify the minimum required columns before performing the merge.
# An unexpected CSV schema must be corrected rather than guessed.
for column in inventory_columns:
    assert column in inventory.columns, (
        f"Missing inventory column: {column}"
    )

for column in review_columns:
    assert column in reviews.columns, (
        f"Missing review column: {column}"
    )

# Remove previously merged review-status columns from the inventory if present.
# This prevents pandas from creating Review_Status_x and Review_Status_y instead of one reliable status field.
inventory_clean = inventory.drop(
    columns=[
        "Review_Status",
        "Review_Status_x",
        "Review_Status_y"
    ],
    errors="ignore"
).copy()

# Normalize the image and lighting keys used to combine the two records.
# Consistent key formatting avoids accidental mismatches caused by extra spaces or capitalization.
for frame in [inventory_clean, reviews]:
    frame["Condition"] = (
        frame["Condition"].astype(str).str.strip().str.lower()
    )
    frame["Image"] = (
        frame["Image"].astype(str).str.strip()
    )

# Verify that each image and lighting combination occurs once in each source file.
# Duplicate records would make it unclear which annotation or manual review should be trusted.
key_columns = ["Condition", "Image"]

assert not inventory_clean.duplicated(key_columns).any(), (
    "Duplicate image records found in the annotation inventory."
)
assert not reviews.duplicated(key_columns).any(), (
    "Duplicate image records found in the manual review CSV."
)

# Merge only the final review status into the annotation inventory.
# Each annotation selection will now use the manually recorded verification decision.
manifest = inventory_clean.merge(
    reviews[key_columns + ["Review_Status"]],
    on=key_columns,
    how="left",
    validate="one_to_one",
    indicator=True
)

# Confirm that every image was matched with a review record.
# Images lacking a review decision cannot be treated as verified ground truth automatically.
assert manifest["_merge"].eq("both").all(), (
    "Some annotation inventory images have no matching review record."
)

manifest = manifest.drop(columns=["_merge"])

# Standardize the review-status values used for selecting annotations.
# Verified images retain their original label files, while corrected images require saved working labels.
manifest["Review_Status"] = (
    manifest["Review_Status"].astype(str).str.strip()
)

expected_statuses = {"Verified", "Needs Correction"}

assert set(manifest["Review_Status"]).issubset(
    expected_statuses
), (
    "Unexpected review status values: "
    f"{set(manifest['Review_Status'])}"
)

# Select the annotation path appropriate for each reviewed image.
# A label originally marked for correction must use its working label file rather than its uncorrected original.
def select_annotation_path(row):
    if row["Review_Status"] == "Verified":
        return str(row["Original_Label_Path"]).strip()

    return str(row["Working_Label_Path"]).strip()

# Add the selected annotation path to the final manifest.
# The original inventory columns are preserved for traceability.
manifest["Selected_Label_Path"] = manifest.apply(
    select_annotation_path,
    axis=1
)

# Check whether the selected images and label files are present in Google Drive.
# This distinguishes a documented annotation path from an annotation file that can actually be read.
manifest["Image_Exists"] = manifest["Image_Path"].apply(
    lambda value: Path(str(value)).is_file()
)

manifest["Label_Exists"] = manifest["Selected_Label_Path"].apply(
    lambda value: bool(str(value).strip())
    and Path(str(value)).is_file()
)

# Extract a source-frame key by removing only the lighting suffix from the filename.
# Variants of the same original frame should share this key when we later create dataset splits.
def extract_source_frame(image_name, condition):
    stem = Path(str(image_name)).stem
    suffix = "_" + str(condition).lower()

    if not stem.lower().endswith(suffix):
        return ""

    return stem[:-len(suffix)]

# Create the source-frame group identifier for every reviewed image.
# Missing identifiers will be reported because they could lead to data leakage during splitting.
manifest["Source_Frame"] = manifest.apply(
    lambda row: extract_source_frame(
        row["Image"],
        row["Condition"]
    ),
    axis=1
)

# Define a validator for YOLO bounding-box annotation files.
# The accepted format is class_id center_x center_y width height with normalized coordinates.
def validate_yolo_label(label_path):

    # Return an explicit failure when the requested label file is unavailable.
    # Empty annotation files are allowed only when an existing readable file contains no objects.
    path = Path(str(label_path))

    if not path.is_file():
        return {
            "Valid": False,
            "Boxes": 0,
            "Issues": "Label file missing"
        }

    # Read the annotation lines as text.
    # Each nonempty line is expected to describe one bounding box.
    try:
        lines = path.read_text(
            encoding="utf-8-sig"
        ).splitlines()
    except Exception as error:
        return {
            "Valid": False,
            "Boxes": 0,
            "Issues": f"Unreadable label: {error}"
        }

    # Initialize the annotation count and any formatting or coordinate issues.
    # Invalid entries must be reported rather than silently discarded.
    box_count = 0
    issues = []

    # Inspect each annotation line using the standard five-column YOLO format.
    # This project expects only the pedestrian class, represented by class identifier zero.
    for line_number, line in enumerate(lines, start=1):
        stripped = line.strip()

        if not stripped:
            continue

        parts = stripped.split()

        if len(parts) != 5:
            issues.append(
                f"Line {line_number}: expected 5 values, found {len(parts)}"
            )
            continue

        # Convert the class identifier and normalized bounding-box coordinates to numbers.
        # NaN, infinity, and nonnumeric values are invalid object annotations.
        try:
            values = [float(value) for value in parts]
        except ValueError:
            issues.append(
                f"Line {line_number}: nonnumeric annotation"
            )
            continue

        if not np.isfinite(values).all():
            issues.append(
                f"Line {line_number}: non-finite annotation"
            )
            continue

        class_id, cx, cy, width, height = values

        # Confirm that the object belongs to the single pedestrian class.
        # This prevents unrelated vehicle or object classes from entering the final project.
        if class_id != 0:
            issues.append(
                f"Line {line_number}: unexpected class ID {class_id}"
            )
            continue

        # Validate the center point and the bounding-box dimensions.
        # A valid normalized YOLO box must have positive width and height.
        if not (
            0 <= cx <= 1
            and 0 <= cy <= 1
            and 0 < width <= 1
            and 0 < height <= 1
        ):
            issues.append(
                f"Line {line_number}: invalid normalized coordinates"
            )
            continue

        # Confirm that the bounding box stays within normalized image boundaries.
        # Out-of-image boxes need review before conversion into COCO pixel coordinates.
        tolerance = 1e-5

        if (
            cx - width / 2 < -tolerance
            or cy - height / 2 < -tolerance
            or cx + width / 2 > 1 + tolerance
            or cy + height / 2 > 1 + tolerance
        ):
            issues.append(
                f"Line {line_number}: bounding box exceeds image bounds"
            )
            continue

        # Count only fully valid pedestrian annotations.
        # The count will later be compared with the number of COCO annotations produced.
        box_count += 1

    # Return the final annotation validity and any reasons for failure.
    # A complete audit makes it easier to diagnose individual files before model training.
    return {
        "Valid": len(issues) == 0,
        "Boxes": box_count,
        "Issues": " | ".join(issues)
    }

# Define a small image reader that verifies image dimensions without modifying the image.
# COCO bounding boxes will later require the width and height of every actual image file.
def inspect_image(image_path):
    path = Path(str(image_path))

    if not path.is_file():
        return {
            "Valid_Image": False,
            "Image_Width": 0,
            "Image_Height": 0
        }

    # Load the image using OpenCV and confirm that it can be decoded.
    # A damaged file must be excluded or repaired before RF-DETR dataset preparation.
    image = cv2.imread(str(path))

    if image is None:
        return {
            "Valid_Image": False,
            "Image_Width": 0,
            "Image_Height": 0
        }

    # Record the actual image dimensions for later annotation conversion.
    # COCO stores bounding boxes in pixel coordinates rather than normalized YOLO coordinates.
    height, width = image.shape[:2]

    return {
        "Valid_Image": width > 0 and height > 0,
        "Image_Width": int(width),
        "Image_Height": int(height)
    }

# Verify every selected annotation file and collect the validation results.
# This operation reads files only and does not alter the manual CO1.3 corrections.
print("\nVALIDATING PEDESTRIAN ANNOTATIONS")
print("=" * 65)

annotation_results = []

for index, row in manifest.iterrows():

    # Validate the annotation format and normalized bounding-box geometry.
    # Failed label files remain in the audit table with their corresponding issue descriptions.
    label_result = validate_yolo_label(
        row["Selected_Label_Path"]
    )

    # Validate the corresponding image and record its actual pixel dimensions.
    # Image dimensions are required for the conversion to COCO bounding-box coordinates.
    image_result = inspect_image(
        row["Image_Path"]
    )

    # Combine the image and annotation audit information for the current record.
    # Both validations must succeed before an image can be used for COCO conversion.
    annotation_results.append({
        **label_result,
        **image_result
    })

    # Display occasional progress updates while processing the images.
    # This helps confirm that the verification is continuing in a large Google Drive directory.
    if (index + 1) % 100 == 0:
        print(
            f"Validated {index + 1}/{len(manifest)} image records"
        )

# Merge the audit results into the verified dataset manifest.
# Every original image retains its lighting label and selected annotation path.
audit_df = pd.concat(
    [
        manifest.reset_index(drop=True),
        pd.DataFrame(annotation_results)
    ],
    axis=1
)

# Determine whether each record is completely suitable for conversion.
# Validity requires a readable image, an existing label, valid boxes, and a source-frame identifier.
audit_df["Ready_For_COCO"] = (
    audit_df["Valid_Image"]
    & audit_df["Label_Exists"]
    & audit_df["Valid"]
    & audit_df["Source_Frame"].ne("")
)

# Save all audit records, including any failures, for the final project documentation.
# We preserve failed rows instead of dropping them so annotation quality can be explained transparently.
AUDIT_CSV = DATASET_DIR / "cell2_annotation_quality_audit.csv"

audit_df.to_csv(
    AUDIT_CSV,
    index=False
)

# Extract the verified records into a separate dataset manifest.
# Only records passing every validation check will be considered for subsequent COCO conversion.
verified_df = audit_df[
    audit_df["Ready_For_COCO"]
].copy()

VERIFIED_CSV = DATASET_DIR / "cell2_verified_dataset_manifest.csv"

verified_df.to_csv(
    VERIFIED_CSV,
    index=False
)

# Display the annotation review and quality-control results.
# The summary distinguishes verified labels, corrected labels, invalid files, and usable bounding boxes.
print("\nRF-DETR DATASET ANNOTATION AUDIT")
print("=" * 65)

print("Total Records:", len(audit_df))
print("Verified Review Status:", int(
    audit_df["Review_Status"].eq("Verified").sum()
))
print("Previously Marked for Correction:", int(
    audit_df["Review_Status"].eq("Needs Correction").sum()
))
print("Existing Images:", int(audit_df["Image_Exists"].sum()))
print("Existing Selected Labels:", int(audit_df["Label_Exists"].sum()))
print("Readable Images:", int(audit_df["Valid_Image"].sum()))
print("Valid YOLO Annotation Files:", int(audit_df["Valid"].sum()))
print("Ready for COCO:", len(verified_df))
print("Not Ready:", int((~audit_df["Ready_For_COCO"]).sum()))
print("Validated Pedestrian Boxes:", int(verified_df["Boxes"].sum()))
print("Unique Source Frames:", verified_df["Source_Frame"].nunique())

# Summarize the number of images and valid pedestrian boxes for each lighting condition.
# This provides information for describing the dataset composition in the final PDF report.
condition_summary = verified_df.groupby(
    "Condition"
).agg(
    Images=("Image", "count"),
    Pedestrian_Boxes=("Boxes", "sum"),
    Unique_Source_Frames=("Source_Frame", "nunique")
).reset_index()

print("\nLIGHTING CONDITION SUMMARY")
print("=" * 65)
print(condition_summary.to_string(index=False))

# Save the class-specific annotation count summary for future reporting.
# Differences between lighting categories can help identify dataset imbalance.
CONDITION_CSV = DATASET_DIR / "cell2_lighting_annotation_summary.csv"
condition_summary.to_csv(CONDITION_CSV, index=False)

# Display records containing annotation or image problems.
# These files must be reviewed before deciding whether to repair or exclude them.
failed_df = audit_df[
    ~audit_df["Ready_For_COCO"]
].copy()

print("\nANNOTATION ISSUES")
print("=" * 65)

if failed_df.empty:
    print("No annotation or image problems were detected.")
else:
    print(
        failed_df[
            [
                "Condition",
                "Image",
                "Label_Exists",
                "Valid_Image",
                "Issues"
            ]
        ].head(25).to_string(index=False)
    )

# Define a visualization function for a valid normalized YOLO annotation file.
# The function converts the stored coordinates into pixel positions for visual inspection.
def display_annotated_image(row, ax):
    image = cv2.imread(str(row["Image_Path"]))

    # Convert the OpenCV BGR image into RGB for Matplotlib.
    # This preserves the correct colors when annotated examples are displayed.
    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    height, width = image.shape[:2]

    # Read each valid pedestrian bounding box and draw it on the source image.
    # The display helps confirm that coordinate validation also corresponds to sensible object locations.
    with open(row["Selected_Label_Path"], "r", encoding="utf-8-sig") as file:
        for line in file:
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            _, cx, cy, bw, bh = map(float, parts)

            x1 = int((cx - bw / 2) * width)
            y1 = int((cy - bh / 2) * height)
            x2 = int((cx + bw / 2) * width)
            y2 = int((cy + bh / 2) * height)

            # Draw a green pedestrian bounding box over the image.
            # These rectangles represent human annotations rather than RF-DETR model predictions.
            cv2.rectangle(
                image,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

    ax.imshow(image)
    ax.set_title(
        f"{row['Condition'].title()}\n"
        f"{int(row['Boxes'])} GT boxes"
    )
    ax.axis("off")

# Select one valid labeled image from each low-light category.
# The preview requires at least one ready image with a pedestrian annotation per category.
preview_rows = []

for condition in ["mild", "moderate", "severe"]:
    candidates = verified_df[
        (verified_df["Condition"] == condition)
        & (verified_df["Boxes"] > 0)
    ]

    if not candidates.empty:
        preview_rows.append(
            candidates.iloc[0]
        )

# Generate a visual review of the selected ground-truth bounding boxes.
# This provides evidence that the label coordinates are aligned with the image dimensions.
if preview_rows:
    fig, axes = plt.subplots(
        1,
        len(preview_rows),
        figsize=(5 * len(preview_rows), 5)
    )

    axes = np.atleast_1d(axes)

    for ax, row in zip(axes, preview_rows):
        display_annotated_image(row, ax)

    # Save the annotation preview for the final RF-DETR report.
    # This figure shows verified dataset labels before COCO conversion or model inference.
    plt.tight_layout()

    PREVIEW_PATH = (
        FIGURES_DIR
        / "cell2_ground_truth_annotation_examples.png"
    )

    plt.savefig(
        PREVIEW_PATH,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print("\nGround Truth Preview:", PREVIEW_PATH)

# Display the saved annotation audit and manifest locations.
# These records will become the basis for source-level splitting and COCO annotation conversion.
print("\nCELL 2 OUTPUT FILES")
print("=" * 65)
print("Annotation Audit:", AUDIT_CSV)
print("Verified Dataset Manifest:", VERIFIED_CSV)
print("Lighting Summary:", CONDITION_CSV)

# Confirm that the annotation audit and manifest generation have completed.
# The next cell will use only verified data and will not invent missing bounding boxes.
print("\nCELL 2 COMPLETED SUCCESSFULLY!")

### 3. COCO Annotation Conversion and Dataset Splitting

This section converts the verified pedestrian bounding-box annotations from YOLO format into COCO format for training and evaluating the RF-DETR object detection model. The conversion transforms normalized bounding-box coordinates into pixel-based COCO annotations, including image identifiers, category identifiers, bounding-box areas, and annotation metadata.

The dataset is partitioned into training, validation, and testing subsets using original source-frame identifiers. This procedure prevents mild, moderate, and severe lighting variants derived from the same source frame from appearing in different dataset partitions. The resulting COCO annotations and corresponding images are organized according to the directory structure required by RF-DETR.

Dataset integrity checks are conducted to confirm image availability, annotation validity, source-frame separation, and consistency between the original YOLO annotation counts and converted COCO records. The final dataset is saved independently from the source files to preserve the original annotations.

In [ ]:

# Import libraries for preparing the RF-DETR COCO dataset.
# These libraries support source-level splitting, image copying, annotation conversion, and dataset verification.
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter
from PIL import Image
import pandas as pd
import numpy as np
import json
import shutil
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Define the original pedestrian project and the new RF-DETR dataset directories.
# All new output files will remain separate from the original manually reviewed annotations.
PROJECT_DIR = Path("/content/drive/MyDrive/Person_Detection")
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"
DATASET_DIR = RFDETR_DIR / "dataset"
FIGURES_DIR = RFDETR_DIR / "figures"

# Create the dataset and figure directories if necessary.
# These directories store converted annotations and report-ready visualizations.
DATASET_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Identify the verified dataset manifest generated in Cell 2.
# This file contains the selected ground-truth annotation paths and source-frame identifiers.
VERIFIED_CSV = DATASET_DIR / "cell2_verified_dataset_manifest.csv"

# Confirm that the verified annotation manifest exists before continuing.
# Missing source records must not be replaced by fabricated bounding-box annotations.
assert VERIFIED_CSV.is_file(), (
    f"Missing verified dataset manifest: {VERIFIED_CSV}"
)

# Load the previously verified pedestrian dataset records.
# The conversion will use the annotations selected during the manual ground-truth review.
verified_df = pd.read_csv(
    VERIFIED_CSV,
    keep_default_na=False
)

# Define the fields required for converting the image records.
# Every record must provide an image, a label file, dimensions, and a source-frame identifier.
required_columns = [
    "Condition",
    "Image",
    "Image_Path",
    "Selected_Label_Path",
    "Source_Frame",
    "Image_Width",
    "Image_Height",
    "Boxes"
]

# Verify that all required fields are present in the dataset manifest.
# The program must stop when essential source information is unavailable.
for column in required_columns:
    assert column in verified_df.columns, (
        f"Missing dataset column: {column}"
    )

# Confirm that the verified manifest contains valid image records.
# An empty dataset cannot be split or converted for RF-DETR training.
assert len(verified_df) > 0, (
    "The verified dataset manifest is empty."
)

# Define the supported lighting categories used in the previous activities.
# Lighting severity is metadata; the object detection category remains pedestrian.
LIGHTING_CONDITIONS = [
    "mild",
    "moderate",
    "severe"
]

# Normalize the lighting labels and source-frame identifiers.
# Consistent strings prevent accidental differences caused by capitalization or whitespace.
verified_df["Condition"] = (
    verified_df["Condition"]
    .astype(str)
    .str.strip()
    .str.lower()
)

verified_df["Source_Frame"] = (
    verified_df["Source_Frame"]
    .astype(str)
    .str.strip()
)

# Confirm that all lighting categories are recognized.
# Unexpected conditions must be investigated before preparing dataset partitions.
assert set(verified_df["Condition"]).issubset(
    set(LIGHTING_CONDITIONS)
), (
    "Unexpected lighting categories were found."
)

# Confirm that every image has a source-frame identifier.
# These identifiers are required to prevent related lighting variants from crossing dataset splits.
assert verified_df["Source_Frame"].ne("").all(), (
    "Some source-frame identifiers are missing."
)

# Confirm that no image is duplicated in the verified manifest.
# Duplicated image records could inflate the COCO dataset and distort model evaluation.
assert not verified_df.duplicated(
    ["Condition", "Image"]
).any(), (
    "Duplicate image records found."
)

# Confirm that the original images and selected annotations are available.
# The conversion must not silently skip missing image or label files.
for _, row in verified_df.iterrows():

    # Verify that the image exists at its recorded path.
    # Missing images cannot be used for RF-DETR training or evaluation.
    assert Path(row["Image_Path"]).is_file(), (
        f"Missing image: {row['Image_Path']}"
    )

    # Verify that the selected ground-truth annotation file exists.
    # This preserves the verified-or-corrected label selection made in Cell 2.
    assert Path(row["Selected_Label_Path"]).is_file(), (
        f"Missing annotation: {row['Selected_Label_Path']}"
    )

# Define a reproducible random seed for dataset splitting.
# The same seed will generate the same source-frame partitions when the dataset is unchanged.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# Extract the unique original source-frame identifiers.
# Images representing different lighting versions of one source will remain together.
unique_sources = sorted(
    verified_df["Source_Frame"].unique()
)

# Confirm that enough independent source frames are available.
# Splitting a very small number of independent scenes would produce unreliable evaluation partitions.
assert len(unique_sources) >= 10, (
    "At least 10 independent source frames are required."
)

# Create the training and temporary source-frame groups.
# Seventy percent of source frames are assigned to training, with the remainder reserved.
train_sources, temporary_sources = train_test_split(
    unique_sources,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

# Divide the temporary source-frame group equally between validation and testing.
# This produces approximately 70 percent training, 15 percent validation, and 15 percent testing.
valid_sources, test_sources = train_test_split(
    temporary_sources,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

# Create source-frame groups for all three dataset partitions.
# No original scene should appear in more than one split.
source_groups = {
    "train": set(train_sources),
    "valid": set(valid_sources),
    "test": set(test_sources)
}

# Confirm that the source-frame groups do not overlap.
# This prevents direct leakage between training, validation, and held-out testing data.
assert source_groups["train"].isdisjoint(source_groups["valid"])
assert source_groups["train"].isdisjoint(source_groups["test"])
assert source_groups["valid"].isdisjoint(source_groups["test"])

# Assign every verified image according to its source-frame group.
# Each image is included once while its related lighting variants remain in the same partition.
split_frames = {}

for split_name, source_set in source_groups.items():
    split_frames[split_name] = (
        verified_df[
            verified_df["Source_Frame"].isin(source_set)
        ]
        .copy()
        .sort_values(
            ["Source_Frame", "Condition", "Image"]
        )
        .reset_index(drop=True)
    )

# Confirm that the split preserves the complete verified dataset.
# No accepted image should be silently removed during dataset partitioning.
assert sum(
    len(frame)
    for frame in split_frames.values()
) == len(verified_df)

# Define the single pedestrian category for COCO annotations.
# YOLO pedestrian class zero will be mapped to COCO category identifier one.
CATEGORIES = [
    {
        "id": 1,
        "name": "person",
        "supercategory": "person"
    }
]

# Define the maximum allowed normalized boundary overflow for rounding correction.
# This tolerance matches the diagnostic that found no significant out-of-bounds annotations.
NORMALIZED_TOLERANCE = 1e-5

# Initialize a collection of minor bounding-box clipping records.
# Recording these adjustments makes the COCO conversion transparent and auditable.
clipping_records = []

# Define the corrected YOLO-to-COCO bounding-box conversion function.
# The converter tolerates only tiny edge deviations and preserves the original source label files.
def convert_yolo_boxes(label_path, image_width, image_height):

    # Read the existing selected annotation file.
    # Empty files represent images without labeled objects and can produce zero COCO annotations.
    lines = Path(label_path).read_text(
        encoding="utf-8-sig"
    ).splitlines()

    # Create a collection for the converted COCO bounding boxes.
    # Each output contains the person category and pixel-based bounding-box coordinates.
    converted_boxes = []

    # Process each nonempty YOLO annotation line.
    # Standard labels contain class ID, center X, center Y, box width, and box height.
    for line_number, line in enumerate(lines, start=1):

        # Skip empty lines without creating artificial object annotations.
        # Only valid annotation entries will be counted.
        if not line.strip():
            continue

        # Parse the five YOLO annotation values.
        # Additional or missing fields are not accepted because this converter supports bounding boxes only.
        parts = line.split()

        if len(parts) != 5:
            raise ValueError(
                f"Invalid YOLO label format: {label_path}, "
                f"line {line_number}"
            )

        # Convert the original annotation values into floating-point numbers.
        # Non-numeric or non-finite values are invalid training labels.
        try:
            values = [
                float(value)
                for value in parts
            ]
        except ValueError:
            raise ValueError(
                f"Nonnumeric annotation: {label_path}, "
                f"line {line_number}"
            )

        if not np.isfinite(values).all():
            raise ValueError(
                f"Non-finite annotation: {label_path}, "
                f"line {line_number}"
            )

        # Separate the object class and normalized box coordinates.
        # This dataset supports only the pedestrian category.
        class_id, cx, cy, bw, bh = values

        # Reject unexpected object categories.
        # Other classes must not be silently converted into pedestrians.
        if class_id != 0:
            raise ValueError(
                f"Unexpected YOLO class {class_id}: "
                f"{label_path}, line {line_number}"
            )

        # Check that normalized center coordinates and box dimensions are valid.
        # Zero-sized or negative-sized objects cannot be used for object detection.
        if not (
            0 <= cx <= 1
            and 0 <= cy <= 1
            and 0 < bw <= 1
            and 0 < bh <= 1
        ):
            raise ValueError(
                f"Invalid normalized YOLO values: "
                f"{label_path}, line {line_number}"
            )

        # Calculate normalized bounding-box corner coordinates.
        # These values make boundary checking independent of the image's pixel resolution.
        x1_normalized = cx - bw / 2.0
        y1_normalized = cy - bh / 2.0
        x2_normalized = cx + bw / 2.0
        y2_normalized = cy + bh / 2.0

        # Measure the maximum amount by which the box extends beyond the image.
        # Small numerical deviations may arise from coordinate rounding during annotation export.
        overflow = max(
            0.0,
            -x1_normalized,
            -y1_normalized,
            x2_normalized - 1.0,
            y2_normalized - 1.0
        )

        # Reject annotations with a significant normalized boundary violation.
        # Only deviations within the documented tolerance will be adjusted automatically.
        if overflow > NORMALIZED_TOLERANCE:
            raise ValueError(
                f"Significant out-of-bounds box: "
                f"{label_path}, line {line_number}, "
                f"overflow={overflow:.10f}"
            )

        # Clip the normalized box corners to the valid zero-to-one coordinate range.
        # The original annotation file remains unchanged and only the exported coordinates are adjusted.
        clipped_x1 = max(
            0.0,
            min(1.0, x1_normalized)
        )

        clipped_y1 = max(
            0.0,
            min(1.0, y1_normalized)
        )

        clipped_x2 = max(
            0.0,
            min(1.0, x2_normalized)
        )

        clipped_y2 = max(
            0.0,
            min(1.0, y2_normalized)
        )

        # Convert the clipped normalized corners into absolute pixel coordinates.
        # COCO stores top-left X and Y together with bounding-box width and height.
        x_min = clipped_x1 * image_width
        y_min = clipped_y1 * image_height
        box_width = (
            clipped_x2 - clipped_x1
        ) * image_width
        box_height = (
            clipped_y2 - clipped_y1
        ) * image_height

        # Reject boxes that have zero or negative area after clipping.
        # This prevents invalid COCO rectangles from entering training or evaluation.
        if box_width <= 0 or box_height <= 0:
            raise ValueError(
                f"Invalid clipped box: "
                f"{label_path}, line {line_number}"
            )

        # Record every annotation that needed a minor coordinate adjustment.
        # This creates a transparent log of rounding corrections made during dataset conversion.
        if overflow > 0:
            clipping_records.append({
                "Label_Path": str(label_path),
                "Line_Number": line_number,
                "Normalized_Overflow": overflow,
                "Original_X1": x1_normalized,
                "Original_Y1": y1_normalized,
                "Original_X2": x2_normalized,
                "Original_Y2": y2_normalized,
                "Clipped_X1": clipped_x1,
                "Clipped_Y1": clipped_y1,
                "Clipped_X2": clipped_x2,
                "Clipped_Y2": clipped_y2
            })

        # Add the validated pedestrian bounding box to the COCO conversion list.
        # Coordinates are stored as pixel values for the matching source image.
        converted_boxes.append({
            "category_id": 1,
            "bbox": [
                round(float(x_min), 5),
                round(float(y_min), 5),
                round(float(box_width), 5),
                round(float(box_height), 5)
            ]
        })

    # Return all valid converted boxes for the current image.
    # The original annotation file is neither edited nor overwritten.
    return converted_boxes

# Define the final COCO dataset export directory.
# Only this generated folder will be replaced if a previous Cell 3 attempt was incomplete.
COCO_DIR = DATASET_DIR / "rfdetr_coco"

# Identify the exact expected parent directory as a deletion safety boundary.
# The cleanup procedure is forbidden from targeting the original Person_Detection files.
EXPECTED_COCO_DIR = (
    RFDETR_DIR
    / "dataset"
    / "rfdetr_coco"
)

# Confirm that the intended export folder matches the allowed path.
# This protects original YOLO labels, model checkpoints, and manually reviewed ground truth.
assert COCO_DIR.resolve() == EXPECTED_COCO_DIR.resolve()

# Preflight all images and annotations before touching an existing export.
# This avoids deleting the old export if a source file is missing or a label is invalid.
print("PREFLIGHT DATASET VERIFICATION")
print("=" * 65)

preflight_box_total = 0

for _, row in verified_df.iterrows():

    # Read the original image dimensions and ensure the file can be opened.
    # COCO coordinates must correspond to the actual saved image size.
    with Image.open(row["Image_Path"]) as image:
        width, height = image.size
        image.verify()

    # Confirm that the image dimensions agree with the previously verified dataset record.
    # Unexpected changes in source image geometry require investigation before conversion.
    assert width == int(row["Image_Width"])
    assert height == int(row["Image_Height"])

    # Validate and convert the bounding boxes without writing any output files.
    # This detects conversion errors before the generated COCO dataset directory is replaced.
    converted = convert_yolo_boxes(
        row["Selected_Label_Path"],
        width,
        height
    )

    # Check that every previously verified pedestrian annotation was converted.
    # A mismatch indicates an inconsistency between the audit and current label files.
    assert len(converted) == int(row["Boxes"]), (
        f"Annotation count mismatch: {row['Image']}"
    )

    preflight_box_total += len(converted)

# Clear the initial preflight clipping log before the actual export.
# Only adjustments performed during the final conversion will appear in the saved audit.
clipping_records.clear()

# Confirm that the complete dataset passed the conversion preflight.
# No existing export will be replaced if a source annotation fails these checks.
print("Verified Images:", len(verified_df))
print("Verified Bounding Boxes:", preflight_box_total)
print("Preflight Status: PASSED")

# Inspect the previous export directory before replacing it.
# Only the exact generated RF-DETR COCO folder may be replaced.
if COCO_DIR.exists():

    # Require the previous export target to be a real directory rather than a file.
    # This prevents accidental deletion of an unrelated file at the expected path.
    assert COCO_DIR.is_dir(), (
        "The COCO output path exists but is not a directory."
    )

    # Allow replacement only when the directory contains expected split folders.
    # Unrecognized files or folders must be reviewed rather than silently deleted.
    allowed_names = {
        "train",
        "valid",
        "test"
    }

    existing_names = {
        path.name
        for path in COCO_DIR.iterdir()
    }

    unexpected_names = existing_names - allowed_names

    assert not unexpected_names, (
        "Unexpected files exist in the COCO output directory: "
        f"{unexpected_names}. Review them before continuing."
    )

    # Require explicit confirmation before deleting an existing complete or partial export.
    # Responding with anything other than YES preserves the existing dataset folder.
    confirmation = input(
        "Existing rfdetr_coco export detected. "
        "Type YES to replace this generated export only: "
    ).strip()

    if confirmation != "YES":
        raise RuntimeError(
            "COCO export replacement cancelled. "
            "Existing files were preserved."
        )

    # Remove only the previous generated dataset export after confirmation.
    # Original images, CO1.3 annotations, and other RF-DETR project files are preserved.
    shutil.rmtree(COCO_DIR)

    print("Previous generated COCO export removed.")

# Create a fresh COCO export directory for the validated dataset.
# Every generated image and JSON file will be written inside this dedicated folder.
COCO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Initialize collections for the converted dataset summary and source-to-export manifest.
# These records will support the final report and dataset integrity checks.
conversion_results = []
split_manifest_rows = []

# Process each source-grouped dataset partition independently.
# The output follows the RF-DETR COCO layout with train, valid, and test folders.
for split_name in ["train", "valid", "test"]:

    # Create the current split directory.
    # Images and their COCO annotation JSON will be placed together in this folder.
    split_dir = COCO_DIR / split_name

    split_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Retrieve the verified images assigned to the current partition.
    # Their source-frame identifiers are already disjoint from the other partitions.
    split_df = split_frames[split_name]

    # Initialize the COCO annotation structure for the current dataset split.
    # Category ID one represents the pedestrian class in all three partitions.
    coco_data = {
        "info": {
            "description": (
                "Low-Light Pedestrian Detection - "
                f"{split_name}"
            ),
            "version": "1.0",
            "source": "Manually reviewed CO1.3 pedestrian annotations"
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": CATEGORIES
    }

    # Initialize sequential COCO identifiers and filename tracking.
    # Each image and annotation requires a unique identifier within its JSON file.
    next_image_id = 1
    next_annotation_id = 1
    used_filenames = set()

    # Convert every image and its selected pedestrian labels.
    # Original files remain unchanged while independent export copies are created.
    for _, row in split_df.iterrows():

        # Retrieve the actual image and annotation locations.
        # These were verified during the preflight before any output files were replaced.
        source_image = Path(
            row["Image_Path"]
        )

        source_label = Path(
            row["Selected_Label_Path"]
        )

        output_filename = source_image.name

        # Prevent duplicated image filenames from overwriting exported files.
        # Filename collisions must be resolved explicitly to preserve image-label correspondence.
        if output_filename in used_filenames:
            raise ValueError(
                f"Duplicate image filename in {split_name}: "
                f"{output_filename}"
            )

        used_filenames.add(
            output_filename
        )

        # Read the image dimensions from the original file.
        # The COCO annotation coordinates will use these exact dimensions.
        with Image.open(source_image) as image:
            width, height = image.size
            image.verify()

        # Convert the selected pedestrian annotations using the corrected converter.
        # Tiny boundary deviations are clipped while larger errors cause an explicit failure.
        converted_boxes = convert_yolo_boxes(
            source_label,
            width,
            height
        )

        # Confirm that the original annotation count was preserved.
        # The conversion must not drop valid pedestrians without documentation.
        assert len(converted_boxes) == int(row["Boxes"]), (
            f"Lost annotations during conversion: {row['Image']}"
        )

        # Copy the original image into the current RF-DETR COCO split.
        # shutil.copy2 preserves the source image without altering its pixels or metadata.
        destination_image = (
            split_dir / output_filename
        )

        shutil.copy2(
            source_image,
            destination_image
        )

        # Add the image metadata required by COCO dataset loaders.
        # The filename refers to the image stored directly beside the current JSON file.
        coco_data["images"].append({
            "id": next_image_id,
            "file_name": output_filename,
            "width": int(width),
            "height": int(height)
        })

        # Create an individual COCO annotation for each pedestrian bounding box.
        # Every annotation references the current image and the person category.
        for box in converted_boxes:

            # Extract the converted pixel-based bounding-box coordinates.
            # COCO represents boxes using top-left X, top-left Y, width, and height.
            bbox = box["bbox"]

            # Add the complete COCO annotation record.
            # The bounding-box area is calculated directly from its exported dimensions.
            coco_data["annotations"].append({
                "id": next_annotation_id,
                "image_id": next_image_id,
                "category_id": 1,
                "bbox": bbox,
                "area": round(
                    bbox[2] * bbox[3],
                    5
                ),
                "iscrowd": 0
            })

            next_annotation_id += 1

        # Record the original and exported locations for traceability.
        # This file will help reproduce the conversion and document dataset provenance.
        split_manifest_rows.append({
            "Split": split_name,
            "Condition": row["Condition"],
            "Source_Frame": row["Source_Frame"],
            "Original_Image": str(source_image),
            "Original_Label": str(source_label),
            "COCO_Image": str(destination_image),
            "COCO_Image_ID": next_image_id,
            "Pedestrian_Boxes": len(converted_boxes)
        })

        next_image_id += 1

    # Save the COCO annotation file under the name expected by RF-DETR.
    # Each split folder contains its own image metadata and bounding-box annotations.
    json_path = (
        split_dir
        / "_annotations.coco.json"
    )

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            coco_data,
            file,
            indent=2
        )

    # Record the completed split and its actual image and annotation totals.
    # These results will be verified against the saved JSON files and source manifest.
    conversion_results.append({
        "Split": split_name,
        "Source_Frames": len(source_groups[split_name]),
        "Images": len(coco_data["images"]),
        "Annotations": len(coco_data["annotations"]),
        "COCO_JSON": str(json_path)
    })

    # Display progress after the current partition has been exported.
    # The annotation count reflects actual pedestrian boxes rather than the number of images.
    print(
        f"{split_name.upper()} COMPLETED | "
        f"Images: {len(coco_data['images'])} | "
        f"Annotations: {len(coco_data['annotations'])}"
    )

# Convert the completed dataset summaries into tables.
# These records provide the exact split sizes and source-to-export mappings.
conversion_df = pd.DataFrame(
    conversion_results
)

split_manifest_df = pd.DataFrame(
    split_manifest_rows
)

# Define the output locations for the conversion summary and split manifest.
# These files will support the dataset preparation section of the final PDF report.
CONVERSION_CSV = (
    DATASET_DIR
    / "cell3_coco_conversion_summary.csv"
)

SPLIT_MANIFEST_CSV = (
    DATASET_DIR
    / "cell3_coco_split_manifest.csv"
)

# Save both dataset documentation tables to Google Drive.
# The original manual annotation inventory remains untouched.
conversion_df.to_csv(
    CONVERSION_CSV,
    index=False
)

split_manifest_df.to_csv(
    SPLIT_MANIFEST_CSV,
    index=False
)

# Save the log of tiny bounding-box corrections made only in the COCO export.
# Even when no clipping is needed, an empty CSV documents that outcome.
CLIPPING_CSV = (
    DATASET_DIR
    / "cell3_bounding_box_clipping_audit.csv"
)

clipping_df = pd.DataFrame(
    clipping_records,
    columns=[
        "Label_Path",
        "Line_Number",
        "Normalized_Overflow",
        "Original_X1",
        "Original_Y1",
        "Original_X2",
        "Original_Y2",
        "Clipped_X1",
        "Clipped_Y1",
        "Clipped_X2",
        "Clipped_Y2"
    ]
)

clipping_df.to_csv(
    CLIPPING_CSV,
    index=False
)

# Validate every saved COCO annotation JSON file.
# The verification checks image references, unique identifiers, box geometry, and annotation counts.
print("\nVERIFYING CONVERTED COCO DATASET")
print("=" * 65)

for split_name in ["train", "valid", "test"]:

    # Load the completed COCO JSON file from the current split folder.
    # Reading the saved file confirms that the exported annotations are valid JSON.
    split_dir = COCO_DIR / split_name
    json_path = split_dir / "_annotations.coco.json"

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as file:
        coco_data = json.load(file)

    # Extract COCO image and annotation identifiers.
    # All identifiers must be unique within the current annotation file.
    image_ids = [
        record["id"]
        for record in coco_data["images"]
    ]

    annotation_ids = [
        record["id"]
        for record in coco_data["annotations"]
    ]

    assert len(image_ids) == len(set(image_ids))
    assert len(annotation_ids) == len(set(annotation_ids))

    # Create a lookup table for image metadata referenced by annotations.
    # Every bounding box must point to a valid image in the same split.
    image_lookup = {
        record["id"]: record
        for record in coco_data["images"]
    }

    # Verify that every COCO image exists inside the current split directory.
    # Missing image files would prevent the RF-DETR training loader from accessing the dataset.
    for record in coco_data["images"]:
        assert (
            split_dir / record["file_name"]
        ).is_file(), (
            f"Missing exported COCO image: {record['file_name']}"
        )

    # Verify each pedestrian annotation and its associated image geometry.
    # These checks ensure that no converted bounding box extends outside the image.
    for annotation in coco_data["annotations"]:

        # Confirm the annotation references an existing image and the pedestrian category.
        # Category ID one must remain consistent throughout all three dataset partitions.
        assert annotation["image_id"] in image_lookup
        assert annotation["category_id"] == 1

        # Retrieve the referenced image dimensions and the annotation rectangle.
        # The COCO box uses X, Y, width, and height in absolute pixels.
        image_info = image_lookup[
            annotation["image_id"]
        ]

        x, y, width, height = annotation["bbox"]

        # Confirm that the exported rectangle has positive area.
        # Invalid box dimensions would cause unreliable training and evaluation.
        assert width > 0
        assert height > 0

        # Confirm that bounding boxes remain within the image boundaries.
        # A tiny pixel tolerance is allowed only for rounding during JSON serialization.
        assert x >= -1e-3
        assert y >= -1e-3
        assert (
            x + width
        ) <= image_info["width"] + 1e-3

        assert (
            y + height
        ) <= image_info["height"] + 1e-3

        # Verify that the stored area corresponds to the exported bounding-box dimensions.
        # This prevents inconsistent metadata from reaching the RF-DETR dataset loader.
        assert np.isclose(
            annotation["area"],
            width * height,
            atol=0.1
        )

    # Compare the exported annotation count against the original selected YOLO labels.
    # This confirms that the conversion preserved every verified pedestrian instance.
    original_box_count = split_manifest_df.loc[
        split_manifest_df["Split"] == split_name,
        "Pedestrian_Boxes"
    ].sum()

    assert len(coco_data["annotations"]) == original_box_count

    # Display the verified image and annotation counts for the current partition.
    # All three splits must pass this inspection before the model is initialized.
    print(
        f"{split_name}: "
        f"{len(coco_data['images'])} valid images, "
        f"{len(coco_data['annotations'])} valid annotations"
    )

# Verify that every source frame belongs to exactly one dataset partition.
# This helps prevent the same scene's lighting variants from crossing train-test boundaries.
source_split_counts = (
    split_manifest_df.groupby("Source_Frame")["Split"]
    .nunique()
)

assert source_split_counts.eq(1).all(), (
    "Source-frame leakage detected after dataset conversion."
)

# Verify that the exported image and box totals match the original accepted manifest.
# No valid image or pedestrian annotation should be lost during the completed conversion.
assert conversion_df["Images"].sum() == len(verified_df)

assert conversion_df["Annotations"].sum() == verified_df["Boxes"].sum()

# Calculate the lighting distribution across the three dataset partitions.
# This will document whether mild, moderate, and severe scenes remain balanced.
lighting_distribution = pd.crosstab(
    split_manifest_df["Split"],
    split_manifest_df["Condition"]
)

# Save the lighting distribution to Google Drive.
# The table can be directly included in the dataset description section of the report.
LIGHTING_CSV = (
    DATASET_DIR
    / "cell3_split_lighting_distribution.csv"
)

lighting_distribution.to_csv(
    LIGHTING_CSV
)

# Display the final converted dataset summary.
# These counts are calculated from the actual output rather than assumed in advance.
print("\nFINAL RF-DETR DATASET SUMMARY")
print("=" * 65)

print(
    conversion_df[
        [
            "Split",
            "Source_Frames",
            "Images",
            "Annotations"
        ]
    ].to_string(index=False)
)

# Display the distribution of exported low-light variants.
# These values help describe how the source-level split distributed the lighting conditions.
print("\nLIGHTING DISTRIBUTION BY SPLIT")
print("=" * 65)
print(
    lighting_distribution.to_string()
)

# Report how many annotations required tiny boundary adjustments.
# Larger boundary violations were rejected during preflight rather than silently clipped.
print("\nBOUNDING-BOX ROUNDING AUDIT")
print("=" * 65)
print("Annotations with tiny clipping adjustments:", len(clipping_df))

if not clipping_df.empty:
    print(
        "Maximum normalized overflow:",
        clipping_df["Normalized_Overflow"].max()
    )

# Load the completed training COCO annotations for visual inspection.
# A plotted pedestrian example will confirm alignment between images and converted boxes.
train_json_path = (
    COCO_DIR
    / "train"
    / "_annotations.coco.json"
)

with open(
    train_json_path,
    "r",
    encoding="utf-8"
) as file:
    train_coco = json.load(file)

# Count annotations associated with each training image.
# A positive example is selected to show at least one converted pedestrian bounding box.
image_annotation_counts = Counter(
    annotation["image_id"]
    for annotation in train_coco["annotations"]
)

# Select the first training image containing at least one pedestrian.
# Images without annotations are not suitable for illustrating bounding-box alignment.
sample_image_info = next(
    (
        image
        for image in train_coco["images"]
        if image_annotation_counts[image["id"]] > 0
    ),
    None
)

# Create a preview figure when an annotated training image is available.
# The figure can be included as evidence of dataset conversion in the final PDF.
if sample_image_info is not None:

    # Open the copied training image in RGB format.
    # This preserves the expected appearance when drawn with Matplotlib.
    sample_image_path = (
        COCO_DIR
        / "train"
        / sample_image_info["file_name"]
    )

    with Image.open(sample_image_path) as image:
        sample_image = image.convert("RGB").copy()

    # Retrieve the COCO bounding boxes belonging to the selected image.
    # Only annotations referencing the selected image ID are displayed.
    sample_annotations = [
        annotation
        for annotation in train_coco["annotations"]
        if annotation["image_id"] == sample_image_info["id"]
    ]

    # Initialize the ground-truth visualization.
    # Green rectangles represent converted annotations and not RF-DETR predictions.
    fig, ax = plt.subplots(
        figsize=(10, 7)
    )

    ax.imshow(
        sample_image
    )

    # Draw every COCO pedestrian bounding box on the selected training image.
    # This checks the visual alignment of exported pixel coordinates.
    for annotation in sample_annotations:
        x, y, width, height = annotation["bbox"]

        rectangle = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )

        ax.add_patch(
            rectangle
        )

    # Add the figure title and hide the coordinate axes.
    # The visualization documents the resulting COCO ground-truth geometry.
    ax.set_title(
        "RF-DETR: Converted COCO Pedestrian Ground Truth"
    )

    ax.axis("off")

    # Save the visualized annotation example for the final report.
    # The original image and its label files remain unchanged.
    plt.tight_layout()

    COCO_PREVIEW = (
        FIGURES_DIR
        / "cell3_converted_coco_annotation_example.png"
    )

    plt.savefig(
        COCO_PREVIEW,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print("\nSaved COCO Preview:", COCO_PREVIEW)

# Display the locations of all newly generated COCO dataset files.
# These paths will be reused in Cell 4 for RF-DETR model initialization.
print("\nCELL 3 OUTPUT FILES")
print("=" * 65)
print("COCO Dataset:", COCO_DIR)
print("Conversion Summary:", CONVERSION_CSV)
print("Split Manifest:", SPLIT_MANIFEST_CSV)
print("Lighting Distribution:", LIGHTING_CSV)
print("Bounding-Box Clipping Audit:", CLIPPING_CSV)
print("Total Images:", int(conversion_df["Images"].sum()))
print("Total Annotations:", int(conversion_df["Annotations"].sum()))
print("Source-Frame Leakage: NONE")

# Confirm that the corrected conversion and validation process is complete.
# The next cell will install RF-DETR and load its pretrained object detection model.
print("\nCELL 3 COMPLETED SUCCESSFULLY!")

### 4. RF-DETR Installation and Pretrained Model Initialization

This section installs and initializes the RF-DETR object detection framework using PyTorch. The RF-DETR Nano configuration is selected as a computationally manageable pretrained detection transformer for experimentation on a Google Colab GPU.

The prepared COCO dataset is examined to verify compatibility with the RF-DETR training pipeline, including annotation files, image references, pedestrian category identifiers, and dataset partitions. The pretrained model is subsequently initialized and tested on a sample image to establish that the environment and inference pipeline are functioning correctly.

This initialization stage serves as a technical verification before model fine-tuning. The preliminary predictions are not treated as final experimental results because the model has not yet been adapted to the project's low-light pedestrian dataset.

In [ ]:

# Install the official RF-DETR package and COCO evaluation utilities.
# These libraries provide the pretrained detection transformer and standard COCO annotation support.
%pip install -q rfdetr pycocotools supervision

# Import libraries for reading annotations, images, and package information.
# These utilities will verify the dataset and document the installed environment.
from pathlib import Path
from importlib.metadata import version
import json
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageDraw

# Import the official RF-DETR Nano detection model.
# This is a pretrained transformer-based object detector provided by Roboflow.
from rfdetr import RFDETRNano

# Import the COCO annotation API.
# It provides a standard way to inspect and validate the prepared object detection dataset.
from pycocotools.coco import COCO

# Define the main RF-DETR project directory.
# This folder contains the COCO dataset created in Cell 3.
PROJECT_DIR = Path("/content/drive/MyDrive/Person_Detection")
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"

# Define the dataset, model, evaluation, and figure directories.
# The model setup outputs will be kept separate from previous YOLOv11 experiments.
DATASET_DIR = RFDETR_DIR / "dataset" / "rfdetr_coco"
MODEL_DIR = RFDETR_DIR / "trained_models"
RESULTS_DIR = RFDETR_DIR / "evaluation_results"
FIGURES_DIR = RFDETR_DIR / "figures"

# Create output directories if they do not already exist.
# Existing model checkpoints and dataset annotations will remain unchanged.
for folder in [MODEL_DIR, RESULTS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Verify that the RF-DETR dataset directory exists.
# Training must not begin if the COCO conversion did not complete successfully.
assert DATASET_DIR.is_dir(), (
    f"RF-DETR dataset not found: {DATASET_DIR}"
)

# Display the installed package and PyTorch versions.
# Recording these versions helps make the training experiment reproducible.
print("RF-DETR SOFTWARE ENVIRONMENT")
print("=" * 65)
print("RF-DETR Package Version:", version("rfdetr"))
print("PyTorch Version:", torch.__version__)
print("Torchvision Version:", version("torchvision"))
print("PyCOCOTools Version:", version("pycocotools"))

# Select CUDA when a supported GPU is available.
# RF-DETR training is resource-intensive and is not recommended on CPU for this experiment.
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Verify the available GPU and display its memory capacity.
# GPU information helps us choose safe training batch sizes in the following cell.
print("\nHARDWARE VERIFICATION")
print("=" * 65)
print("Selected Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        )
    )
else:
    print(
        "WARNING: No CUDA GPU was detected. "
        "Enable GPU under Runtime > Change runtime type."
    )

# Define the required COCO annotation files for the three dataset partitions.
# RF-DETR expects an _annotations.coco.json file in each split folder.
SPLIT_NAMES = ["train", "valid", "test"]

annotation_paths = {
    split: DATASET_DIR / split / "_annotations.coco.json"
    for split in SPLIT_NAMES
}

# Verify that all three dataset partitions have saved COCO annotations.
# A missing validation or test split would prevent complete project evaluation.
for split_name, annotation_path in annotation_paths.items():
    assert annotation_path.is_file(), (
        f"Missing COCO annotation file: {annotation_path}"
    )

# Initialize a collection for the dataset compatibility summary.
# Each split will contribute image, annotation, and category statistics.
dataset_records = []

# Load and inspect the COCO annotation file for every dataset split.
# This confirms that the annotations follow the expected object detection structure.
for split_name in SPLIT_NAMES:

    # Load the current split using the official COCO API.
    # Successful loading verifies that the JSON can be interpreted as COCO annotations.
    annotation_path = annotation_paths[split_name]
    coco = COCO(str(annotation_path))

    # Retrieve the image IDs, annotation IDs, and category IDs.
    # These collections allow us to validate all image references and object categories.
    image_ids = coco.getImgIds()
    annotation_ids = coco.getAnnIds()
    category_ids = coco.getCatIds()

    # Confirm that the dataset contains the single pedestrian category.
    # This project performs pedestrian-only object detection rather than multiclass classification.
    categories = coco.loadCats(category_ids)

    assert len(categories) == 1, (
        f"{split_name}: expected exactly one category."
    )

    assert categories[0]["id"] == 1, (
        f"{split_name}: unexpected COCO category identifier."
    )

    assert categories[0]["name"] == "person", (
        f"{split_name}: expected the person category."
    )

    # Verify that every COCO image record points to an existing image file.
    # Broken image references must be corrected before training.
    for image_info in coco.loadImgs(image_ids):
        image_path = (
            DATASET_DIR
            / split_name
            / image_info["file_name"]
        )

        assert image_path.is_file(), (
            f"Missing dataset image: {image_path}"
        )

    # Verify that the annotations reference known image IDs and the person category.
    # This confirms the structural consistency of the converted COCO labels.
    for annotation in coco.loadAnns(annotation_ids):
        assert annotation["image_id"] in coco.imgs
        assert annotation["category_id"] == 1

    # Add the verified split counts to the compatibility summary.
    # These measured counts will support the dataset description in the final report.
    dataset_records.append({
        "Split": split_name,
        "Images": len(image_ids),
        "Annotations": len(annotation_ids),
        "Categories": len(category_ids),
        "Category_Name": categories[0]["name"],
        "Status": "Passed"
    })

# Convert the dataset compatibility results into a table.
# All three splits should show one object category and successful verification.
dataset_summary = pd.DataFrame(dataset_records)

# Display the dataset verification results.
# The counts should agree with the COCO conversion summary produced in Cell 3.
print("\nRF-DETR COCO DATASET COMPATIBILITY")
print("=" * 65)
print(dataset_summary.to_string(index=False))

# Confirm that the COCO dataset contains the expected partitions.
# This prevents an incomplete export from being used for model training.
assert len(dataset_summary) == 3
assert dataset_summary["Status"].eq("Passed").all()

# Confirm that the training dataset contains annotated pedestrian examples.
# A detector cannot be meaningfully fine-tuned on a split without any object annotations.
train_row = dataset_summary.loc[
    dataset_summary["Split"] == "train"
].iloc[0]

assert int(train_row["Images"]) > 0
assert int(train_row["Annotations"]) > 0

# Save the dataset compatibility summary for project documentation.
# This CSV records the image counts and annotation counts verified before model initialization.
DATASET_SUMMARY_CSV = (
    RESULTS_DIR / "cell4_rfdetr_dataset_compatibility.csv"
)

dataset_summary.to_csv(
    DATASET_SUMMARY_CSV,
    index=False
)

# Initialize the official pretrained RF-DETR Nano model.
# Its pretrained weights may need to be downloaded during the first execution of this cell.
print("\nINITIALIZING PRETRAINED RF-DETR NANO")
print("=" * 65)

model = RFDETRNano()

# Display the initialized model wrapper type.
# This confirms that the official RF-DETR Nano class was successfully constructed.
print("Model Wrapper:", type(model).__name__)
print("Model Initialization: SUCCESS")

# Inspect the model's public training and prediction interfaces.
# These methods will be used for fine-tuning and evaluating the pedestrian detector.
print("\nRF-DETR MODEL INTERFACE")
print("=" * 65)
print("Training Method:", inspect.signature(model.train))
print("Prediction Method:", inspect.signature(model.predict))

# Retrieve the model configuration where supported by the installed package.
# The configuration may expose the backbone, resolution, and detection head settings.
if hasattr(model, "get_model_config"):
    config = model.get_model_config()

    print("\nMODEL CONFIGURATION")
    print("=" * 65)
    print(config)
else:
    print("\nModel configuration accessor is unavailable.")

# Select the first training image as a preliminary inference example.
# The image has already been verified and belongs to the COCO training split.
train_coco = COCO(str(annotation_paths["train"]))

sample_image_id = train_coco.getImgIds()[0]
sample_image_info = train_coco.loadImgs(
    [sample_image_id]
)[0]

sample_image_path = (
    DATASET_DIR
    / "train"
    / sample_image_info["file_name"]
)

# Load the example image in RGB format for inference and visualization.
# Its original dimensions will be preserved in the displayed prediction.
with Image.open(sample_image_path) as image:
    sample_image = image.convert("RGB").copy()

# Run a preliminary prediction using the pretrained RF-DETR model.
# This is only an environment check and is not the final fine-tuned test evaluation.
print("\nRUNNING PRETRAINED MODEL INFERENCE")
print("=" * 65)

detections = model.predict(
    sample_image,
    threshold=0.35
)

# Display the total number of predicted detections.
# The pretrained model may detect other COCO classes because it has not yet been fine-tuned.
print("Total Preliminary Detections:", len(detections))

# Prepare an RGB copy for drawing preliminary detection boxes.
# The visualized predictions will be saved as an initialization figure.
preview_image = sample_image.copy()
draw = ImageDraw.Draw(preview_image)

# Iterate through the preliminary detections.
# Bounding boxes are drawn in pixel coordinates as returned by the inference library.
for index in range(len(detections)):

    # Retrieve the predicted bounding box, confidence, and pretrained class identifier.
    # These predictions may contain multiple COCO object categories before custom fine-tuning.
    x1, y1, x2, y2 = [
        float(value)
        for value in detections.xyxy[index]
    ]

    confidence = float(
        detections.confidence[index]
    )

    class_id = int(
        detections.class_id[index]
    )

    # Draw a visible rectangle around the predicted object.
    # A text label identifies the numeric pretrained class and confidence score.
    draw.rectangle(
        [x1, y1, x2, y2],
        outline="lime",
        width=3
    )

    draw.text(
        (x1, max(0, y1 - 16)),
        f"Class {class_id}: {confidence:.2f}",
        fill="lime"
    )

# Display the preliminary object detection visualization.
# It confirms that the pretrained model can process an image from the exported dataset.
plt.figure(figsize=(11, 7))
plt.imshow(preview_image)
plt.title(
    "RF-DETR Nano: Preliminary Pretrained Predictions"
)
plt.axis("off")
plt.tight_layout()

# Save the prediction preview to the RF-DETR figures directory.
# This image is useful for documenting model initialization but not final trained accuracy.
PREVIEW_PATH = (
    FIGURES_DIR / "cell4_pretrained_rfdetr_inference.png"
)

plt.savefig(
    PREVIEW_PATH,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close()

# Record the initialized model and dataset configuration for reproducibility.
# This metadata will be reused when preparing the final PDF report and GitHub repository.
initialization_record = {
    "Model": "RF-DETR Nano",
    "Package_Version": version("rfdetr"),
    "PyTorch_Version": torch.__version__,
    "Device": str(DEVICE),
    "Dataset_Directory": str(DATASET_DIR),
    "Dataset_Format": "COCO",
    "Object_Class": "person",
    "Number_of_Classes": 1,
    "Pretrained_Model_Initialized": True,
    "Preliminary_Inference_Completed": True
}

# Save the configuration information as a CSV file.
# The file documents the software environment and successful pretrained inference.
INITIALIZATION_CSV = (
    RESULTS_DIR / "cell4_rfdetr_model_initialization.csv"
)

pd.DataFrame(
    [initialization_record]
).to_csv(
    INITIALIZATION_CSV,
    index=False
)

# Display the final initialization summary and saved output files.
# The next stage will prepare the training settings and fine-tuning procedure.
print("\nCELL 4 OUTPUT FILES")
print("=" * 65)
print("Dataset Summary:", DATASET_SUMMARY_CSV)
print("Model Initialization:", INITIALIZATION_CSV)
print("Pretrained Prediction Figure:", PREVIEW_PATH)
print("Dataset Directory:", DATASET_DIR)
print("Model Type:", type(model).__name__)

# Confirm that the pretrained model and COCO dataset passed initialization checks.
# The next cell will begin configuring model fine-tuning and training optimization.
print("\nCELL 4 COMPLETED SUCCESSFULLY!")

### 5. RF-DETR Model Fine-Tuning and Optimization

This section fine-tunes the pretrained RF-DETR Nano object detection model using the prepared low-light pedestrian dataset. The training procedure uses the RF-DETR framework's integrated optimization pipeline, which performs object detection loss computation, parameter updates, validation, and checkpoint management.

The experiment employs the AdamW optimizer, cosine learning-rate scheduling, and gradient accumulation to support training on available GPU resources. Early stopping monitors validation mean Average Precision (mAP), and the best-performing model weights are retained for subsequent evaluation. The training process uses only the training and validation partitions, preserving the test dataset for independent final assessment.

Training settings, software versions, execution duration, and available checkpoints are recorded to support reproducibility and discussion of optimization methods in the final project report.

In [ ]:
# Install the additional dependencies required for RF-DETR training.
# This includes PyTorch Lightning and the training and logging components.
%pip install -q "rfdetr[train,loggers]"

# Import the libraries required for training configuration and result management.
# These utilities record experiment settings, validate dataset locations, and track completed training.
from pathlib import Path
from datetime import datetime
from time import perf_counter
from importlib.metadata import version
import inspect
import json
import random
import gc
import numpy as np
import pandas as pd
import torch

# Import the official pretrained RF-DETR Nano object detection model.
# This is the same model architecture initialized and verified during Cell 4.
from rfdetr import RFDETRNano

# Define the project and COCO dataset directories.
# The model will train on the verified images and pedestrian annotations produced in Cell 3.
PROJECT_DIR = Path("/content/drive/MyDrive/Person_Detection")
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"
DATASET_DIR = RFDETR_DIR / "dataset" / "rfdetr_coco"
MODEL_DIR = RFDETR_DIR / "trained_models"
RESULTS_DIR = RFDETR_DIR / "evaluation_results"

# Define a dedicated training output directory.
# Checkpoints, training logs, and optimization artifacts will be stored together.
TRAINING_DIR = MODEL_DIR / "rfdetr_nano_training"

# Create the output folders if they do not already exist.
# Original COCO images and corrected YOLO annotations remain unchanged.
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Verify that the required COCO dataset partitions exist.
# All three partitions must be available before starting the RF-DETR training experiment.
for split_name in ["train", "valid", "test"]:
    split_dir = DATASET_DIR / split_name
    annotation_file = split_dir / "_annotations.coco.json"

    assert split_dir.is_dir(), (
        f"Missing dataset partition: {split_dir}"
    )

    assert annotation_file.is_file(), (
        f"Missing COCO annotation file: {annotation_file}"
    )

# Confirm that training and validation datasets contain annotated images.
# This prevents an empty or incomplete COCO export from being passed to the training library.
for split_name in ["train", "valid"]:
    annotation_file = (
        DATASET_DIR
        / split_name
        / "_annotations.coco.json"
    )

    with open(annotation_file, "r", encoding="utf-8") as file:
        annotation_data = json.load(file)

    assert len(annotation_data["images"]) > 0, (
        f"{split_name} contains no images."
    )

    assert len(annotation_data["annotations"]) > 0, (
        f"{split_name} contains no bounding-box annotations."
    )

    assert (
        len(annotation_data["categories"]) == 1
        and annotation_data["categories"][0]["name"] == "person"
    ), (
        f"{split_name} must contain the person category."
    )

# Require an available CUDA GPU before training.
# RF-DETR transformer fine-tuning is computationally intensive and this configuration targets Colab GPU hardware.
assert torch.cuda.is_available(), (
    "CUDA GPU is unavailable. "
    "Go to Runtime > Change runtime type and select a GPU."
)

# Select the CUDA device for the RF-DETR training process.
# GPU information will also be included in the saved experiment configuration.
DEVICE = "cuda"

GPU_NAME = torch.cuda.get_device_name(0)

GPU_MEMORY_GB = round(
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3),
    2
)

# Configure the number of epochs and the optimization hyperparameters.
# These settings define a reproducible initial fine-tuning experiment.
SEED = 42
EPOCHS = 15
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
EVAL_BATCH_SIZE = 1
LEARNING_RATE = 0.0001

# Configure validation monitoring and checkpoint intervals.
# Early stopping uses validation mAP, while additional checkpoints are retained periodically.
EARLY_STOPPING_PATIENCE = 5
CHECKPOINT_INTERVAL = 5

# Seed Python, NumPy, and PyTorch before constructing the training model.
# This reduces random variation, although hardware execution may still introduce nondeterminism.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Calculate the effective batch size after gradient accumulation.
# Optimizer updates accumulate gradients across four two-image minibatches.
EFFECTIVE_BATCH_SIZE = (
    BATCH_SIZE * GRAD_ACCUM_STEPS
)

# Display the selected hardware and training configuration.
# Reviewing these settings before training helps prevent accidental expensive runs.
print("RF-DETR NANO TRAINING CONFIGURATION")
print("=" * 65)
print("GPU:", GPU_NAME)
print("GPU Memory (GB):", GPU_MEMORY_GB)
print("Dataset:", DATASET_DIR)
print("Epochs:", EPOCHS)
print("Physical Batch Size:", BATCH_SIZE)
print("Gradient Accumulation:", GRAD_ACCUM_STEPS)
print("Effective Batch Size:", EFFECTIVE_BATCH_SIZE)
print("Learning Rate:", LEARNING_RATE)
print("Optimizer:", "AdamW")
print("Learning Rate Scheduler:", "Cosine")
print("Early Stopping:", True)
print("Early Stopping Patience:", EARLY_STOPPING_PATIENCE)
print("Early Stopping Monitor:", "Validation mAP")
print("Checkpoint Interval:", CHECKPOINT_INTERVAL)

# Define the official RF-DETR training arguments.
# These parameters use the documented training API rather than a custom unverified training loop.
training_config = {
    "dataset_dir": str(DATASET_DIR),
    "output_dir": str(TRAINING_DIR),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "lr": LEARNING_RATE,
    "optimizer": "adamw",
    "lr_scheduler": "cosine",
    "early_stopping": True,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": 0.001,
    "best_model_metric": "map",
    "skip_best_epochs": 1,
    "checkpoint_interval": CHECKPOINT_INTERVAL,
    "seed": SEED,
    "device": DEVICE
}

# Prepare a reproducibility record describing the training run.
# This record includes library versions, dataset location, GPU hardware, and optimization settings.
experiment_record = {
    "experiment": "RF-DETR Nano Low-Light Pedestrian Detection",
    "model": "RFDETRNano",
    "dataset_format": "COCO",
    "dataset_directory": str(DATASET_DIR),
    "object_class": "person",
    "seed": SEED,
    "gpu": GPU_NAME,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "pytorch_version": torch.__version__,
    "rfdetr_version": version("rfdetr"),
    "training_parameters": training_config,
    "early_stopping_note": (
        "Built-in RF-DETR early stopping monitors validation mAP, "
        "not validation loss."
    )
}

# Define the metadata file used to document this training run.
# The settings will be saved before training begins so they remain available if Colab disconnects.
CONFIG_PATH = RESULTS_DIR / "cell5_rfdetr_training_configuration.json"

# Check whether this experiment already has training outputs.
# An existing training directory should not be silently overwritten by a new run.
existing_training_files = (
    list(TRAINING_DIR.iterdir())
    if TRAINING_DIR.is_dir()
    else []
)

# Stop if an earlier training attempt already produced files.
# This protects previously trained checkpoints from being replaced without inspection.
if existing_training_files:
    print("\nEXISTING TRAINING OUTPUT DETECTED")
    print("=" * 65)

    for path in existing_training_files[:20]:
        print(path.name)

    raise RuntimeError(
        "This training directory already contains files. "
        "Inspect or resume the existing experiment before starting a new run. "
        "Do not delete trained checkpoints automatically."
    )

# Create the dedicated output directory after verifying it is safe.
# Only the new experiment's training artifacts will be written here.
TRAINING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save the complete training configuration as readable JSON.
# This file supports reproducibility and the training methodology in the final PDF report.
with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(
        experiment_record,
        file,
        indent=2,
        default=str
    )

# Initialize a fresh pretrained RF-DETR Nano model.
# Using a new instance ensures this training run begins from pretrained weights rather than an earlier modified state.
print("\nINITIALIZING RF-DETR NANO FOR FINE-TUNING")
print("=" * 65)

training_model = RFDETRNano()

# Inspect the installed model's training method.
# This helps detect an incompatible package API before beginning an expensive training process.
train_signature = inspect.signature(
    training_model.train
)

print("RF-DETR Training Signature:")
print(train_signature)

# Verify that the installed training API accepts the required configuration fields.
# A method with **kwargs may validate them internally, while explicit signatures expose accepted parameter names.
parameters = train_signature.parameters

accepts_kwargs = any(
    parameter.kind == inspect.Parameter.VAR_KEYWORD
    for parameter in parameters.values()
)

if not accepts_kwargs:
    unsupported = [
        key
        for key in training_config
        if key not in parameters
    ]

    assert not unsupported, (
        "Unsupported RF-DETR training parameters: "
        f"{unsupported}"
    )

# Clear unused GPU memory before starting the fine-tuning process.
# This can help recover memory occupied by temporary inference tensors from Cell 4.
gc.collect()
torch.cuda.empty_cache()

# Begin measuring the full fine-tuning execution duration.
# The timing includes data loading, training, validation, and checkpoint operations.
print("\nSTARTING RF-DETR FINE-TUNING")
print("=" * 65)
print("This operation may take a while depending on GPU availability.")
print("The validation set will be used for checkpoint selection.")
print("The test set is reserved for final evaluation.\n")

torch.cuda.synchronize()
training_start = perf_counter()

# Fine-tune the pretrained RF-DETR Nano model using the official training pipeline.
# The framework handles matching, detection losses, optimization, validation, and checkpoint callbacks.
training_model.train(
    **training_config
)

# Record the elapsed training duration after synchronization.
# Synchronizing CUDA ensures that pending GPU computations finish before timing ends.
torch.cuda.synchronize()

training_duration_seconds = (
    perf_counter() - training_start
)

# Locate the checkpoints and other training artifacts generated by RF-DETR.
# The exact files depend on the installed package version and completed epochs.
checkpoint_files = sorted([
    path
    for path in TRAINING_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".pth", ".ckpt"}
])

# Identify the validation-selected best checkpoint when present.
# Official RF-DETR training commonly saves checkpoint_best_total.pth for subsequent inference.
best_checkpoint_candidates = [
    path
    for path in checkpoint_files
    if path.name == "checkpoint_best_total.pth"
]

best_checkpoint = (
    best_checkpoint_candidates[0]
    if best_checkpoint_candidates
    else None
)

# Record the completed training status and saved checkpoint locations.
# These metadata values help verify training completion before the final test evaluation.
experiment_record["training_completed"] = True
experiment_record["training_duration_seconds"] = round(
    training_duration_seconds,
    2
)
experiment_record["checkpoint_count"] = len(
    checkpoint_files
)
experiment_record["best_checkpoint_path"] = (
    str(best_checkpoint)
    if best_checkpoint is not None
    else None
)

# Update the saved experiment configuration with completed training metadata.
# This makes the record useful for the final report and GitHub documentation.
with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(
        experiment_record,
        file,
        indent=2,
        default=str
    )

# Create a structured checkpoint inventory for the completed training run.
# Each record provides the filename, file size, and Google Drive location.
checkpoint_inventory = pd.DataFrame([
    {
        "Checkpoint": path.name,
        "Path": str(path),
        "Size_MB": round(
            path.stat().st_size / (1024 ** 2),
            2
        )
    }
    for path in checkpoint_files
], columns=[
    "Checkpoint",
    "Path",
    "Size_MB"
])

# Save the checkpoint inventory for subsequent model selection and report generation.
# The file provides evidence that RF-DETR weights were saved during fine-tuning.
CHECKPOINT_CSV = (
    RESULTS_DIR / "cell5_rfdetr_checkpoint_inventory.csv"
)

checkpoint_inventory.to_csv(
    CHECKPOINT_CSV,
    index=False
)

# Display the completed training duration and available model checkpoints.
# The final evaluation stage will select and load the best available trained weights.
print("\nRF-DETR TRAINING RESULTS")
print("=" * 65)

print(
    "Training Duration (minutes):",
    round(training_duration_seconds / 60, 2)
)

print(
    "Number of Checkpoint Files:",
    len(checkpoint_files)
)

print("\nSAVED CHECKPOINTS")
print("=" * 65)

if checkpoint_files:
    print(
        checkpoint_inventory.to_string(index=False)
    )
else:
    print(
        "WARNING: No checkpoint files were found. "
        "Inspect the training output directory."
    )

# Display the best checkpoint path if RF-DETR generated the expected file.
# A missing best checkpoint requires inspection rather than assuming a model was successfully saved.
if best_checkpoint is not None:
    print("\nBEST MODEL CHECKPOINT")
    print(best_checkpoint)
else:
    print(
        "\nWARNING: checkpoint_best_total.pth was not found. "
        "Inspect the generated files before proceeding."
    )

# Verify that the training run produced at least one saved model checkpoint.
# Without trained weights, final test metrics cannot represent the fine-tuned RF-DETR model.
assert checkpoint_files, (
    "Training ended without a saved checkpoint. "
    "Review the training logs and output folder."
)

# Display the final training record and output locations.
# These files will be used in the next cell to inspect losses and validation performance.
print("\nCELL 5 OUTPUT FILES")
print("=" * 65)
print("Training Directory:", TRAINING_DIR)
print("Configuration Record:", CONFIG_PATH)
print("Checkpoint Inventory:", CHECKPOINT_CSV)
print("Best Checkpoint:", best_checkpoint)

# Confirm successful completion of the fine-tuning stage.
# The next cell will inspect training logs and evaluate the trained detector.
print("\nCELL 5 COMPLETED SUCCESSFULLY!")

### 5.1 Recovery and Execution of RF-DETR Fine-Tuning

This section resumes the RF-DETR training workflow after resolving missing training dependencies. The environment is verified before execution, and existing training artifacts are inspected to determine whether a previous training checkpoint can be resumed safely.

The pretrained RF-DETR Nano model is fine-tuned using the pedestrian-only COCO dataset with AdamW optimization, cosine learning-rate scheduling, gradient accumulation, validation-based early stopping, and model checkpointing. The training and validation datasets are used for model optimization and selection, while the independent test dataset remains reserved for subsequent final evaluation.

The training configuration and resulting checkpoint inventory are saved to support reproducibility, performance analysis, and final project documentation.

In [ ]:

# Import libraries for inspecting checkpoint files and training results.
# These utilities allow the notebook to verify previously saved RF-DETR artifacts without retraining.
from pathlib import Path
from importlib.metadata import version
import json
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

# Import the official RF-DETR Nano model.
# The checkpoint will be restored using the model's supported pretrained-weight loading interface.
from rfdetr import RFDETRNano

# Define the original pedestrian detection project directory.
# This is the Google Drive location used throughout the earlier RF-DETR preparation cells.
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Person_Detection"
)

# Define the RF-DETR project and results folders.
# Existing dataset files and trained weights will remain unchanged.
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"
MODEL_DIR = RFDETR_DIR / "trained_models"
RESULTS_DIR = RFDETR_DIR / "evaluation_results"
FIGURES_DIR = RFDETR_DIR / "figures"

# Define the training folder that contains the previously saved checkpoint.
# This path matches the training output directory reported in your completed checkpoint message.
TRAINING_DIR = (
    MODEL_DIR
    / "rfdetr_nano_training"
)

# Create separate output folders for checkpoint analysis and graphs.
# No files inside the original training checkpoint directory will be deleted.
ANALYSIS_DIR = RESULTS_DIR / "cell6_training_analysis"
ANALYSIS_FIGURES_DIR = FIGURES_DIR / "cell6_training_analysis"

# Create the output directories if they do not already exist.
# These folders will contain report-ready records and visualizations.
ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ANALYSIS_FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Define the actual fine-tuned model checkpoint discovered in Cell 5B.
# The model must be loaded from these trained weights rather than the original pretrained checkpoint.
BEST_CHECKPOINT = (
    TRAINING_DIR
    / "checkpoint_best_total.pth"
)

# Confirm that the trained checkpoint exists in Google Drive.
# A missing checkpoint means the previous training outputs must be located before proceeding.
assert BEST_CHECKPOINT.is_file(), (
    f"Trained RF-DETR checkpoint not found: {BEST_CHECKPOINT}"
)

# Confirm that the checkpoint file contains data.
# This check helps detect empty or incomplete saved files.
checkpoint_size_bytes = BEST_CHECKPOINT.stat().st_size

assert checkpoint_size_bytes > 1024, (
    "The RF-DETR checkpoint is unexpectedly small."
)

# Calculate the size of the checkpoint in megabytes.
# This value will be recorded as part of the trained model documentation.
checkpoint_size_mb = (
    checkpoint_size_bytes / (1024 ** 2)
)

# Display the checkpoint and software information.
# These details verify the file that will be used for the project's final evaluation.
print("RF-DETR CHECKPOINT VERIFICATION")
print("=" * 70)

print("Model:", "RF-DETR Nano")
print("Checkpoint:", BEST_CHECKPOINT)
print("Checkpoint Size (MB):", round(checkpoint_size_mb, 2))
print("RF-DETR Version:", version("rfdetr"))
print("PyTorch Version:", torch.__version__)

# Select the available device for restoring the trained model.
# A GPU is preferred for evaluating transformer-based object detection models.
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Display the selected model evaluation device.
# This confirms whether CUDA acceleration is available for later test inference.
print("\nCOMPUTING DEVICE")
print("=" * 70)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        )
    )

# Load the saved checkpoint file into CPU memory for structural verification.
# The weights_only option restricts deserialization to supported tensor and simple data structures.
print("\nINSPECTING SAVED CHECKPOINT")
print("=" * 70)

checkpoint_data = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

# Confirm that the checkpoint contains a dictionary-like structure.
# RF-DETR model files typically contain either a parameter dictionary or a wrapped model state dictionary.
assert isinstance(checkpoint_data, dict), (
    "The saved checkpoint is not a dictionary."
)

# Display the available top-level keys from the checkpoint.
# These fields help identify how the model parameters were stored.
checkpoint_keys = list(checkpoint_data.keys())

print("Checkpoint Keys:")
print(checkpoint_keys[:30])

# Identify common containers used for saved model parameters.
# Different RF-DETR versions may store weights directly or inside a named state dictionary.
possible_state_keys = [
    "model",
    "state_dict",
    "model_state_dict"
]

# Search for a dictionary containing model parameter tensors.
# The code does not assume that every checkpoint uses the same internal key.
model_state = None

for key in possible_state_keys:
    candidate = checkpoint_data.get(key)

    if isinstance(candidate, dict):
        model_state = candidate
        break

# Use the top-level checkpoint dictionary when no nested state dictionary is present.
# This supports ordinary PyTorch checkpoints that save parameter names directly.
if model_state is None:
    model_state = checkpoint_data

# Count the tensor values available in the identified model state.
# At least one parameter tensor must exist for this file to qualify as a model checkpoint.
parameter_tensor_count = sum(
    isinstance(value, torch.Tensor)
    for value in model_state.values()
)

assert parameter_tensor_count > 0, (
    "No model parameter tensors were found in the saved checkpoint."
)

# Count the total number of tensor elements stored in the model state.
# This provides a general checkpoint complexity indicator, not necessarily the exact trainable parameter count.
stored_tensor_elements = sum(
    value.numel()
    for value in model_state.values()
    if isinstance(value, torch.Tensor)
)

# Display the validated checkpoint contents.
# This confirms that the file is readable and contains learned parameter tensors.
print("\nCHECKPOINT CONTENT SUMMARY")
print("=" * 70)

print("Parameter Tensors:", parameter_tensor_count)
print("Stored Tensor Elements:", stored_tensor_elements)
print("Checkpoint Structure:", "PASSED")

# Release the raw checkpoint dictionary to reduce memory use.
# The official RF-DETR constructor will read the checkpoint again during restoration.
del checkpoint_data
del model_state

gc.collect()

# Clear unused GPU memory before restoring the trained model.
# This helps prevent unnecessary memory pressure from earlier Colab operations.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Initialize RF-DETR Nano using the saved fine-tuned checkpoint.
# Passing pretrain_weights points the model to the actual trained weights rather than the original pretrained model.
print("\nRESTORING FINE-TUNED RF-DETR NANO")
print("=" * 70)

trained_model = RFDETRNano(
    pretrain_weights=str(BEST_CHECKPOINT)
)

# Confirm that model restoration returned an RF-DETR model instance.
# Successful initialization indicates that the saved weights are compatible with the model constructor.
assert trained_model is not None

print("Restored Model:", type(trained_model).__name__)
print("Checkpoint Loaded From:", BEST_CHECKPOINT)
print("Model Restoration: PASSED")

# Locate all saved model checkpoints inside the original training directory.
# Both .pth and .ckpt files are inventoried because they can serve different recovery purposes.
checkpoint_files = sorted([
    path
    for path in TRAINING_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".pth", ".ckpt"}
])

# Locate CSV files that might contain epoch-level training or validation metrics.
# Logger filenames may vary, so the code searches all CSV files before inspecting their columns.
csv_files = sorted([
    path
    for path in TRAINING_DIR.rglob("*.csv")
    if path.is_file()
])

# Locate JSON files that may describe training configuration or history.
# These records can support documentation of the optimization procedure.
json_files = sorted([
    path
    for path in TRAINING_DIR.rglob("*.json")
    if path.is_file()
])

# Locate event files generated by TensorBoard-style training loggers.
# These files may contain metrics even when no readable CSV history was generated.
tensorboard_files = sorted([
    path
    for path in TRAINING_DIR.rglob("*")
    if path.is_file()
    and "tfevents" in path.name.lower()
])

# Display the discovered training artifacts.
# This identifies what recorded evidence is available for analyzing model optimization.
print("\nTRAINING ARTIFACT DISCOVERY")
print("=" * 70)

print("Checkpoint Files:", len(checkpoint_files))
print("CSV Files:", len(csv_files))
print("JSON Files:", len(json_files))
print("TensorBoard Event Files:", len(tensorboard_files))

# Display the saved checkpoint filenames.
# These records may include periodic checkpoints and the validation-selected best model.
print("\nAVAILABLE CHECKPOINTS")
print("-" * 70)

for path in checkpoint_files:
    print(path.relative_to(TRAINING_DIR))

# Display any discovered CSV metric or summary files.
# Their contents will be inspected rather than assuming a specific training logger format.
print("\nAVAILABLE CSV LOGS")
print("-" * 70)

if csv_files:
    for path in csv_files[:30]:
        print(path.relative_to(TRAINING_DIR))
else:
    print("No CSV logs were found.")

# Display TensorBoard event files when available.
# Event logs will be parsed if no suitable CSV metrics can be recovered.
print("\nAVAILABLE TENSORBOARD LOGS")
print("-" * 70)

if tensorboard_files:
    for path in tensorboard_files[:20]:
        print(path.relative_to(TRAINING_DIR))
else:
    print("No TensorBoard event files were found.")

# Create a structured inventory covering all discovered artifacts.
# This provides an auditable file listing for the final RF-DETR report.
artifact_files = sorted(
    set(
        checkpoint_files
        + csv_files
        + json_files
        + tensorboard_files
    )
)

artifact_records = []

for path in artifact_files:

    # Determine the artifact type based on the filename and extension.
    # This distinguishes model checkpoints from numerical logs and configuration files.
    if path.suffix.lower() in {".pth", ".ckpt"}:
        artifact_type = "Checkpoint"
    elif "tfevents" in path.name.lower():
        artifact_type = "TensorBoard Event"
    elif path.suffix.lower() == ".csv":
        artifact_type = "CSV Log"
    else:
        artifact_type = "JSON Metadata"

    # Record file information without reading or modifying the artifact.
    # The full path allows training records to be traced back to Google Drive.
    artifact_records.append({
        "Type": artifact_type,
        "Filename": path.name,
        "Relative_Path": str(path.relative_to(TRAINING_DIR)),
        "Full_Path": str(path),
        "Size_MB": round(
            path.stat().st_size / (1024 ** 2),
            4
        )
    })

# Convert the artifact records into a DataFrame.
# The result can be exported and used as evidence of completed model training.
artifact_df = pd.DataFrame(
    artifact_records,
    columns=[
        "Type",
        "Filename",
        "Relative_Path",
        "Full_Path",
        "Size_MB"
    ]
)

# Save the complete training artifact inventory.
# This file will help assemble the training methodology and optimization evidence.
ARTIFACT_CSV = (
    ANALYSIS_DIR
    / "cell6_training_artifact_inventory.csv"
)

artifact_df.to_csv(
    ARTIFACT_CSV,
    index=False
)

# Initialize the extracted training metrics record.
# The program will use actual logged values and will not invent missing training losses or validation scores.
metrics_df = None
metrics_source = ""

# Search through the available CSV files for numerical training information.
# The first suitable history will be used if it includes loss or mAP-related columns.
for csv_path in csv_files:

    # Attempt to read the current CSV log.
    # Unreadable files are skipped with a diagnostic message rather than terminating model verification.
    try:
        candidate_df = pd.read_csv(
            csv_path
        )
    except Exception as error:
        print(
            f"Skipping unreadable CSV {csv_path.name}: {error}"
        )
        continue

    # Check whether the CSV contains any numerical loss or mAP measurements.
    # Files that only list checkpoint names or configuration values are not training histories.
    metric_columns = [
        column
        for column in candidate_df.columns
        if (
            "loss" in str(column).lower()
            or "map" in str(column).lower()
        )
        and pd.to_numeric(
            candidate_df[column],
            errors="coerce"
        ).notna().any()
    ]

    # Select the first readable CSV containing actual metric values.
    # Its original measurements will be preserved in the exported analysis record.
    if metric_columns:
        metrics_df = candidate_df.copy()
        metrics_source = str(csv_path)
        break

# Attempt to extract TensorBoard scalar metrics when no suitable CSV is available.
# TensorBoard event files may preserve epoch-level values from the RF-DETR training logger.
if metrics_df is None and tensorboard_files:

    # Import TensorBoard's event accumulator for reading logged scalar values.
    # The required package is typically installed alongside the RF-DETR logging dependencies.
    try:
        from tensorboard.backend.event_processing.event_accumulator import (
            EventAccumulator
        )

        # Combine the recorded scalar values into a common step-indexed table.
        # Each recorded scalar retains its original tag rather than being renamed to an assumed metric.
        scalar_records = []

        for event_path in tensorboard_files:

            # Load the event file and inspect its available scalar tags.
            # Some event files may contain no scalar values, which is not treated as an error.
            event_accumulator = EventAccumulator(
                str(event_path),
                size_guidance={"scalars": 0}
            )

            event_accumulator.Reload()
            tags = event_accumulator.Tags().get("scalars", [])

            # Extract numerical values from relevant training and validation metric tags.
            # Only logged loss or mAP values are selected for the training analysis.
            for tag in tags:
                if not (
                    "loss" in tag.lower()
                    or "map" in tag.lower()
                ):
                    continue

                for event in event_accumulator.Scalars(tag):
                    scalar_records.append({
                        "Step": event.step,
                        "Metric": tag,
                        "Value": event.value
                    })

        # Convert the recovered scalar records into a table.
        # The result will provide a usable numerical history when the event files contain suitable values.
        if scalar_records:
            scalar_df = pd.DataFrame(
                scalar_records
            )

            metrics_df = scalar_df.pivot_table(
                index="Step",
                columns="Metric",
                values="Value",
                aggfunc="last"
            ).reset_index()

            metrics_df.columns.name = None
            metrics_source = "TensorBoard Event Logs"

    except Exception as error:
        print(
            "\nTensorBoard scalar extraction was unavailable:",
            error
        )

# Define output paths for any recovered numerical training history.
# These files will be useful for creating graphs and discussing training optimization.
METRICS_CSV = (
    ANALYSIS_DIR
    / "cell6_recovered_training_metrics.csv"
)

METRICS_FIGURE = (
    ANALYSIS_FIGURES_DIR
    / "cell6_training_metric_history.png"
)

# Generate results only when usable numerical metric records were found.
# If no log exists, the notebook will report that limitation instead of creating estimated curves.
if metrics_df is not None:

    # Save the extracted numerical metrics as a standalone CSV file.
    # This makes the real training history available outside the original logger directory.
    metrics_df.to_csv(
        METRICS_CSV,
        index=False
    )

    # Display the source and columns of the recovered training records.
    # The report can identify exactly where the numerical optimization evidence originated.
    print("\nRECOVERED TRAINING METRICS")
    print("=" * 70)
    print("Source:", metrics_source)
    print("Records:", len(metrics_df))
    print("Columns:", metrics_df.columns.tolist())

    print("\nLAST RECORDED METRICS")
    print("-" * 70)
    print(
        metrics_df.tail(12).to_string(index=False)
    )

    # Identify any numerical columns that contain recorded loss or mAP measurements.
    # These values will be plotted without assuming that their numerical scales are identical.
    plot_columns = [
        column
        for column in metrics_df.columns
        if (
            "loss" in str(column).lower()
            or "map" in str(column).lower()
        )
        and pd.to_numeric(
            metrics_df[column],
            errors="coerce"
        ).notna().any()
    ]

    # Identify a suitable epoch or training-step column for the horizontal axis.
    # A sequential record index is used only when the training logs provide no explicit step information.
    x_candidates = [
        column
        for column in metrics_df.columns
        if str(column).lower() in {
            "epoch",
            "trainer/epoch",
            "step",
            "global_step"
        }
    ]

    if x_candidates:
        x_values = pd.to_numeric(
            metrics_df[x_candidates[0]],
            errors="coerce"
        ).to_numpy()

        x_label = str(x_candidates[0])
    else:
        x_values = np.arange(
            len(metrics_df)
        )

        x_label = "Logged Record"

    # Generate separate loss and mAP panels when those metrics exist.
    # Separate vertical scales avoid misleading comparisons between losses and performance percentages.
    loss_columns = [
        column
        for column in plot_columns
        if "loss" in str(column).lower()
    ]

    map_columns = [
        column
        for column in plot_columns
        if "map" in str(column).lower()
    ]

    # Create a training performance figure containing the available metric groups.
    # Each panel uses only actual measurements recovered from the RF-DETR training logs.
    available_groups = [
        ("Loss", loss_columns),
        ("mAP", map_columns)
    ]

    available_groups = [
        group
        for group in available_groups
        if group[1]
    ]

    if available_groups:

        # Create one plot for each metric group.
        # Separate plots make it easier to interpret changes during model optimization.
        fig, axes = plt.subplots(
            len(available_groups),
            1,
            figsize=(12, 5 * len(available_groups)),
            squeeze=False
        )

        for group_index, (group_name, columns) in enumerate(
            available_groups
        ):
            ax = axes[group_index, 0]

            # Plot up to eight recorded metrics in the current group.
            # All values are taken directly from the available training history.
            for column in columns[:8]:
                y_values = pd.to_numeric(
                    metrics_df[column],
                    errors="coerce"
                )

                ax.plot(
                    x_values,
                    y_values,
                    marker=".",
                    label=str(column)
                )

            # Label the chart according to its actual metric type.
            # This keeps loss values and evaluation measures clearly distinguished.
            ax.set_title(
                f"RF-DETR Recorded {group_name} History"
            )

            ax.set_xlabel(
                x_label
            )

            ax.set_ylabel(
                group_name
            )

            ax.legend(
                fontsize=8
            )

            ax.grid(
                alpha=0.25
            )

        # Save the resulting figure for the final RF-DETR report.
        # The figure will not be generated when appropriate training metrics are unavailable.
        plt.tight_layout()

        plt.savefig(
            METRICS_FIGURE,
            dpi=200,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)

        print(
            "\nTraining Metrics Figure:",
            METRICS_FIGURE
        )

else:

    # State clearly that numerical training histories were not found.
    # A loadable checkpoint alone does not establish the values of training loss or validation mAP.
    print("\nTRAINING METRIC HISTORY")
    print("=" * 70)
    print("No usable training loss or mAP history was found.")
    print("The trained checkpoint was verified successfully.")
    print("Training graphs cannot be generated without actual logged measurements.")

# Create a structured summary of the verified trained checkpoint.
# This metadata identifies the exact weights to be used during final test evaluation.
checkpoint_summary = {
    "Model": "RF-DETR Nano",
    "Checkpoint_Path": str(BEST_CHECKPOINT),
    "Checkpoint_Size_MB": round(
        checkpoint_size_mb,
        3
    ),
    "Parameter_Tensors": parameter_tensor_count,
    "Stored_Tensor_Elements": stored_tensor_elements,
    "RFDETR_Version": version("rfdetr"),
    "PyTorch_Version": torch.__version__,
    "Device": str(DEVICE),
    "Checkpoint_Readable": True,
    "Model_Restored": True,
    "Training_Metrics_Available": metrics_df is not None,
    "Training_Metrics_Source": metrics_source
}

# Save the checkpoint summary as a CSV file.
# This report-ready record confirms the source of the fine-tuned RF-DETR evaluation model.
CHECKPOINT_SUMMARY_CSV = (
    ANALYSIS_DIR
    / "cell6_verified_rfdetr_checkpoint.csv"
)

pd.DataFrame(
    [checkpoint_summary]
).to_csv(
    CHECKPOINT_SUMMARY_CSV,
    index=False
)

# Save the verified trained checkpoint path as a JSON reference.
# Later notebook cells can load this file without relying on variables from the current runtime.
CHECKPOINT_REFERENCE_JSON = (
    ANALYSIS_DIR
    / "cell6_best_checkpoint_reference.json"
)

with open(
    CHECKPOINT_REFERENCE_JSON,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            "model": "RFDETRNano",
            "checkpoint_path": str(BEST_CHECKPOINT),
            "checkpoint_verified": True
        },
        file,
        indent=2
    )

# Confirm that the checkpoint and saved analysis records remain accessible.
# This final verification ensures that Cell 7 can locate the fine-tuned model.
assert BEST_CHECKPOINT.is_file()
assert CHECKPOINT_SUMMARY_CSV.is_file()
assert CHECKPOINT_REFERENCE_JSON.is_file()
assert ARTIFACT_CSV.is_file()

# Display the completed model verification summary.
# The next stage will run independent detection evaluation using the held-out COCO test set.
print("\nFINAL RF-DETR CHECKPOINT VERIFICATION")
print("=" * 70)

print("Model:", "RF-DETR Nano")
print("Checkpoint Readable:", True)
print("Fine-Tuned Model Restored:", True)
print("Parameter Tensors:", parameter_tensor_count)
print("Available Checkpoints:", len(checkpoint_files))
print("Training Metrics Available:", metrics_df is not None)

# Display the saved results for report and evaluation preparation.
# All original training and annotation files remain unchanged.
print("\nCELL 6 OUTPUT FILES")
print("=" * 70)

print("Best Checkpoint:", BEST_CHECKPOINT)
print("Checkpoint Summary:", CHECKPOINT_SUMMARY_CSV)
print("Training Artifact Inventory:", ARTIFACT_CSV)
print("Checkpoint Reference:", CHECKPOINT_REFERENCE_JSON)

if metrics_df is not None:
    print("Recovered Training Metrics:", METRICS_CSV)

# Confirm successful completion of model checkpoint verification.
# The next cell will compute detection metrics from actual predictions on the reserved test images.
print("\nCELL 6 COMPLETED SUCCESSFULLY!")

### 7. Final RF-DETR Test Evaluation and Detection Visualization

This section evaluates the fine-tuned RF-DETR Nano model using the independent COCO test dataset. Object predictions are compared against manually verified pedestrian ground-truth annotations to measure detection accuracy, precision, recall, F1-score, Intersection over Union (IoU), and mean Average Precision (mAP).

True-positive, false-positive, and false-negative counts are calculated using confidence-filtered detections and one-to-one matching at an IoU threshold of 0.50. COCO average precision is evaluated separately using the confidence-ranked detection results, while matched-object classification accuracy is reported with the limitation that the dataset contains only one object class.

Representative test images are visualized with ground-truth and predicted pedestrian bounding boxes to support interpretation of model performance and identification of missed detections or false positives. All metrics are calculated from the trained model's actual predictions and saved for the final project report.

In [ ]:

# Import libraries for reading the test dataset and evaluating object detections.
# The code uses the saved fine-tuned checkpoint and does not repeat model training.
from pathlib import Path
from collections import defaultdict
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image
from scipy.optimize import linear_sum_assignment
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from rfdetr import RFDETRNano

# Define the RF-DETR dataset, checkpoint, and evaluation output directories.
# The held-out COCO test split is independent of the training and validation images.
PROJECT_DIR = Path("/content/drive/MyDrive/Person_Detection")
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"
COCO_DIR = RFDETR_DIR / "dataset" / "rfdetr_coco"
MODEL_DIR = RFDETR_DIR / "trained_models"
RESULTS_DIR = RFDETR_DIR / "evaluation_results"
EVAL_DIR = RESULTS_DIR / "cell7_final_test_evaluation"
FIGURES_DIR = RFDETR_DIR / "figures" / "cell7_final_test_evaluation"

# Create separate folders for evaluation tables and prediction visualizations.
# Original dataset images and trained checkpoints are not changed by this procedure.
EVAL_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Define the trained model checkpoint selected during validation.
# This is the existing checkpoint verified in Cell 6.
BEST_CHECKPOINT = (
    MODEL_DIR
    / "rfdetr_nano_training"
    / "checkpoint_best_total.pth"
)

# Define the held-out COCO test annotation file.
# It contains the ground-truth pedestrian bounding boxes for final evaluation.
TEST_JSON = (
    COCO_DIR
    / "test"
    / "_annotations.coco.json"
)

# Confirm that the checkpoint and independent test annotations exist.
# Missing inputs must be fixed rather than replaced by estimated performance measurements.
assert BEST_CHECKPOINT.is_file(), (
    f"Missing trained checkpoint: {BEST_CHECKPOINT}"
)

assert TEST_JSON.is_file(), (
    f"Missing test annotations: {TEST_JSON}"
)

# Define a confidence threshold and IoU matching threshold for operating-point metrics.
# COCO mAP will use a separate, much lower confidence cutoff to retain ranked predictions.
CONFIDENCE_THRESHOLD = 0.25
MAP_PREDICTION_THRESHOLD = 0.001
IOU_THRESHOLD = 0.50

# Load the COCO test annotation file using the official COCO API.
# This provides consistent image identifiers, category definitions, and bounding-box references.
coco_gt = COCO(str(TEST_JSON))
image_ids = sorted(coco_gt.getImgIds())

# Verify that the test set contains images and the single pedestrian category.
# This experiment must not accidentally evaluate a different dataset or object class.
assert len(image_ids) > 0
assert coco_gt.getCatIds() == [1]
assert coco_gt.cats[1]["name"] == "person"

# Initialize the fine-tuned RF-DETR Nano detector using the saved checkpoint.
# Predictions will come from the trained pedestrian model, not the original pretrained detector.
print("LOADING FINE-TUNED RF-DETR NANO")
print("=" * 65)

model = RFDETRNano(
    pretrain_weights=str(BEST_CHECKPOINT)
)

print("Model:", type(model).__name__)
print("Checkpoint:", BEST_CHECKPOINT)
print("Test Images:", len(image_ids))
print("Operating Confidence:", CONFIDENCE_THRESHOLD)
print("IoU Matching Threshold:", IOU_THRESHOLD)

# Define a bounding-box IoU function using XYXY pixel coordinates.
# Intersection over Union measures overlap between a predicted box and a ground-truth box.
def calculate_iou(box_a, box_b):

    # Calculate the overlapping rectangle coordinates.
    # Negative overlap dimensions are clipped to zero because disjoint boxes have no intersection.
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    intersection = (
        max(0.0, x2 - x1)
        * max(0.0, y2 - y1)
    )

    # Calculate the areas of both original rectangles.
    # The union is their combined area minus the overlapping area.
    area_a = max(0.0, box_a[2] - box_a[0]) * max(
        0.0, box_a[3] - box_a[1]
    )
    area_b = max(0.0, box_b[2] - box_b[0]) * max(
        0.0, box_b[3] - box_b[1]
    )

    union = area_a + area_b - intersection

    # Return zero when the union has no positive area.
    # Otherwise return the proportion of the combined area that overlaps.
    return intersection / union if union > 0 else 0.0

# Define a function that converts each COCO ground-truth box into XYXY coordinates.
# RF-DETR predictions already use XYXY, so both formats must agree before matching.
def coco_gt_boxes(image_id):
    annotations = coco_gt.loadAnns(
        coco_gt.getAnnIds(imgIds=[int(image_id)])
    )

    boxes = []
    classes = []

    for annotation in annotations:

        # Convert COCO [x, y, width, height] to [x1, y1, x2, y2].
        # This makes IoU calculations consistent with the model prediction coordinates.
        x, y, width, height = annotation["bbox"]

        boxes.append([
            float(x),
            float(y),
            float(x + width),
            float(y + height)
        ])

        classes.append(int(annotation["category_id"]))

    return boxes, classes

# Define confidence-ranked one-to-one matching for object detection evaluation.
# A prediction counts as TP only when its class matches and IoU is at least 0.50.
def match_detections(predictions, gt_boxes, threshold=0.5):

    # Sort predictions by confidence so that stronger detections are considered first.
    # Each ground-truth pedestrian is allowed to match only one prediction.
    sorted_predictions = sorted(
        predictions,
        key=lambda item: item["score"],
        reverse=True
    )

    matched_gt = set()
    matched_iou = []
    tp = 0
    fp = 0

    for prediction in sorted_predictions:
        best_iou = 0.0
        best_gt_index = None

        # Find the highest-IoU unmatched ground-truth pedestrian.
        # Only the declared person category is accepted as a correct class.
        for gt_index, gt_box in enumerate(gt_boxes):
            if gt_index in matched_gt:
                continue

            overlap = calculate_iou(
                prediction["xyxy"],
                gt_box
            )

            if overlap > best_iou:
                best_iou = overlap
                best_gt_index = gt_index

        # Count a true positive when a valid one-to-one match exists.
        # All other confidence-filtered detections count as false positives.
        if (
            prediction["category_id"] == 1
            and best_gt_index is not None
            and best_iou >= threshold
        ):
            tp += 1
            matched_gt.add(best_gt_index)
            matched_iou.append(best_iou)
        else:
            fp += 1

    # Ground-truth pedestrians left unmatched are false negatives.
    # These represent objects missed at the selected confidence and IoU thresholds.
    fn = len(gt_boxes) - len(matched_gt)

    return tp, fp, fn, matched_iou

# Initialize collections for COCO evaluation predictions and per-image results.
# These results are calculated directly from the held-out test images.
coco_predictions = []
image_results = []
prediction_records = []
visualization_records = []

# Load and evaluate each test image using the fine-tuned detector.
# A low prediction threshold retains confidence-ranked detections for standard COCO mAP evaluation.
print("\nRUNNING FINAL RF-DETR TEST INFERENCE")
print("=" * 65)

for image_number, image_id in enumerate(image_ids, start=1):

    # Retrieve the COCO metadata and actual test image path.
    # The original pixel dimensions are preserved for IoU calculations and visualization.
    image_info = coco_gt.loadImgs([int(image_id)])[0]
    image_path = COCO_DIR / "test" / image_info["file_name"]

    assert image_path.is_file(), (
        f"Missing test image: {image_path}"
    )

    # Load the image using RGB color channels.
    # The official RF-DETR prediction interface accepts PIL images.
    with Image.open(image_path) as image:
        image_rgb = image.convert("RGB").copy()

    # Execute inference and record the elapsed prediction time.
    # This timing is descriptive and includes the predict call, not complete deployment overhead.
    start_time = time.perf_counter()

    with torch.inference_mode():
        detections = model.predict(
            image_rgb,
            threshold=MAP_PREDICTION_THRESHOLD
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    inference_seconds = time.perf_counter() - start_time

    # Retrieve the predicted bounding boxes, scores, and class identifiers.
    # RF-DETR returns supervision-style detections with image-space XYXY coordinates.
    predicted_boxes = np.asarray(
        detections.xyxy, dtype=float
    ).reshape(-1, 4)

    predicted_scores = np.asarray(
        detections.confidence, dtype=float
    ).reshape(-1)

    # Use a single-person-class identifier for every detection only when supported.
    # A multi-class output must not be silently relabeled as pedestrian predictions.
    raw_classes = np.asarray(
        detections.class_id, dtype=int
    ).reshape(-1)

    assert len(predicted_boxes) == len(predicted_scores)
    assert len(raw_classes) == len(predicted_scores)

    if len(raw_classes):
        assert set(raw_classes.tolist()).issubset({0, 1}), (
            f"Unexpected prediction class IDs: {set(raw_classes.tolist())}. "
            "Inspect the fine-tuned checkpoint class mapping."
        )

    # Convert the supported one-class identifiers to COCO person category ID 1.
    # The zero and one conventions can vary between model output and COCO annotations.
    prediction_list = []

    for box, score, class_id in zip(
        predicted_boxes,
        predicted_scores,
        raw_classes
    ):

        # Reject invalid coordinate or confidence values.
        # Only finite, positive-area bounding boxes can contribute to reliable evaluation.
        if not (
            np.isfinite(box).all()
            and np.isfinite(score)
            and 0 <= score <= 1
        ):
            raise ValueError(
                f"Non-finite model prediction for {image_info['file_name']}"
            )

        x1, y1, x2, y2 = [
            float(value) for value in box
        ]

        # Clip model predictions to the image boundary.
        # This prevents invalid rectangles from entering COCO evaluation.
        x1 = max(0.0, min(image_rgb.width, x1))
        x2 = max(0.0, min(image_rgb.width, x2))
        y1 = max(0.0, min(image_rgb.height, y1))
        y2 = max(0.0, min(image_rgb.height, y2))

        if x2 <= x1 or y2 <= y1:
            continue

        prediction = {
            "xyxy": [x1, y1, x2, y2],
            "score": float(score),
            "category_id": 1
        }

        prediction_list.append(prediction)

        # Record COCO-compatible detections at the low threshold for AP computation.
        # COCO evaluates ranked predictions across recall levels and IoU thresholds.
        coco_predictions.append({
            "image_id": int(image_id),
            "category_id": 1,
            "bbox": [
                x1,
                y1,
                x2 - x1,
                y2 - y1
            ],
            "score": float(score)
        })

        # Save individual prediction records for later error analysis.
        # These records allow detections to be traced back to their original test images.
        prediction_records.append({
            "Image_ID": int(image_id),
            "Image": image_info["file_name"],
            "Confidence": float(score),
            "X1": x1,
            "Y1": y1,
            "X2": x2,
            "Y2": y2,
            "Class": "person"
        })

    # Obtain the verified pedestrian ground truth for the current image.
    # These annotations were exported from the reviewed CO1.3 labels.
    gt_boxes, gt_classes = coco_gt_boxes(image_id)

    # Apply the operating confidence threshold for TP, FP, and FN calculation.
    # The threshold is separate from the low threshold used for mAP.
    selected_predictions = [
        item for item in prediction_list
        if item["score"] >= CONFIDENCE_THRESHOLD
    ]

    # Match confident predictions against ground truth using IoU of at least 0.50.
    # This produces the detection counts used to calculate precision and recall.
    tp, fp, fn, matched_ious = match_detections(
        selected_predictions,
        gt_boxes,
        threshold=IOU_THRESHOLD
    )

    # Use Hungarian assignment for the geometric matched-object classification calculation.
    # This calculation is distinct from the confidence-ranked matching used for detection metrics.
    geometric_matches = 0
    correctly_classified_matches = 0

    if selected_predictions and gt_boxes:

        # Construct an IoU matrix for all predictions and reference objects.
        # The Hungarian algorithm finds a one-to-one assignment maximizing total overlap.
        iou_matrix = np.array([
            [
                calculate_iou(
                    prediction["xyxy"],
                    gt_box
                )
                for gt_box in gt_boxes
            ]
            for prediction in selected_predictions
        ], dtype=float)

        # Perform the optimal assignment using negative IoU as the cost.
        # Only assigned pairs with IoU at least 0.50 are counted as localized matches.
        pred_indices, gt_indices = linear_sum_assignment(
            -iou_matrix
        )

        for pred_index, gt_index in zip(
            pred_indices,
            gt_indices
        ):
            if iou_matrix[pred_index, gt_index] >= IOU_THRESHOLD:
                geometric_matches += 1

                if (
                    selected_predictions[pred_index]["category_id"]
                    == gt_classes[gt_index]
                ):
                    correctly_classified_matches += 1

    # Save image-level performance results for detailed error analysis.
    # These measurements allow us to compare successes and failures under different lighting conditions.
    image_results.append({
        "Image_ID": int(image_id),
        "Image": image_info["file_name"],
        "Ground_Truth_Objects": len(gt_boxes),
        "Predicted_Objects": len(selected_predictions),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Mean_Matched_IoU": (
            float(np.mean(matched_ious))
            if matched_ious else np.nan
        ),
        "Geometric_Matches": geometric_matches,
        "Correct_Class_Matches": correctly_classified_matches,
        "Inference_Seconds": inference_seconds
    })

    # Select representative images for the report visualizations.
    # Two examples per lighting category will show ground truth and model predictions.
    filename_lower = image_info["file_name"].lower()

    condition = next(
        (
            name
            for name in ["mild", "moderate", "severe"]
            if filename_lower.endswith(f"_{name}{image_path.suffix.lower()}")
        ),
        "unknown"
    )

    visualization_records.append({
        "Image_ID": int(image_id),
        "Image_Path": image_path,
        "Condition": condition,
        "GT_Boxes": gt_boxes,
        "Predictions": selected_predictions,
        "TP": tp,
        "FP": fp,
        "FN": fn
    })

    # Display progress throughout the held-out test evaluation.
    # This confirms the number of processed images without claiming results before completion.
    if image_number % 10 == 0 or image_number == len(image_ids):
        print(
            f"Evaluated {image_number}/{len(image_ids)} test images"
        )

# Save the COCO prediction records for reproducible mAP evaluation.
# No detections are fabricated if the model predicts zero objects.
PREDICTIONS_JSON = EVAL_DIR / "cell7_coco_test_predictions.json"

with open(PREDICTIONS_JSON, "w", encoding="utf-8") as file:
    json.dump(coco_predictions, file, indent=2)

# Save all image-level and individual detection results.
# The CSV files will support final performance tables and error analysis.
image_results_df = pd.DataFrame(image_results)
prediction_records_df = pd.DataFrame(
    prediction_records,
    columns=[
        "Image_ID", "Image", "Confidence",
        "X1", "Y1", "X2", "Y2", "Class"
    ]
)

IMAGE_RESULTS_CSV = EVAL_DIR / "cell7_per_image_detection_results.csv"
PREDICTIONS_CSV = EVAL_DIR / "cell7_individual_predictions.csv"

image_results_df.to_csv(IMAGE_RESULTS_CSV, index=False)
prediction_records_df.to_csv(PREDICTIONS_CSV, index=False)

# Calculate dataset-wide true positives, false positives, and false negatives.
# These totals use one-to-one IoU matching on confidence-filtered predictions.
TP = int(image_results_df["TP"].sum())
FP = int(image_results_df["FP"].sum())
FN = int(image_results_df["FN"].sum())

# Calculate overall object detection precision and recall.
# Precision measures prediction correctness, while recall measures how many ground-truth objects were found.
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0

# Calculate the harmonic mean of precision and recall.
# F1-score summarizes the balance between false positives and missed pedestrians.
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0.0
)

# Calculate detection accuracy using the TP, FP, and FN formulation.
# This is a detection-focused Jaccard-style accuracy and is not ordinary classification accuracy.
detection_accuracy = (
    TP / (TP + FP + FN)
    if (TP + FP + FN) > 0
    else 0.0
)

# Calculate matched-object classification accuracy using the Hungarian matching results.
# With only the person category, this value is not a meaningful measure of multi-class discrimination.
geometric_matches = int(
    image_results_df["Geometric_Matches"].sum()
)

correct_class_matches = int(
    image_results_df["Correct_Class_Matches"].sum()
)

matched_class_accuracy = (
    correct_class_matches / geometric_matches
    if geometric_matches > 0
    else np.nan
)

# Calculate the mean IoU of successful confidence-ranked detection matches.
# This measures localization quality only for predictions that achieved a valid match.
matched_iou_values = (
    image_results_df["Mean_Matched_IoU"]
    .dropna()
    .to_numpy()
)

# Report the image-averaged matched IoU separately from standard COCO metrics.
# A per-image average weights images equally, not individual matched boxes.
mean_image_matched_iou = (
    float(np.mean(matched_iou_values))
    if len(matched_iou_values)
    else np.nan
)

# Prepare a COCO detection-results object for standard mAP evaluation.
# When the model predicts no objects, an empty result dataset is created safely.
if coco_predictions:
    coco_dt = coco_gt.loadRes(
        str(PREDICTIONS_JSON)
    )
else:
    coco_dt = COCO()
    coco_dt.dataset = {
        "images": list(coco_gt.dataset["images"]),
        "categories": list(coco_gt.dataset["categories"]),
        "annotations": []
    }
    coco_dt.createIndex()

# Evaluate the confidence-ranked detections using the standard COCO bounding-box protocol.
# The default IoU thresholds range from 0.50 to 0.95 in increments of 0.05.
print("\nSTANDARD COCO OBJECT DETECTION EVALUATION")
print("=" * 65)

coco_evaluator = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)

# Restrict the evaluation to the reserved test images and the person category.
# This prevents results from unrelated images or categories from entering the final scores.
coco_evaluator.params.imgIds = image_ids
coco_evaluator.params.catIds = [1]

# Run COCO per-image matching and aggregate precision-recall measurements.
# This produces the standard Average Precision and Average Recall metrics.
coco_evaluator.evaluate()
coco_evaluator.accumulate()
coco_evaluator.summarize()

# Extract standard COCO mAP measurements after evaluation.
# Index zero is AP@[0.50:0.95] and index one is AP@0.50.
MAP_50_95 = float(coco_evaluator.stats[0])
MAP_50 = float(coco_evaluator.stats[1])

# Assemble the final object detection evaluation results.
# Every numerical value is calculated from the actual fine-tuned model predictions.
overall_results = pd.DataFrame([{
    "Model": "RF-DETR Nano",
    "Test_Images": len(image_ids),
    "Ground_Truth_Objects": TP + FN,
    "Predicted_Objects_at_Confidence": TP + FP,
    "TP": TP,
    "FP": FP,
    "FN": FN,
    "Confidence_Threshold": CONFIDENCE_THRESHOLD,
    "IoU_Threshold": IOU_THRESHOLD,
    "Detection_Accuracy_Percent": round(detection_accuracy * 100, 3),
    "Precision_Percent": round(precision * 100, 3),
    "Recall_Percent": round(recall * 100, 3),
    "F1_Percent": round(f1 * 100, 3),
    "mAP50_Percent": round(MAP_50 * 100, 3),
    "mAP50_95_Percent": round(MAP_50_95 * 100, 3),
    "Hungarian_Geometric_Matches": geometric_matches,
    "Hungarian_Correct_Class_Matches": correct_class_matches,
    "Matched_Object_Class_Accuracy_Percent": (
        round(matched_class_accuracy * 100, 3)
        if np.isfinite(matched_class_accuracy)
        else np.nan
    ),
    "Image_Averaged_Matched_IoU": mean_image_matched_iou,
    "Mean_Inference_Seconds": float(
        image_results_df["Inference_Seconds"].mean()
    )
}])

# Save the final numerical results table to Google Drive.
# These measurements will be used in the final project report without manually retyping values.
OVERALL_CSV = EVAL_DIR / "cell7_rfdetr_final_test_metrics.csv"

overall_results.to_csv(
    OVERALL_CSV,
    index=False
)

# Display the complete detection evaluation metrics.
# Percentage values are reported with clear distinctions between detection and classification measures.
print("\nFINAL RF-DETR TEST PERFORMANCE")
print("=" * 65)
print(overall_results.to_string(index=False))

# Create representative visualization figures for each lighting condition.
# These figures compare ground truth against predictions without altering the image dataset.
selected_visualizations = []

for condition in ["mild", "moderate", "severe"]:
    candidates = [
        record
        for record in visualization_records
        if record["Condition"] == condition
    ]

    selected_visualizations.extend(
        candidates[:2]
    )

# Generate side-by-side ground-truth and prediction displays.
# Green boxes represent verified annotations, while red boxes represent RF-DETR detections.
for figure_number, record in enumerate(
    selected_visualizations,
    start=1
):

    # Load the selected test image for visual comparison.
    # Each panel uses identical pixels so differences are limited to the displayed bounding boxes.
    with Image.open(record["Image_Path"]) as image:
        image_rgb = image.convert("RGB").copy()

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 6)
    )

    # Display the ground-truth annotations in the first panel.
    # These are the reference pedestrian boxes exported from the reviewed dataset.
    axes[0].imshow(image_rgb)

    for gt_box in record["GT_Boxes"]:
        x1, y1, x2, y2 = gt_box

        rectangle = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )

        axes[0].add_patch(rectangle)

    axes[0].set_title(
        f"Ground Truth - {record['Condition'].title()}"
    )
    axes[0].axis("off")

    # Display RF-DETR predicted bounding boxes in the second panel.
    # Only predictions meeting the selected confidence threshold are shown.
    axes[1].imshow(image_rgb)

    for prediction in record["Predictions"]:
        x1, y1, x2, y2 = prediction["xyxy"]

        rectangle = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="red",
            facecolor="none"
        )

        axes[1].add_patch(rectangle)

        axes[1].text(
            x1,
            max(0, y1 - 5),
            f"{prediction['score']:.2f}",
            color="white",
            fontsize=9,
            bbox={
                "facecolor": "red",
                "alpha": 0.7
            }
        )

    # Add the actual detection counts to the prediction visualization.
    # These numbers help distinguish correct detections from false positives and missed objects.
    axes[1].set_title(
        f"RF-DETR | TP={record['TP']} "
        f"FP={record['FP']} FN={record['FN']}"
    )
    axes[1].axis("off")

    # Save the comparison image for the final RF-DETR results report.
    # Each output is generated from actual model inference and corrected ground truth.
    plt.tight_layout()

    FIGURE_PATH = (
        FIGURES_DIR
        / f"cell7_prediction_example_{figure_number:02d}.png"
    )

    plt.savefig(
        FIGURE_PATH,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

# Verify that every COCO test image was evaluated exactly once.
# The resulting metrics must not accidentally omit images or include duplicates.
assert len(image_results_df) == len(image_ids)

assert image_results_df["Image_ID"].is_unique

# Verify that the confusion counts are internally consistent.
# Every ground-truth object must be either matched or counted as missed.
assert TP + FN == int(
    image_results_df["Ground_Truth_Objects"].sum()
)

assert TP + FP == int(
    image_results_df["Predicted_Objects"].sum()
)

# Display the saved evaluation outputs and completion status.
# These files will be used for error analysis and final PDF report generation.
print("\nCELL 7 OUTPUT FILES")
print("=" * 65)
print("Final Test Metrics:", OVERALL_CSV)
print("Per-Image Results:", IMAGE_RESULTS_CSV)
print("Prediction Records:", PREDICTIONS_CSV)
print("COCO Prediction JSON:", PREDICTIONS_JSON)
print("Prediction Figures:", FIGURES_DIR)

# Confirm that the independent RF-DETR test evaluation completed successfully.
# The next cell will analyze model errors across lighting conditions.
print("\nCELL 7 COMPLETED SUCCESSFULLY!")

### 8. Error Analysis and Performance Comparison Across Low-Light Conditions

This section investigates the detection performance of the fine-tuned RF-DETR Nano model under mild, moderate, and severe low-light conditions. The evaluation uses the test predictions and corrected ground-truth annotations obtained in the previous stage to calculate lighting-specific true positives, false positives, false negatives, precision, recall, F1-score, and detection accuracy.

Error analysis identifies images with missed pedestrians, false-positive detections, and reduced localization quality. Performance differences across lighting conditions are examined to determine whether increasing darkness is associated with changes in detection reliability, while recognizing that the observed results may also be influenced by image content, object size, and scene complexity.

The findings are summarized in comparative tables, figures, and written discussions for inclusion in the final project report. Limitations of the dataset and evaluation methodology are acknowledged, and future improvements are proposed based on the measured results.

In [ ]:
# Import libraries for reading the recorded RF-DETR evaluation results.
# This cell analyzes saved predictions and does not repeat model training or inference.
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from contextlib import redirect_stdout
from io import StringIO

# Define the RF-DETR project and final evaluation directories.
# All analysis outputs will be saved separately from the existing dataset and trained checkpoint.
PROJECT_DIR = Path("/content/drive/MyDrive/Person_Detection")
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"
DATASET_DIR = RFDETR_DIR / "dataset"
COCO_DIR = DATASET_DIR / "rfdetr_coco"
RESULTS_DIR = RFDETR_DIR / "evaluation_results"
CELL7_DIR = RESULTS_DIR / "cell7_final_test_evaluation"
CELL8_DIR = RESULTS_DIR / "cell8_error_analysis"
FIGURES_DIR = RFDETR_DIR / "figures" / "cell8_error_analysis"

# Create the folders needed for analysis tables, charts, and error visualizations.
# The original COCO annotations and RF-DETR model checkpoints will remain unchanged.
CELL8_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Define the actual evaluation files produced in Cell 7.
# These files contain image-level detection counts and the confidence-ranked model predictions.
IMAGE_RESULTS_CSV = CELL7_DIR / "cell7_per_image_detection_results.csv"
PREDICTIONS_JSON = CELL7_DIR / "cell7_coco_test_predictions.json"
OVERALL_CSV = CELL7_DIR / "cell7_rfdetr_final_test_metrics.csv"

# Define the saved COCO test annotations and the source-grouped dataset manifest.
# The manifest identifies each test image's lighting condition and original source frame.
TEST_JSON = COCO_DIR / "test" / "_annotations.coco.json"
SPLIT_MANIFEST_CSV = DATASET_DIR / "cell3_coco_split_manifest.csv"

# Verify that all analysis inputs exist before beginning.
# Missing results must not be replaced by assumed values or regenerated predictions.
required_files = [
    IMAGE_RESULTS_CSV,
    PREDICTIONS_JSON,
    OVERALL_CSV,
    TEST_JSON,
    SPLIT_MANIFEST_CSV
]

for path in required_files:
    assert path.is_file(), (
        f"Required evaluation file not found: {path}"
    )

# Load the final RF-DETR evaluation tables and dataset split records.
# These contain the actual measured performance and ground-truth image metadata.
image_results = pd.read_csv(IMAGE_RESULTS_CSV)
overall_results = pd.read_csv(OVERALL_CSV)
split_manifest = pd.read_csv(
    SPLIT_MANIFEST_CSV,
    keep_default_na=False
)

# Load the complete low-threshold RF-DETR predictions from Cell 7.
# Their confidence scores are preserved for COCO mAP calculations.
with open(PREDICTIONS_JSON, "r", encoding="utf-8") as file:
    coco_predictions = json.load(file)

# Initialize the official COCO test ground-truth object.
# Its image identifiers and annotations provide the reference for condition-specific evaluation.
coco_gt = COCO(str(TEST_JSON))

# Define the lighting categories and operating thresholds used during Cell 7.
# Keeping the thresholds consistent makes lighting-specific counts directly comparable.
LIGHTING_CONDITIONS = ["mild", "moderate", "severe"]
CONFIDENCE_THRESHOLD = float(
    overall_results.iloc[0]["Confidence_Threshold"]
)
IOU_THRESHOLD = float(
    overall_results.iloc[0]["IoU_Threshold"]
)

# Confirm that the evaluation records contain every required detection count.
# Each image must have true-positive, false-positive, and false-negative measurements.
required_columns = [
    "Image_ID",
    "Image",
    "Ground_Truth_Objects",
    "Predicted_Objects",
    "TP",
    "FP",
    "FN",
    "Mean_Matched_IoU",
    "Inference_Seconds"
]

for column in required_columns:
    assert column in image_results.columns, (
        f"Missing image evaluation column: {column}"
    )

# Verify that the test evaluation contains the same images as the COCO annotation file.
# Any missing or duplicated image would invalidate the lighting-specific results.
test_image_ids = set(coco_gt.getImgIds())

assert len(image_results) == len(test_image_ids)
assert image_results["Image_ID"].is_unique
assert set(image_results["Image_ID"]) == test_image_ids

# Select only test records from the saved dataset split manifest.
# These records provide the original source frame and actual low-light condition for each image.
test_manifest = split_manifest.loc[
    split_manifest["Split"].astype(str).str.lower() == "test"
].copy()

# Extract the exported image filename from its saved COCO image path.
# Joining on the filename avoids depending on generated image IDs from different source files.
test_manifest["Image"] = test_manifest["COCO_Image"].apply(
    lambda path: Path(str(path)).name
)

# Confirm that exported image filenames are unique within the test split.
# Duplicate filenames would prevent reliable lighting-condition assignments.
assert test_manifest["Image"].is_unique, (
    "Duplicate image filenames found in the test manifest."
)

# Merge lighting conditions and source-frame identifiers into the measured evaluation table.
# Every image's classification as mild, moderate, or severe comes from the original dataset metadata.
analysis_df = image_results.merge(
    test_manifest[
        ["Image", "Condition", "Source_Frame"]
    ],
    on="Image",
    how="left",
    validate="one_to_one"
)

# Normalize the condition labels and check that each test image was assigned correctly.
# Unmatched metadata must be fixed instead of guessing the lighting category.
analysis_df["Condition"] = (
    analysis_df["Condition"]
    .astype(str)
    .str.strip()
    .str.lower()
)

assert set(analysis_df["Condition"]) == set(LIGHTING_CONDITIONS), (
    "Unexpected or missing lighting conditions in the evaluation records."
)

assert analysis_df["Source_Frame"].notna().all(), (
    "Some test images could not be matched to their source frames."
)

# Verify that each image's object counts are internally consistent.
# Every ground-truth pedestrian must be either detected correctly or counted as a false negative.
assert (
    analysis_df["TP"] + analysis_df["FN"]
    == analysis_df["Ground_Truth_Objects"]
).all()

assert (
    analysis_df["TP"] + analysis_df["FP"]
    == analysis_df["Predicted_Objects"]
).all()

# Compare aggregated image-level detection counts against the previously saved overall metrics.
# This ensures that the new analysis does not silently change the original evaluation.
for metric in ["TP", "FP", "FN"]:
    assert int(analysis_df[metric].sum()) == int(
        overall_results.iloc[0][metric]
    ), f"Mismatch in the saved overall {metric} count."

# Calculate a percentage safely when the denominator is positive.
# An undefined metric is recorded as NaN rather than being interpreted as genuine zero performance.
def safe_percentage(numerator, denominator):
    return (
        100.0 * numerator / denominator
        if denominator > 0
        else np.nan
    )

# Initialize a list for the three lighting-specific performance summaries.
# Detection counts are aggregated before calculating precision, recall, F1-score, and accuracy.
lighting_rows = []

for condition in LIGHTING_CONDITIONS:

    # Select the independent test images corresponding to the current lighting category.
    # All observations come from the original held-out COCO test partition.
    subset = analysis_df[
        analysis_df["Condition"] == condition
    ].copy()

    # Sum true positives, false positives, and false negatives across the selected images.
    # These totals form the basis for condition-specific detection performance.
    tp = int(subset["TP"].sum())
    fp = int(subset["FP"].sum())
    fn = int(subset["FN"].sum())

    # Calculate precision and recall at the original confidence and IoU thresholds.
    # These are micro-aggregated object detection metrics rather than per-image percentages.
    precision = safe_percentage(tp, tp + fp)
    recall = safe_percentage(tp, tp + fn)

    # Calculate F1-score from the aggregated detection counts.
    # This formula remains valid even when the class distribution differs between conditions.
    f1 = safe_percentage(
        2 * tp,
        2 * tp + fp + fn
    )

    # Calculate detection accuracy using TP divided by TP plus FP plus FN.
    # This is a Jaccard-style detection measure, not standard image classification accuracy.
    detection_accuracy = safe_percentage(
        tp,
        tp + fp + fn
    )

    # Record the measured performance and the number of evaluated images.
    # The final table will support comparisons between mild, moderate, and severe conditions.
    lighting_rows.append({
        "Condition": condition,
        "Images": len(subset),
        "Source_Frames": subset["Source_Frame"].nunique(),
        "Ground_Truth_Objects": int(
            subset["Ground_Truth_Objects"].sum()
        ),
        "Predicted_Objects": int(
            subset["Predicted_Objects"].sum()
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Detection_Accuracy_Percent": detection_accuracy,
        "Precision_Percent": precision,
        "Recall_Percent": recall,
        "F1_Percent": f1,
        "Mean_Inference_Seconds": float(
            subset["Inference_Seconds"].mean()
        )
    })

# Convert the lighting summaries into a comparison table.
# This table contains the directly comparable metrics for the three lighting conditions.
lighting_df = pd.DataFrame(lighting_rows)

# Create a COCO detection results object from the saved prediction JSON.
# The same confidence-ranked predictions are used for overall and lighting-specific mAP calculations.
if coco_predictions:
    coco_dt = coco_gt.loadRes(str(PREDICTIONS_JSON))
else:
    coco_dt = COCO()
    coco_dt.dataset = {
        "images": list(coco_gt.dataset["images"]),
        "categories": list(coco_gt.dataset["categories"]),
        "annotations": []
    }
    coco_dt.createIndex()

# Evaluate standard COCO average precision separately for every lighting category.
# COCO AP uses confidence-ranked predictions and IoU thresholds independent of the selected operating point.
map_rows = []

for condition in LIGHTING_CONDITIONS:

    # Identify the COCO image IDs belonging to the current lighting condition.
    # This restricts the standard evaluation protocol to the corresponding held-out subset.
    condition_ids = analysis_df.loc[
        analysis_df["Condition"] == condition,
        "Image_ID"
    ].astype(int).tolist()

    # Initialize the COCO bounding-box evaluator for the current image subset.
    # The pedestrian category and COCO default maximum-detection settings are retained.
    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )

    evaluator.params.imgIds = condition_ids
    evaluator.params.catIds = [1]

    # Perform the standard COCO evaluation and accumulate its results.
    # Console output is captured to keep the notebook readable while numerical results remain available.
    log_buffer = StringIO()

    with redirect_stdout(log_buffer):
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    # Extract AP at IoU 0.50 and AP over IoU thresholds 0.50 through 0.95.
    # COCO may return -1 for undefined evaluations, which should not be reported as a negative percentage.
    map_50_95 = float(evaluator.stats[0])
    map_50 = float(evaluator.stats[1])

    map_rows.append({
        "Condition": condition,
        "mAP50_Percent": (
            map_50 * 100 if map_50 >= 0 else np.nan
        ),
        "mAP50_95_Percent": (
            map_50_95 * 100 if map_50_95 >= 0 else np.nan
        )
    })

# Combine the detection operating-point metrics with standard COCO AP measurements.
# This produces one complete evaluation record per lighting condition.
map_df = pd.DataFrame(map_rows)

lighting_df = lighting_df.merge(
    map_df,
    on="Condition",
    how="left",
    validate="one_to_one"
)

# Save the complete lighting-specific evaluation table.
# This file will be used directly in the final RF-DETR PDF report.
LIGHTING_CSV = CELL8_DIR / "cell8_lighting_performance_comparison.csv"

lighting_df.to_csv(
    LIGHTING_CSV,
    index=False
)

# Display the measured performance for all three lighting conditions.
# Every result is derived from the actual held-out test predictions and annotations.
print("\nRF-DETR PERFORMANCE BY LOW-LIGHT CONDITION")
print("=" * 80)

print(
    lighting_df[
        [
            "Condition",
            "Images",
            "TP",
            "FP",
            "FN",
            "Precision_Percent",
            "Recall_Percent",
            "F1_Percent",
            "mAP50_Percent",
            "mAP50_95_Percent"
        ]
    ].round(3).to_string(index=False)
)

# Calculate the number of missed pedestrians and false-positive detections per test image.
# These counts help identify difficult scenes for qualitative error analysis.
analysis_df["Missed_Pedestrians"] = analysis_df["FN"].astype(int)
analysis_df["False_Positive_Detections"] = analysis_df["FP"].astype(int)

# Calculate each image's share of missed ground-truth objects.
# Images with no annotated pedestrians receive an undefined miss rate rather than an artificial zero.
analysis_df["Miss_Rate_Percent"] = np.where(
    analysis_df["Ground_Truth_Objects"] > 0,
    100.0
    * analysis_df["FN"]
    / analysis_df["Ground_Truth_Objects"].replace(0, np.nan),
    np.nan
)

# Identify images with the highest number of missed pedestrians.
# These cases help determine whether small, occluded, or dark pedestrians are being overlooked.
missed_images = analysis_df.sort_values(
    ["FN", "FP", "Ground_Truth_Objects"],
    ascending=[False, False, False]
).head(10).copy()

# Identify images containing the most false-positive predictions.
# These may reveal duplicate boxes, background confusion, or incorrectly localized detections.
false_positive_images = analysis_df.sort_values(
    ["FP", "FN"],
    ascending=[False, False]
).head(10).copy()

# Identify images with the largest combined detection-error counts.
# The total error count provides a simple and interpretable ranking for qualitative inspection.
analysis_df["Total_Detection_Errors"] = (
    analysis_df["FP"] + analysis_df["FN"]
)

worst_images = analysis_df.sort_values(
    ["Total_Detection_Errors", "FN", "FP"],
    ascending=[False, False, False]
).head(10).copy()

# Save the complete image-level analysis and ranked error tables.
# These records provide traceable evidence for the discussion rather than relying on anecdotal examples.
IMAGE_ANALYSIS_CSV = CELL8_DIR / "cell8_image_level_error_analysis.csv"
MISSED_CSV = CELL8_DIR / "cell8_most_missed_pedestrians.csv"
FP_CSV = CELL8_DIR / "cell8_most_false_positive_images.csv"
WORST_CSV = CELL8_DIR / "cell8_highest_error_images.csv"

analysis_df.to_csv(IMAGE_ANALYSIS_CSV, index=False)
missed_images.to_csv(MISSED_CSV, index=False)
false_positive_images.to_csv(FP_CSV, index=False)
worst_images.to_csv(WORST_CSV, index=False)

# Display the highest-error test images for manual interpretation.
# The table reveals which lighting conditions and scenes require the most attention.
print("\nTEST IMAGES WITH THE MOST DETECTION ERRORS")
print("=" * 80)

print(
    worst_images[
        [
            "Image",
            "Condition",
            "Ground_Truth_Objects",
            "Predicted_Objects",
            "TP",
            "FP",
            "FN",
            "Total_Detection_Errors"
        ]
    ].to_string(index=False)
)

# Create a grouped bar chart comparing precision, recall, and F1-score.
# These three measures capture different aspects of detection quality across lighting conditions.
fig, ax = plt.subplots(figsize=(11, 6))

positions = np.arange(len(LIGHTING_CONDITIONS))
bar_width = 0.25

for index, metric in enumerate([
    "Precision_Percent",
    "Recall_Percent",
    "F1_Percent"
]):

    # Draw the corresponding metric for mild, moderate, and severe lighting.
    # Grouping the bars keeps the conditions and measured percentages directly comparable.
    ax.bar(
        positions + (index - 1) * bar_width,
        lighting_df[metric],
        width=bar_width,
        label=metric.replace("_Percent", "").replace("_", " ")
    )

# Format the lighting performance chart.
# A common zero-to-100 scale prevents small differences from being visually exaggerated.
ax.set_xticks(positions)
ax.set_xticklabels(
    [condition.title() for condition in LIGHTING_CONDITIONS]
)
ax.set_ylim(0, 100)
ax.set_ylabel("Performance (%)")
ax.set_title("RF-DETR Detection Performance by Lighting Condition")
ax.legend()
ax.grid(axis="y", alpha=0.25)

# Save the precision, recall, and F1-score comparison figure.
# It will be included in the evaluation results section of the final report.
plt.tight_layout()

METRICS_FIGURE = (
    FIGURES_DIR / "cell8_lighting_precision_recall_f1.png"
)

plt.savefig(
    METRICS_FIGURE,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# Create a chart showing standard COCO mAP values across lighting conditions.
# The figure distinguishes AP@0.50 from the stricter AP@0.50:0.95 evaluation.
fig, ax = plt.subplots(figsize=(10, 6))

width = 0.35

ax.bar(
    positions - width / 2,
    lighting_df["mAP50_Percent"],
    width=width,
    label="mAP@0.50"
)

ax.bar(
    positions + width / 2,
    lighting_df["mAP50_95_Percent"],
    width=width,
    label="mAP@0.50:0.95"
)

# Apply clear axis labels and a consistent percentage scale.
# Standard COCO AP can be lower than operating-point recall because it summarizes ranked detections.
ax.set_xticks(positions)
ax.set_xticklabels(
    [condition.title() for condition in LIGHTING_CONDITIONS]
)
ax.set_ylim(0, 100)
ax.set_ylabel("COCO Average Precision (%)")
ax.set_title("RF-DETR mAP Comparison by Low-Light Severity")
ax.legend()
ax.grid(axis="y", alpha=0.25)

# Save the standard COCO mAP chart for report documentation.
# These values are calculated independently for each lighting subset.
plt.tight_layout()

MAP_FIGURE = (
    FIGURES_DIR / "cell8_lighting_map_comparison.png"
)

plt.savefig(
    MAP_FIGURE,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# Create a chart comparing missed detections and false positives.
# Absolute error counts should be considered alongside the number of ground-truth pedestrians.
fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    positions - width / 2,
    lighting_df["FN"],
    width=width,
    label="False Negatives"
)

ax.bar(
    positions + width / 2,
    lighting_df["FP"],
    width=width,
    label="False Positives"
)

# Label the error-count chart without interpreting absolute counts as error rates.
# Scenes containing more pedestrians may naturally contribute more total errors.
ax.set_xticks(positions)
ax.set_xticklabels(
    [condition.title() for condition in LIGHTING_CONDITIONS]
)
ax.set_ylabel("Number of Detections")
ax.set_title("RF-DETR Detection Errors by Lighting Condition")
ax.legend()
ax.grid(axis="y", alpha=0.25)

# Save the false-positive and false-negative comparison figure.
# It supports discussion of whether missed detections or false alarms dominate.
plt.tight_layout()

ERROR_FIGURE = (
    FIGURES_DIR / "cell8_false_positives_false_negatives.png"
)

plt.savefig(
    ERROR_FIGURE,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# Build a lookup of all previously saved model predictions by COCO image identifier.
# Error visualizations will use these saved detections without running the model again.
prediction_lookup = {}

for prediction in coco_predictions:
    image_id = int(prediction["image_id"])

    if image_id not in prediction_lookup:
        prediction_lookup[image_id] = []

    prediction_lookup[image_id].append(prediction)

# Define a function for drawing saved ground-truth or predicted boxes.
# Green indicates actual annotations, while red indicates confidence-filtered RF-DETR detections.
def draw_boxes(ax, annotations, mode):

    # Process the objects associated with the selected image.
    # Ground-truth and COCO prediction records both use the [x, y, width, height] format.
    for annotation in annotations:

        # Ignore predictions below the confidence threshold used for detection metrics.
        # Ground-truth boxes are displayed without prediction confidence filtering.
        if (
            mode == "prediction"
            and annotation["score"] < CONFIDENCE_THRESHOLD
        ):
            continue

        x, y, width, height = annotation["bbox"]

        # Use distinct colors to distinguish annotated pedestrians from predicted objects.
        # The rectangle preserves the actual pixel coordinates from the COCO records.
        color = (
            "lime"
            if mode == "ground_truth"
            else "red"
        )

        rectangle = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=2,
            edgecolor=color,
            facecolor="none"
        )

        ax.add_patch(rectangle)

        # Display the prediction confidence next to RF-DETR bounding boxes.
        # This helps interpret possible false positives and uncertain detections.
        if mode == "prediction":
            ax.text(
                x,
                max(0, y - 4),
                f"{annotation['score']:.2f}",
                color="white",
                fontsize=8,
                bbox={
                    "facecolor": "red",
                    "alpha": 0.7
                }
            )

# Select up to two high-error images per lighting condition.
# This produces a balanced qualitative comparison rather than choosing only one severity level.
selected_errors = []

for condition in LIGHTING_CONDITIONS:

    # Rank images in the current lighting condition by their observed errors.
    # Missed pedestrians are given priority when total error counts are equal.
    candidates = analysis_df.loc[
        analysis_df["Condition"] == condition
    ].sort_values(
        ["Total_Detection_Errors", "FN", "FP"],
        ascending=[False, False, False]
    )

    selected_errors.extend(
        candidates.head(2).to_dict("records")
    )

# Generate the ground-truth versus prediction comparisons for the selected difficult images.
# These figures make it possible to visually inspect localization failures and missed pedestrians.
for index, record in enumerate(selected_errors, start=1):

    # Retrieve the original test image and its COCO image identifier.
    # The displayed scene must match the saved model predictions and ground-truth annotations.
    image_id = int(record["Image_ID"])
    image_info = coco_gt.loadImgs([image_id])[0]
    image_path = COCO_DIR / "test" / image_info["file_name"]

    with Image.open(image_path) as image:
        rgb_image = image.convert("RGB").copy()

    # Retrieve the actual pedestrian ground-truth annotations for the image.
    # These represent the corrected reference boxes used during evaluation.
    gt_annotations = coco_gt.loadAnns(
        coco_gt.getAnnIds(imgIds=[image_id])
    )

    # Retrieve all saved RF-DETR predictions for the same image.
    # The drawing function will display only those meeting the operating confidence threshold.
    predicted_annotations = prediction_lookup.get(
        image_id,
        []
    )

    # Create side-by-side displays of reference annotations and model outputs.
    # Identical source images allow the locations of missed and false-positive detections to be compared.
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 6)
    )

    axes[0].imshow(rgb_image)
    draw_boxes(
        axes[0],
        gt_annotations,
        mode="ground_truth"
    )

    axes[0].set_title(
        f"Ground Truth - {record['Condition'].title()}"
    )
    axes[0].axis("off")

    # Display the confidence-filtered RF-DETR detections in the second panel.
    # The title includes the measured true-positive, false-positive, and false-negative counts.
    axes[1].imshow(rgb_image)
    draw_boxes(
        axes[1],
        predicted_annotations,
        mode="prediction"
    )

    axes[1].set_title(
        f"RF-DETR | TP={int(record['TP'])} "
        f"FP={int(record['FP'])} FN={int(record['FN'])}"
    )
    axes[1].axis("off")

    # Save the comparison as evidence for the error analysis section.
    # Predictions and ground truth are read from saved files without changing the underlying dataset.
    plt.tight_layout()

    ERROR_PREVIEW = (
        FIGURES_DIR
        / f"cell8_error_example_{index:02d}_{record['Condition']}.png"
    )

    plt.savefig(
        ERROR_PREVIEW,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

# Identify the highest and lowest observed lighting-condition F1-scores.
# These comparisons describe the measured data without establishing a causal relationship with darkness.
ranked_lighting = lighting_df.sort_values(
    "F1_Percent",
    ascending=False
).reset_index(drop=True)

best_condition = ranked_lighting.iloc[0]
worst_condition = ranked_lighting.iloc[-1]

# Compare the mild and severe lighting results explicitly.
# Both conditions originate from related source frames, but this analysis is descriptive rather than a statistical significance test.
mild_row = lighting_df.set_index("Condition").loc["mild"]
severe_row = lighting_df.set_index("Condition").loc["severe"]

f1_difference = (
    float(severe_row["F1_Percent"])
    - float(mild_row["F1_Percent"])
)

recall_difference = (
    float(severe_row["Recall_Percent"])
    - float(mild_row["Recall_Percent"])
)

# Build a report-ready analysis using only the measured evaluation statistics.
# Potential causes of errors are presented as hypotheses that require visual confirmation.
discussion = f"""
RF-DETR FINAL PROJECT - LOW-LIGHT ERROR ANALYSIS

1. OVERALL EVALUATION

The fine-tuned RF-DETR Nano model was evaluated on
{len(analysis_df)} held-out images using an operating confidence
threshold of {CONFIDENCE_THRESHOLD:.2f} and an IoU threshold
of {IOU_THRESHOLD:.2f}.

The complete evaluation produced
{int(analysis_df['TP'].sum())} true positives,
{int(analysis_df['FP'].sum())} false positives, and
{int(analysis_df['FN'].sum())} false negatives.

2. PERFORMANCE BY LIGHTING CONDITION

Mild lighting achieved {float(mild_row['Precision_Percent']):.2f}%
precision, {float(mild_row['Recall_Percent']):.2f}% recall,
and {float(mild_row['F1_Percent']):.2f}% F1-score.

Severe lighting achieved {float(severe_row['Precision_Percent']):.2f}%
precision, {float(severe_row['Recall_Percent']):.2f}% recall,
and {float(severe_row['F1_Percent']):.2f}% F1-score.

The difference in F1-score between severe and mild lighting was
{f1_difference:+.2f} percentage points, while the recall difference
was {recall_difference:+.2f} percentage points.

The highest observed F1-score occurred in
{best_condition['Condition']} lighting at
{float(best_condition['F1_Percent']):.2f}%.
The lowest observed F1-score occurred in
{worst_condition['Condition']} lighting at
{float(worst_condition['F1_Percent']):.2f}%.

3. ERROR ANALYSIS

False negatives represent pedestrians that were not successfully
matched with predictions at the selected thresholds.
False positives represent predicted boxes that were not
successfully matched with annotated pedestrians.

Potential explanations include low object contrast,
small pedestrian size, partial occlusion, background clutter,
and inaccurate localization. These explanations should be
checked against the saved error visualizations before being
presented as confirmed causes.

4. TRAINING OPTIMIZATION

The experiment used pretrained RF-DETR Nano weights and the
optimization settings documented in the training configuration.
The final detection results demonstrate performance after
fine-tuning, but do not isolate the effect of each optimization
because no controlled optimizer or learning-rate ablation
was performed.

5. LIMITATIONS

The test dataset contains a limited number of independent
source frames and multiple derived lighting variants.
Results may not generalize to different cameras, locations,
weather conditions, or real-world nighttime environments.

The evaluation measures pedestrian detection performance
using corrected annotations, but annotation uncertainty
and preprocessing artifacts may still influence the results.

The reported detection accuracy is the TP/(TP+FP+FN)
measure, not conventional classification accuracy.

6. FUTURE WORK

Possible extensions include collecting more independent
nighttime pedestrian scenes, evaluating additional RF-DETR
model sizes, testing different learning rates and training
schedules, measuring inference speed on deployment hardware,
and investigating image-enhancement methods.

Further evaluation using multiple training seeds and external
test datasets would help assess the robustness and
generalizability of the observed results.
""".strip()

# Save the complete comparative discussion as a text file.
# This file can be imported directly into the final PDF report.
DISCUSSION_PATH = (
    CELL8_DIR / "cell8_error_analysis_discussion.txt"
)

DISCUSSION_PATH.write_text(
    discussion,
    encoding="utf-8"
)

# Display the report-ready error analysis in the Colab notebook.
# Review the example images to determine whether the proposed error explanations fit the observed scenes.
print("\nFINAL RF-DETR ERROR ANALYSIS AND DISCUSSION")
print("=" * 80)
print(discussion)

# Verify that the aggregated lighting results reproduce the original Cell 7 counts.
# This protects against accidental image duplication or missing lighting metadata.
for metric in ["TP", "FP", "FN"]:
    assert int(lighting_df[metric].sum()) == int(
        analysis_df[metric].sum()
    )

# Confirm that all evaluated test images were assigned to exactly one lighting category.
# Lighting-specific analysis must cover the same image population as the final overall evaluation.
assert int(lighting_df["Images"].sum()) == len(image_results)

# Display the saved analysis outputs and final completion status.
# The next stage will use these records to assemble the final RF-DETR PDF report.
print("\nCELL 8 OUTPUT FILES")
print("=" * 80)

print("Lighting Performance:", LIGHTING_CSV)
print("Image-Level Error Analysis:", IMAGE_ANALYSIS_CSV)
print("Most Missed Pedestrians:", MISSED_CSV)
print("Most False Positives:", FP_CSV)
print("Highest-Error Images:", WORST_CSV)
print("Written Discussion:", DISCUSSION_PATH)
print("Precision/Recall/F1 Chart:", METRICS_FIGURE)
print("mAP Chart:", MAP_FIGURE)
print("False-Positive/False-Negative Chart:", ERROR_FIGURE)
print("Error Visualization Folder:", FIGURES_DIR)

# Confirm successful completion of the lighting-specific evaluation and error analysis.
# All numerical findings are based on the actual RF-DETR test predictions saved in Cell 7.
print("\nCELL 8 COMPLETED SUCCESSFULLY!")

In [ ]:
# Install PyMuPDF, which provides the fitz module.
# This package is used to verify the generated PDF and inspect its pages.
%pip install -q pymupdf

# Install the PDF generation library used to assemble the final academic report.
# ReportLab allows headings, tables, page numbers, figures, and references to be written directly to PDF.
%pip install -q reportlab

# Import libraries for reading completed experimental results and generating PDF documents.
# These tools allow the report to use the existing measured results without running model inference again.
from pathlib import Path
from datetime import datetime
from xml.sax.saxutils import escape
import json
import re
import pandas as pd
import numpy as np

# Import ReportLab components for academic document formatting.
# These components create readable report pages with consistently styled text, tables, and images.
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image as ReportImage,
    PageBreak,
    KeepTogether,
    Preformatted
)
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics
from PIL import Image as PILImage

# Define the previously established RF-DETR project directory.
# The report will be generated directly into the Google Drive project folder.
PROJECT_DIR = Path("/content/drive/MyDrive/Person_Detection")
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"

# Define all input and output directories for the academic report.
# Original datasets, ground-truth annotation files, and trained model weights will not be modified.
DATASET_DIR = RFDETR_DIR / "dataset"
RESULTS_DIR = RFDETR_DIR / "evaluation_results"
FIGURES_DIR = RFDETR_DIR / "figures"
MODEL_DIR = RFDETR_DIR / "trained_models"
REPORT_DIR = RFDETR_DIR / "final_report"

# Define the saved output directories created during the final evaluation.
# These folders contain actual experiment measurements and performance visualizations.
CELL7_DIR = RESULTS_DIR / "cell7_final_test_evaluation"
CELL8_DIR = RESULTS_DIR / "cell8_error_analysis"
CELL8_FIGURES = FIGURES_DIR / "cell8_error_analysis"
CELL7_FIGURES = FIGURES_DIR / "cell7_final_test_evaluation"

# Create the final report directory if it does not already exist.
# The academic PDF will be saved independently from the experimental outputs.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Define the required experimental files.
# A report must not claim completed results when the corresponding evaluation records are unavailable.
required_files = {
    "COCO Conversion Summary": DATASET_DIR / "cell3_coco_conversion_summary.csv",
    "Dataset Split Manifest": DATASET_DIR / "cell3_coco_split_manifest.csv",
    "Final Test Metrics": CELL7_DIR / "cell7_rfdetr_final_test_metrics.csv",
    "Per-Image Test Results": CELL7_DIR / "cell7_per_image_detection_results.csv",
    "Lighting Performance": CELL8_DIR / "cell8_lighting_performance_comparison.csv",
    "Error Analysis": CELL8_DIR / "cell8_image_level_error_analysis.csv",
    "Written Discussion": CELL8_DIR / "cell8_error_analysis_discussion.txt"
}

# Check that all required experimental outputs exist.
# Missing files are displayed explicitly instead of producing a partially fabricated final report.
missing_files = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not path.is_file()
]

assert not missing_files, (
    "Required report inputs are missing:\n"
    + "\n".join(missing_files)
    + "\nComplete the relevant earlier cells first."
)

# Load the verified dataset, test evaluation, and lighting-comparison results.
# Every numerical table in the report will be constructed from these actual saved files.
dataset_summary = pd.read_csv(
    required_files["COCO Conversion Summary"]
)

split_manifest = pd.read_csv(
    required_files["Dataset Split Manifest"]
)

test_metrics = pd.read_csv(
    required_files["Final Test Metrics"]
)

image_results = pd.read_csv(
    required_files["Per-Image Test Results"]
)

lighting_metrics = pd.read_csv(
    required_files["Lighting Performance"]
)

error_analysis = pd.read_csv(
    required_files["Error Analysis"]
)

# Load the discussion produced from the measured lighting-condition evaluation.
# Its findings will be included after reviewing the metric tables and generated charts.
discussion_text = required_files["Written Discussion"].read_text(
    encoding="utf-8"
)

# Verify that the final performance tables contain evaluated observations.
# Empty tables cannot support valid academic conclusions.
assert not dataset_summary.empty
assert len(test_metrics) == 1
assert not image_results.empty
assert not lighting_metrics.empty

# Verify that the summary metrics agree with the per-image test results.
# This protects the final report against inconsistencies between different saved evaluation files.
test_row = test_metrics.iloc[0]

for metric in ["TP", "FP", "FN"]:
    assert int(test_row[metric]) == int(
        image_results[metric].sum()
    ), (
        f"Inconsistent {metric} totals in saved evaluation files."
    )

# Verify that the lighting-specific table includes all expected severity categories.
# A missing lighting category would make the stated three-condition comparison incomplete.
assert set(
    lighting_metrics["Condition"].astype(str).str.lower()
) == {"mild", "moderate", "severe"}

# Locate the already trained RF-DETR Nano checkpoint.
# The document will identify the model artifact used during held-out test evaluation.
BEST_CHECKPOINT = (
    MODEL_DIR
    / "rfdetr_nano_training"
    / "checkpoint_best_total.pth"
)

assert BEST_CHECKPOINT.is_file(), (
    "The validated RF-DETR model checkpoint is missing."
)

# Identify available training configuration records without assuming a fixed successful run history.
# Settings will be included only when a recorded experiment configuration is accessible.
configuration_candidates = [
    RESULTS_DIR / "cell5_rfdetr_training_configuration.json",
    MODEL_DIR / "rfdetr_nano_training" / "cell5b_run_configuration.json"
]

training_configuration = {}
configuration_source = None

for path in configuration_candidates:
    if path.is_file():
        with open(path, "r", encoding="utf-8") as file:
            training_configuration = json.load(file)

        configuration_source = path
        break

# Extract the recorded training arguments from the available configuration.
# The report will not assume an optimizer or batch size if the saved metadata differs.
training_arguments = training_configuration.get(
    "training_parameters",
    training_configuration.get("training_config", {})
)

# Locate an optional training-loss or validation-metric figure from Cell 6.
# This figure will be included only if it exists as an actual generated image.
training_figure_candidates = [
    FIGURES_DIR / "cell6_training_analysis" / "cell6_training_metric_history.png",
    FIGURES_DIR / "cell6_recorded_training_metrics.png"
]

training_figure = next(
    (
        path
        for path in training_figure_candidates
        if path.is_file()
    ),
    None
)

# Locate the available COCO annotation example and evaluation visualizations.
# The report will use actual figures saved by the earlier notebook cells.
coco_example = (
    FIGURES_DIR
    / "cell3_converted_coco_annotation_example.png"
)

lighting_figures = [
    CELL8_FIGURES / "cell8_lighting_precision_recall_f1.png",
    CELL8_FIGURES / "cell8_lighting_map_comparison.png",
    CELL8_FIGURES / "cell8_false_positives_false_negatives.png"
]

prediction_figures = sorted(
    CELL7_FIGURES.glob("cell7_prediction_example_*.png")
)

error_figures = sorted(
    CELL8_FIGURES.glob("cell8_error_example_*.png")
)

# Define the final academic PDF filename.
# The file will be available inside Google Drive after the report generation cell completes.
PDF_PATH = (
    REPORT_DIR
    / "RF_DETR_Nano_Low_Light_Pedestrian_Detection_Final_Report.pdf"
)

# Register a suitable academic font when available.
# DejaVu Sans supports common punctuation and produces readable PDF text.
regular_font_path = Path(
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
)

bold_font_path = Path(
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
)

if regular_font_path.is_file() and bold_font_path.is_file():
    pdfmetrics.registerFont(
        TTFont("ReportRegular", str(regular_font_path))
    )

    pdfmetrics.registerFont(
        TTFont("ReportBold", str(bold_font_path))
    )

    FONT_REGULAR = "ReportRegular"
    FONT_BOLD = "ReportBold"

else:
    # Use built-in fonts if the preferred font files are not available.
    # Report generation can still proceed without installing additional font packages.
    FONT_REGULAR = "Helvetica"
    FONT_BOLD = "Helvetica-Bold"

# Configure the academic document page geometry.
# A4 paper with uniform margins provides enough room for paragraphs, tables, and figures.
document = SimpleDocTemplate(
    str(PDF_PATH),
    pagesize=A4,
    rightMargin=48,
    leftMargin=48,
    topMargin=53,
    bottomMargin=53,
    title="RF-DETR Nano Low-Light Pedestrian Detection",
    author="Aguirre, Czarina Nicole R."
)

# Define the report's typography and heading styles.
# Section headings, subheadings, captions, and references use consistent spacing.
styles = getSampleStyleSheet()

styles.add(ParagraphStyle(
    name="AcademicTitle",
    fontName=FONT_BOLD,
    fontSize=17,
    leading=24,
    alignment=TA_CENTER,
    spaceAfter=18
))

styles.add(ParagraphStyle(
    name="AcademicSubtitle",
    fontName=FONT_REGULAR,
    fontSize=11,
    leading=17,
    alignment=TA_CENTER,
    spaceAfter=8
))

styles.add(ParagraphStyle(
    name="AcademicHeading",
    fontName=FONT_BOLD,
    fontSize=12,
    leading=17,
    spaceBefore=18,
    spaceAfter=9
))

styles.add(ParagraphStyle(
    name="AcademicSubheading",
    fontName=FONT_BOLD,
    fontSize=10,
    leading=15,
    spaceBefore=12,
    spaceAfter=6
))

styles.add(ParagraphStyle(
    name="AcademicBody",
    fontName=FONT_REGULAR,
    fontSize=9,
    leading=15,
    alignment=TA_JUSTIFY,
    spaceAfter=9
))

styles.add(ParagraphStyle(
    name="AcademicCaption",
    fontName=FONT_REGULAR,
    fontSize=8,
    leading=12,
    alignment=TA_CENTER,
    spaceBefore=6,
    spaceAfter=11
))

styles.add(ParagraphStyle(
    name="AcademicTable",
    fontName=FONT_REGULAR,
    fontSize=7.5,
    leading=11
))

styles.add(ParagraphStyle(
    name="AcademicTableHeader",
    fontName=FONT_BOLD,
    fontSize=7.5,
    leading=11,
    textColor=colors.white
))

styles.add(ParagraphStyle(
    name="AcademicReference",
    fontName=FONT_REGULAR,
    fontSize=8,
    leading=13,
    leftIndent=12,
    firstLineIndent=-12,
    spaceAfter=7
))

# Initialize the ordered collection of PDF report elements.
# Paragraphs, tables, images, and page breaks will be added sequentially.
story = []

# Define a helper for converting plain text into a safe ReportLab paragraph.
# XML-sensitive characters are escaped to prevent formatting errors.
def add_paragraph(text):
    story.append(
        Paragraph(
            escape(str(text)),
            styles["AcademicBody"]
        )
    )

# Define a helper for adding numbered report sections.
# Consistent heading formatting improves the report's readability and hierarchy.
def add_heading(text):
    story.append(
        Paragraph(
            escape(str(text)),
            styles["AcademicHeading"]
        )
    )

# Define a helper for adding numbered subsections.
# Subheadings divide longer methodology and results sections into manageable parts.
def add_subheading(text):
    story.append(
        Paragraph(
            escape(str(text)),
            styles["AcademicSubheading"]
        )
    )

# Define a helper for rendering numerical values without fabricating unavailable results.
# Missing or undefined measurements are displayed as N/A.
def format_number(value, decimals=2):
    if pd.isna(value):
        return "N/A"

    try:
        return f"{float(value):.{decimals}f}"
    except (TypeError, ValueError):
        return str(value)

# Define a helper for creating report tables with automatic text wrapping.
# Using Paragraph elements prevents long cell labels from overflowing table boundaries.
def add_table(headers, rows, widths=None):

    # Create wrapped table headers and body cells.
    # All values are escaped and converted to printable text.
    table_data = [[
        Paragraph(
            escape(str(value)),
            styles["AcademicTableHeader"]
        )
        for value in headers
    ]]

    for row in rows:
        table_data.append([
            Paragraph(
                escape(str(value)),
                styles["AcademicTable"]
            )
            for value in row
        ])

    # Create a table constrained to the available page width.
    # Alternating row backgrounds and internal padding improve legibility.
    table = Table(
        table_data,
        colWidths=widths,
        repeatRows=1,
        hAlign="CENTER"
    )

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#243447")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [
            colors.white,
            colors.HexColor("#F1F4F7")
        ]),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 7),
        ("RIGHTPADDING", (0, 0), (-1, -1), 7),
        ("TOPPADDING", (0, 0), (-1, -1), 7),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
        ("LINEBELOW", (0, 0), (-1, 0), 0.5, colors.grey)
    ]))

    story.append(table)
    story.append(Spacer(1, 9))

# Define a helper for adding existing experimental figures to the report.
# Image dimensions are scaled proportionally to fit the usable page area.
def add_figure(path, caption, max_height=290):

    # Skip nonexistent optional images and document their absence separately.
    # No illustrative figures are fabricated or substituted for missing experimental outputs.
    if not Path(path).is_file():
        return False

    # Read the image dimensions to preserve its original aspect ratio.
    # This prevents stretching when embedding plots or annotated pedestrian images.
    with PILImage.open(path) as image:
        width, height = image.size

    max_width = A4[0] - 110
    scale = min(max_width / width, max_height / height)

    figure = ReportImage(
        str(path),
        width=width * scale,
        height=height * scale
    )

    # Add the figure together with its caption when space permits.
    # Keeping these elements together prevents orphaned figure descriptions.
    story.append(
        KeepTogether([
            figure,
            Paragraph(
                escape(caption),
                styles["AcademicCaption"]
            )
        ])
    )

    return True

# Define a function to add page numbers to every report page.
# The footer provides consistent identification and navigation throughout the document.
def draw_page_footer(canvas, doc):
    canvas.saveState()
    canvas.setFont(FONT_REGULAR, 8)

    canvas.drawString(
        48,
        30,
        "RF-DETR Nano | Low-Light Pedestrian Detection"
    )

    canvas.drawRightString(
        A4[0] - 48,
        30,
        f"Page {doc.page}"
    )

    canvas.restoreState()

# Begin the title page of the final academic report.
# No course, school, or instructor information is inserted into the cover.
story.append(Spacer(1, 95))

story.append(Paragraph(
    "RF-DETR Nano for Low-Light Pedestrian Detection",
    styles["AcademicTitle"]
))

story.append(Paragraph(
    "Model Development, Fine-Tuning, Evaluation, and Error Analysis",
    styles["AcademicSubtitle"]
))

story.append(Spacer(1, 33))

story.append(Paragraph(
    "Prepared by",
    styles["AcademicSubtitle"]
))

story.append(Paragraph(
    "Aguirre, Czarina Nicole R.",
    styles["AcademicSubtitle"]
))

story.append(Spacer(1, 30))

story.append(Paragraph(
    datetime.now().strftime("%B %d, %Y"),
    styles["AcademicSubtitle"]
))

story.append(PageBreak())

# Add the abstract using the actual held-out test results.
# No performance claims are made beyond the stored evaluation measurements.
add_heading("Abstract")

add_paragraph(
    "This project developed and evaluated a low-light pedestrian "
    "detection system using RF-DETR Nano, a pretrained transformer-based "
    "object detector. Previously verified pedestrian annotations were "
    "converted into COCO format, and related lighting variants were "
    "partitioned by original source frame to reduce data leakage. "
    "The model was fine-tuned using the training and validation partitions, "
    "while the held-out test partition was reserved for final assessment."
)

add_paragraph(
    f"The independent test evaluation included "
    f"{int(test_row['Test_Images'])} images. "
    f"The detector achieved {format_number(test_row['Precision_Percent'])}% "
    f"precision, {format_number(test_row['Recall_Percent'])}% recall, "
    f"{format_number(test_row['F1_Percent'])}% F1-score, "
    f"and {format_number(test_row['mAP50_Percent'])}% mAP at IoU 0.50. "
    "Performance was also examined separately under mild, moderate, "
    "and severe low-light conditions. The report discusses measured "
    "detection errors, experimental limitations, and possible extensions."
)

add_paragraph(
    "Keywords: RF-DETR, pedestrian detection, low-light images, "
    "object detection, vision transformer, COCO, mean Average Precision."
)

# Introduce the motivation and scope of pedestrian detection under low-light conditions.
# The discussion remains focused on the single-class dataset used in this project.
add_heading("1. Introduction")

add_paragraph(
    "Pedestrian detection is an important computer vision task in "
    "traffic monitoring, road-scene understanding, and automated visual "
    "analysis. Poor illumination can reduce object contrast and obscure "
    "visual features needed for reliable detection. This motivates the "
    "evaluation of modern object detection models under different "
    "levels of low-light image degradation."
)

add_paragraph(
    "This study investigates the application of RF-DETR Nano to "
    "single-class pedestrian detection. Rather than developing a new "
    "object detection architecture from scratch, the project fine-tunes "
    "an existing pretrained detection transformer and evaluates its "
    "performance against corrected ground-truth annotations."
)

# State the research objectives supported by the completed experiment.
# The objectives correspond to dataset preparation, model fine-tuning, and measured detection analysis.
add_heading("2. Project Objectives")

for objective in [
    "Prepare a pedestrian-only COCO dataset from reviewed and corrected low-light annotations.",
    "Divide independent source frames into training, validation, and testing partitions.",
    "Fine-tune an RF-DETR Nano model using pretrained weights and documented optimization settings.",
    "Evaluate detection performance using precision, recall, F1-score, IoU, and COCO mAP.",
    "Compare detection performance across mild, moderate, and severe lighting conditions.",
    "Identify false positives, missed pedestrians, and other observable detection limitations."
]:
    add_paragraph("- " + objective)

# Describe the original annotation selection and COCO conversion methodology.
# Source-frame grouping is highlighted because lighting variants of one scene should not cross splits.
add_heading("3. Methodology")

add_subheading("3.1 Dataset Preparation")

add_paragraph(
    "The dataset was prepared using the project's previously collected "
    "pedestrian images and reviewed ground-truth annotations. Selected "
    "YOLO-format labels were verified for class consistency, valid "
    "coordinate ranges, and image correspondence before being converted "
    "into COCO bounding-box annotations."
)

add_paragraph(
    "COCO annotation records contain image identifiers, pedestrian "
    "category identifiers, image dimensions, bounding boxes, and box areas. "
    "Minor floating-point boundary deviations were clipped within a "
    "documented tolerance during export, while original label files "
    "were preserved."
)

add_subheading("3.2 Dataset Splitting")

add_paragraph(
    "Source frames were divided into approximately 70% training, "
    "15% validation, and 15% testing groups. Images derived from the same "
    "original frame were kept within one partition to reduce leakage "
    "between closely related lighting variants."
)

# Construct the recorded dataset split table.
# Each row comes from the actual COCO conversion summary.
dataset_rows = []

for _, row in dataset_summary.iterrows():
    dataset_rows.append([
        str(row["Split"]),
        str(int(row["Source_Frames"])),
        str(int(row["Images"])),
        str(int(row["Annotations"]))
    ])

add_table(
    ["Dataset Split", "Source Frames", "Images", "Annotations"],
    dataset_rows,
    widths=[105, 105, 100, 105]
)

# Include a saved example of actual COCO ground-truth annotation geometry.
# The figure provides visual evidence of successful image and annotation conversion.
add_figure(
    coco_example,
    "Figure 1. Converted COCO pedestrian ground-truth annotation example.",
    max_height=280
)

# Describe the pretrained detection transformer without inventing undocumented architecture modifications.
# RF-DETR is treated as an established pretrained detector rather than a custom ResNet-based implementation.
add_subheading("3.3 RF-DETR Model Architecture")

add_paragraph(
    "RF-DETR Nano was selected as the detection architecture. RF-DETR "
    "is a real-time detection transformer that uses a vision-transformer "
    "backbone and transformer-based detection components to predict "
    "object bounding boxes and category confidence scores. The project's "
    "implementation used the official pretrained RF-DETR model and "
    "fine-tuned it for pedestrian detection."
)

add_paragraph(
    "The experiment did not introduce a separately implemented receptive-field "
    "enhancement module. Therefore, any such architectural modification "
    "must not be claimed as part of this project's implementation."
)

# Present the actual saved training configuration when it is available.
# The report does not assume that a setting was executed unless supported by the recorded configuration.
add_subheading("3.4 Model Fine-Tuning and Optimization")

if training_arguments:

    # Select training fields relevant to the documented optimization procedure.
    # Missing configuration fields are omitted rather than filled with assumed values.
    training_fields = [
        ("epochs", "Maximum Epochs"),
        ("batch_size", "Batch Size"),
        ("grad_accum_steps", "Gradient Accumulation"),
        ("lr", "Learning Rate"),
        ("optimizer", "Optimizer"),
        ("lr_scheduler", "Learning Rate Scheduler"),
        ("early_stopping", "Early Stopping"),
        ("early_stopping_patience", "Early Stopping Patience"),
        ("best_model_metric", "Model Selection Metric"),
        ("checkpoint_interval", "Checkpoint Interval")
    ]

    training_rows = [
        [label, str(training_arguments[key])]
        for key, label in training_fields
        if key in training_arguments
    ]

    add_table(
        ["Training Parameter", "Recorded Value"],
        training_rows,
        widths=[205, 210]
    )

else:
    add_paragraph(
        "The trained model checkpoint was available, but a complete "
        "training-configuration JSON record was not found. The exact "
        "optimization parameters could not be independently verified "
        "from the saved configuration files."
    )

add_paragraph(
    "The training partition was used to update model parameters, "
    "while the validation partition supported checkpoint selection. "
    "The independent test partition was not used for training or "
    "model selection."
)

# Include a training history figure only if the logged measurements were available.
# The PDF clearly distinguishes missing training histories from successfully measured test metrics.
add_heading("4. Training Results and Checkpoint Verification")

add_paragraph(
    "The saved fine-tuned RF-DETR Nano checkpoint was identified at "
    f"{BEST_CHECKPOINT.name}. Its file size was "
    f"{BEST_CHECKPOINT.stat().st_size / (1024 ** 2):.2f} MB. "
    "This checkpoint was used as the model artifact for held-out "
    "test evaluation."
)

if training_figure is not None:
    add_figure(
        training_figure,
        "Figure 2. Recorded RF-DETR training or validation metrics.",
        max_height=310
    )
else:
    add_paragraph(
        "An exportable training-loss or validation-mAP history figure "
        "was not available in the saved report inputs. Therefore, "
        "the report does not present or estimate a training convergence curve."
    )

# Define the operating-point detection metrics in clear terms.
# These definitions help distinguish thresholded detection counts from COCO average precision.
add_heading("5. Evaluation Metrics")

add_paragraph(
    "Predicted boxes were matched with ground-truth boxes at the "
    "recorded Intersection over Union threshold. Each ground-truth "
    "pedestrian could match at most one prediction. Unmatched predictions "
    "were counted as false positives, while unmatched reference objects "
    "were counted as false negatives."
)

add_paragraph(
    "Precision = TP / (TP + FP). Recall = TP / (TP + FN). "
    "F1-score is the harmonic mean of precision and recall. "
    "The reported detection accuracy is TP / (TP + FP + FN), "
    "a detection-focused Jaccard-style measure rather than conventional "
    "image classification accuracy."
)

add_paragraph(
    "COCO mAP@0.50 measures Average Precision at an IoU threshold "
    "of 0.50. COCO mAP@0.50:0.95 averages precision across multiple "
    "IoU thresholds from 0.50 to 0.95."
)

# Present the final held-out test metrics from the saved Cell 7 results.
# All values are read from the actual evaluation CSV without altering their numerical meaning.
add_heading("6. Experimental Results")

add_subheading("6.1 Overall Test Performance")

result_fields = [
    ("Test_Images", "Test Images"),
    ("Ground_Truth_Objects", "Ground-Truth Pedestrians"),
    ("TP", "True Positives"),
    ("FP", "False Positives"),
    ("FN", "False Negatives"),
    ("Detection_Accuracy_Percent", "Detection Accuracy (%)"),
    ("Precision_Percent", "Precision (%)"),
    ("Recall_Percent", "Recall (%)"),
    ("F1_Percent", "F1-Score (%)"),
    ("mAP50_Percent", "COCO mAP@0.50 (%)"),
    ("mAP50_95_Percent", "COCO mAP@0.50:0.95 (%)"),
    ("Matched_Object_Class_Accuracy_Percent", "Matched-Object Class Accuracy (%)"),
    ("Image_Averaged_Matched_IoU", "Image-Averaged Matched IoU")
]

overall_rows = []

for key, label in result_fields:
    if key in test_metrics.columns:
        value = test_row[key]

        if key in [
            "Test_Images",
            "Ground_Truth_Objects",
            "TP",
            "FP",
            "FN"
        ]:
            display_value = (
                str(int(value))
                if pd.notna(value)
                else "N/A"
            )
        else:
            display_value = format_number(value, 3)

        overall_rows.append([
            label,
            display_value
        ])

add_table(
    ["Metric", "Measured Result"],
    overall_rows,
    widths=[270, 145]
)

add_paragraph(
    "Matched-object classification accuracy has limited discriminative "
    "value in this experiment because all annotated objects belong to "
    "the pedestrian category. Precision, recall, IoU, and COCO mAP "
    "provide more meaningful evidence of detection reliability."
)

# Present the independent lighting-specific performance results.
# Mild, moderate, and severe conditions use the same measured thresholds and evaluation procedure.
add_subheading("6.2 Comparison Across Lighting Conditions")

lighting_rows = []

for _, row in lighting_metrics.iterrows():
    lighting_rows.append([
        str(row["Condition"]).title(),
        format_number(row["Precision_Percent"]),
        format_number(row["Recall_Percent"]),
        format_number(row["F1_Percent"]),
        format_number(row["mAP50_Percent"]),
        format_number(row["mAP50_95_Percent"])
    ])

add_table(
    ["Lighting", "Precision %", "Recall %", "F1 %", "mAP50 %", "mAP50:95 %"],
    lighting_rows,
    widths=[75, 70, 65, 60, 70, 75]
)

# Add the saved precision, recall, F1-score, and mAP comparisons.
# Only figures generated by actual Cell 8 results are included.
for index, path in enumerate(lighting_figures, start=1):
    add_figure(
        path,
        f"Figure {index + 2}. RF-DETR low-light evaluation comparison.",
        max_height=300
    )

# Provide visual comparisons of verified ground truth and RF-DETR predictions.
# A limited number of representative examples keeps the report readable.
add_subheading("6.3 Pedestrian Detection Visualizations")

if prediction_figures:
    for index, path in enumerate(prediction_figures[:3], start=1):
        add_figure(
            path,
            f"Prediction Example {index}. Ground truth compared with fine-tuned RF-DETR detections.",
            max_height=290
        )
else:
    add_paragraph(
        "No exported prediction comparison figures were found. "
        "The numerical evaluation results remain available in the saved test CSV."
    )

# Summarize the measured error counts and include selected difficult test scenes.
# These examples provide qualitative context for false-positive and false-negative measurements.
add_heading("7. Detection Error Analysis")

add_paragraph(
    f"The evaluation recorded {int(test_row['FP'])} false-positive "
    f"detections and {int(test_row['FN'])} missed pedestrians at the "
    "selected operating thresholds. The error analysis considered "
    "the distribution of these outcomes across lighting conditions "
    "and identified images requiring additional qualitative inspection."
)

if error_figures:
    for index, path in enumerate(error_figures[:3], start=1):
        add_figure(
            path,
            f"Error Example {index}. Ground truth and model predictions on a difficult test image.",
            max_height=290
        )

# Include the data-derived written discussion from the previous analysis cell.
# Headings and paragraphs are preserved without repeating the generated discussion word for word.
add_heading("8. Discussion")

discussion_sections = re.split(
    r"\n\s*\n",
    discussion_text
)

for block in discussion_sections:

    # Remove repeated standalone report-title text and normalize whitespace.
    # The extracted discussion remains based on the actual previously saved analysis.
    cleaned = " ".join(block.split())

    if not cleaned:
        continue

    if cleaned.startswith(
        "RF-DETR FINAL PROJECT - LOW-LIGHT ERROR ANALYSIS"
    ):
        continue

    # Use subsection styling for numbered discussion topics.
    # Ordinary explanatory paragraphs receive the academic body style.
    if re.match(r"^\d+\.\s+[A-Z][A-Z ]+$", cleaned):
        add_subheading(cleaned.title())
    else:
        add_paragraph(cleaned)

# Present the constraints relevant to interpreting the experimental findings.
# These limitations prevent overgeneralization from a small, related-image test dataset.
add_heading("9. Limitations")

add_paragraph(
    "The dataset contains derived lighting variants of a limited "
    "number of original source frames. Although source-level splitting "
    "reduces direct image leakage, related acquisition conditions "
    "and scene characteristics can still limit generalization "
    "to new cameras, locations, and real-world nighttime environments."
)

add_paragraph(
    "The experiment also evaluated a single object category. "
    "Consequently, matched-object classification accuracy cannot "
    "establish the ability to distinguish pedestrians from other "
    "object categories in a multiclass setting."
)

add_paragraph(
    "Finally, the experiment did not include an independently "
    "controlled architecture or optimizer ablation. Performance "
    "differences should not be attributed causally to individual "
    "components without further experimentation."
)

# Identify reasonable follow-up experiments linked to observed model limitations.
# These recommendations are future work rather than claims about completed project results.
add_heading("10. Future Work")

add_paragraph(
    "Future experiments could include more independent real-world "
    "nighttime recordings, additional camera viewpoints, and "
    "external test datasets. Robustness could also be assessed "
    "through multiple random seeds, controlled augmentation "
    "experiments, and comparisons across different RF-DETR model sizes."
)

add_paragraph(
    "Additional investigations may test image-enhancement methods, "
    "alternative learning rates, confidence-threshold calibration, "
    "deployment latency, and edge-device inference performance."
)

# Provide a conclusion summarizing the recorded test performance.
# The conclusion reports measured outcomes without promising real-world robustness.
add_heading("11. Conclusion")

add_paragraph(
    "This project implemented and evaluated a fine-tuned RF-DETR Nano "
    "model for pedestrian detection under varying low-light conditions. "
    "The dataset was converted into COCO format using corrected "
    "ground-truth annotations and partitioned by source frame "
    "to reduce data leakage."
)

add_paragraph(
    f"On the held-out test dataset, the model obtained "
    f"{format_number(test_row['Precision_Percent'])}% precision, "
    f"{format_number(test_row['Recall_Percent'])}% recall, "
    f"{format_number(test_row['F1_Percent'])}% F1-score, "
    f"and {format_number(test_row['mAP50_Percent'])}% mAP@0.50. "
    "The lighting-specific comparisons and error visualizations "
    "provide additional evidence of how the detector behaved "
    "under the evaluated conditions. The findings support further "
    "investigation using more independent nighttime scenes "
    "and controlled model comparisons."
)

# Define verifiable technical and scholarly sources relevant to the implementation.
# References support COCO evaluation, transformers, pretrained visual representations, and model optimization.
references = [
    "Roboflow. (n.d.). RF-DETR: Real-time detection transformer. "
    "https://github.com/roboflow/rf-detr",

    "Carion, N., Massa, F., Synnaeve, G., Usunier, N., Kirillov, A., "
    "& Zagoruyko, S. (2020). End-to-end object detection with transformers. "
    "In Computer Vision - ECCV 2020 (pp. 213-229). "
    "https://doi.org/10.1007/978-3-030-58452-8_13",

    "Oquab, M., Darcet, T., Moutakanni, T., Vo, H., Szafraniec, M., "
    "Khalidov, V., Fernandez, P., Haziza, D., Massa, F., El-Nouby, A., "
    "Assran, M., Ballas, N., Galuba, W., Howes, R., Huang, P.-Y., "
    "Li, S.-W., Misra, I., Rabbat, M., Sharma, V., ... Bojanowski, P. "
    "(2024). DINOv2: Learning robust visual features without supervision. "
    "Transactions on Machine Learning Research. "
    "https://arxiv.org/abs/2304.07193",

    "Lin, T.-Y., Maire, M., Belongie, S., Hays, J., Perona, P., "
    "Ramanan, D., Dollár, P., & Zitnick, C. L. (2014). "
    "Microsoft COCO: Common objects in context. "
    "In Computer Vision - ECCV 2014 (pp. 740-755). "
    "https://doi.org/10.1007/978-3-319-10602-1_48",

    "Loshchilov, I., & Hutter, F. (2019). "
    "Decoupled weight decay regularization. "
    "International Conference on Learning Representations. "
    "https://arxiv.org/abs/1711.05101",

    "Chen, C., Chen, Q., Xu, J., & Koltun, V. (2018). "
    "Learning to see in the dark. "
    "In Proceedings of the IEEE Conference on Computer Vision "
    "and Pattern Recognition (pp. 3291-3300). "
    "https://openaccess.thecvf.com/content_cvpr_2018/html/"
    "Chen_Learning_to_See_CVPR_2018_paper.html",

    "Roboflow. (n.d.). RF-DETR documentation. "
    "https://rfdetr.roboflow.com/"
]

# Add the reference list in an APA-style academic format.
# All sources are included as external methodological references, not as claims of project-specific results.
add_heading("References")

for reference in references:
    story.append(
        Paragraph(
            escape(reference),
            styles["AcademicReference"]
        )
    )

# Add a reproducibility appendix that identifies the essential saved experimental artifacts.
# Actual Colab code screenshots may still need to be inserted manually if specifically required by the instructor.
add_heading("Appendix A. Experimental Artifacts")

add_paragraph(
    "The implementation is documented in the associated Google Colab "
    "notebook, converted COCO dataset, saved training checkpoint, "
    "and evaluation output tables. The files below provide the primary "
    "records needed to inspect or reproduce the reported results."
)

artifact_rows = [
    ["Dataset Conversion", "cell3_coco_conversion_summary.csv"],
    ["Dataset Split Manifest", "cell3_coco_split_manifest.csv"],
    ["Fine-Tuned Weights", "checkpoint_best_total.pth"],
    ["Final Test Metrics", "cell7_rfdetr_final_test_metrics.csv"],
    ["Test Predictions", "cell7_coco_test_predictions.json"],
    ["Lighting Comparison", "cell8_lighting_performance_comparison.csv"],
    ["Error Analysis", "cell8_image_level_error_analysis.csv"]
]

add_table(
    ["Artifact", "Saved Filename"],
    artifact_rows,
    widths=[170, 245]
)

# Add a transparent note about any evidence that requires manual capture.
# This prevents the report from falsely claiming to contain actual screenshots of every executed code cell.
add_paragraph(
    "Notebook execution screenshots and code line-number evidence "
    "should be added separately if required by the submission instructions. "
    "The figures included in this PDF are generated experimental outputs, "
    "not screenshots of the Colab code editor."
)

# Build the PDF document using the complete collection of report elements.
# Page numbering and consistent margins are applied to every page automatically.
print("GENERATING FINAL RF-DETR ACADEMIC REPORT")
print("=" * 70)

document.build(
    story,
    onFirstPage=draw_page_footer,
    onLaterPages=draw_page_footer
)

# Confirm that the PDF file was successfully created.
# A nonempty PDF is required before the report is considered ready for review.
assert PDF_PATH.is_file()
assert PDF_PATH.stat().st_size > 1024

# Check the generated PDF page count and basic readability.
# PyMuPDF reads the final document and confirms that its pages contain extractable text.
import fitz

with fitz.open(PDF_PATH) as pdf_document:
    page_count = len(pdf_document)

    assert page_count > 0

    first_page_text = pdf_document[0].get_text()

    assert "RF-DETR" in first_page_text

    # Render the first page to an image for a quick visual verification.
    # This preview can reveal obvious layout issues before the file is submitted.
    first_page = pdf_document[0]

    preview_pixmap = first_page.get_pixmap(
        matrix=fitz.Matrix(1.5, 1.5),
        alpha=False
    )

    PREVIEW_PATH = REPORT_DIR / "cell9_report_cover_preview.png"

    preview_pixmap.save(
        str(PREVIEW_PATH)
    )

# Display the report's verified file information.
# The PDF is saved to the project's final_report folder in Google Drive.
print("\nFINAL RF-DETR PDF REPORT SUMMARY")
print("=" * 70)

print("PDF File:", PDF_PATH)
print("Total Pages:", page_count)
print(
    "File Size (MB):",
    round(PDF_PATH.stat().st_size / (1024 ** 2), 2)
)

print("Cover Preview:", PREVIEW_PATH)
print("Total Test Images:", int(test_row["Test_Images"]))
print("Recorded Precision (%):", format_number(test_row["Precision_Percent"]))
print("Recorded Recall (%):", format_number(test_row["Recall_Percent"]))
print("Recorded mAP@0.50 (%):", format_number(test_row["mAP50_Percent"]))

# Provide a direct Colab download option for the completed PDF.
# The report will also remain stored in Google Drive if the download is interrupted.
from google.colab import files

print("\nCELL 9 COMPLETED SUCCESSFULLY!")
print("The final RF-DETR academic report has been generated.")

files.download(
    str(PDF_PATH)
)

### 10. RF-DETR Architecture, Data Processing, and Training Compliance Verification

This section audits the implemented RF-DETR Nano system against the technical requirements of the final project. It documents the original dataset provenance, converted COCO annotations, available image processing settings, detector architecture, matching and loss components, training configuration, early stopping, and saved model checkpoints.

The audit examines the installed RF-DETR package and previously generated experimental artifacts without modifying the dataset or retraining the model. Implementation details are classified as confirmed, unverified, or different from the assignment requirements, ensuring that the final report distinguishes actual implementation evidence from intended methodology.

The results are saved as structured evidence for the revised project report and GitHub documentation.

In [ ]:

# Import Python utilities for inspecting installed package files and previous experiment records.
# This audit checks implementation evidence without changing source code, images, labels, or weights.
from pathlib import Path
from importlib.metadata import version
from datetime import datetime
import inspect
import json
import re
import sys
import pandas as pd
import torch

# Import the installed RF-DETR package to identify its implementation location.
# The original installed package files are inspected in read-only mode.
import rfdetr
from rfdetr import RFDETRNano

# Define the project, dataset, checkpoint, and evidence directories.
# All new audit files are stored separately from the original pedestrian images.
PROJECT_DIR = Path("/content/drive/MyDrive/Person_Detection")
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"
DATASET_DIR = RFDETR_DIR / "dataset"
COCO_DIR = DATASET_DIR / "rfdetr_coco"
MODEL_DIR = RFDETR_DIR / "trained_models"
RESULTS_DIR = RFDETR_DIR / "evaluation_results"
AUDIT_DIR = RESULTS_DIR / "cell10_implementation_audit"

# Create the audit folder for CSV, JSON, and implementation evidence files.
# Existing model checkpoints and earlier experimental results will remain untouched.
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Define the verified dataset and trained-checkpoint locations.
# These paths are based on the earlier COCO conversion and training activities.
MANIFEST_PATH = DATASET_DIR / "cell3_coco_split_manifest.csv"
CHECKPOINT_PATH = (
    MODEL_DIR
    / "rfdetr_nano_training"
    / "checkpoint_best_total.pth"
)

# Verify the essential dataset and model artifacts.
# The audit must not replace missing evidence with assumed values.
assert MANIFEST_PATH.is_file(), (
    f"Missing split manifest: {MANIFEST_PATH}"
)

assert CHECKPOINT_PATH.is_file(), (
    f"Missing trained checkpoint: {CHECKPOINT_PATH}"
)

# Record the true dataset provenance supplied by the researcher.
# The original images were extracted from a personally recorded pedestrian video, not downloaded from Kaggle or Roboflow.
DATASET_PROVENANCE = (
    "Original researcher-recorded pedestrian video; "
    "frames extracted and processed in earlier CO1 activities."
)

# Record the precise dataset-source compliance distinction.
# Original video data must not be falsely described as a third-party Kaggle or Roboflow download.
SOURCE_COMPLIANCE = (
    "Own-recorded dataset: acquisition is documented, but the "
    "Kaggle/Roboflow source instruction is not literally met."
)

# Load the actual source-to-COCO manifest created in Cell 3.
# It documents split membership, image locations, and pedestrian annotation counts.
manifest_df = pd.read_csv(
    MANIFEST_PATH,
    keep_default_na=False
)

# Confirm that the manifest contains the required data fields.
# These fields allow the audit to verify dataset partitioning and low-light metadata.
required_fields = [
    "Split",
    "Condition",
    "Source_Frame",
    "Pedestrian_Boxes"
]

for field in required_fields:
    assert field in manifest_df.columns, (
        f"Missing manifest column: {field}"
    )

# Calculate the actual image and source-frame counts for every dataset split.
# The measured partition sizes will be compared with the intended 70/15/15 distribution.
split_summary = (
    manifest_df
    .groupby("Split")
    .agg(
        Images=("Source_Frame", "size"),
        Source_Frames=("Source_Frame", "nunique"),
        Pedestrian_Boxes=("Pedestrian_Boxes", "sum")
    )
    .reset_index()
)

# Calculate the proportion of images assigned to each split.
# This reports actual dataset organization rather than assuming perfect percentage rounding.
total_images = len(manifest_df)

split_summary["Image_Percent"] = (
    100 * split_summary["Images"] / total_images
)

# Verify that every independent source frame belongs to exactly one partition.
# Source-level separation reduces leakage between derived lighting variants.
source_split_counts = (
    manifest_df
    .groupby("Source_Frame")["Split"]
    .nunique()
)

assert source_split_counts.eq(1).all(), (
    "Dataset leakage detected between source-frame splits."
)

# Display the recorded dataset provenance and dataset partition sizes.
# These findings will be incorporated into the final methodology and compliance discussion.
print("CELL 10 - DATASET PROVENANCE AND SPLIT AUDIT")
print("=" * 75)
print("Original Source:", DATASET_PROVENANCE)
print("Source Requirement:", SOURCE_COMPLIANCE)
print("\nDataset Split Summary:")
print(split_summary.round(2).to_string(index=False))
print("Source-Frame Leakage: NONE")

# Identify the installed RF-DETR package directory and its version.
# Inspecting local package files provides evidence about the implementation actually installed in Colab.
PACKAGE_DIR = Path(rfdetr.__file__).resolve().parent

print("\nRF-DETR INSTALLED IMPLEMENTATION")
print("=" * 75)
print("Package Version:", version("rfdetr"))
print("Package Directory:", PACKAGE_DIR)
print("Model Class:", RFDETRNano.__name__)
print("Model Class Source:", inspect.getfile(RFDETRNano))
print("PyTorch Version:", torch.__version__)

# Build a list of relevant Python source files in the installed package.
# The audit searches source code without executing internal training functions.
source_files = sorted(
    path
    for path in PACKAGE_DIR.rglob("*.py")
    if path.is_file()
)

# Define technical components that the instructor explicitly requires.
# Each keyword group represents evidence to locate, not automatic proof of full implementation compliance.
component_patterns = {
    "Transformer Architecture": [
        r"TransformerEncoder",
        r"TransformerDecoder",
        r"encoder_layers",
        r"decoder_layers"
    ],
    "DINOv2 Backbone": [
        r"dinov2",
        r"DINOv2"
    ],
    "CNN or ResNet Backbone": [
        r"\bResNet\b",
        r"resnet50",
        r"resnet101"
    ],
    "Hungarian Matcher": [
        r"HungarianMatcher",
        r"linear_sum_assignment"
    ],
    "Classification Loss": [
        r"loss_labels",
        r"loss_ce",
        r"loss_class"
    ],
    "Bounding Box L1 Loss": [
        r"\bloss_bbox\b",
        r"\bl1_loss\b"
    ],
    "Generalized IoU Loss": [
        r"generalized_box_iou",
        r"\bloss_giou\b"
    ],
    "Image Resizing": [
        r"RandomResize",
        r"\bResize\b",
        r"resolution"
    ],
    "Image Normalization": [
        r"\bNormalize\b",
        r"normalize\("
    ],
    "Image Augmentation": [
        r"RandomHorizontalFlip",
        r"RandomRotation",
        r"ColorJitter",
        r"RandomSelect"
    ],
    "Receptive Field Enhancement": [
        r"ReceptiveField",
        r"ReceptiveFieldEnhancement",
        r"RFEModule"
    ],
    "Prediction Heads": [
        r"class_embed",
        r"bbox_embed",
        r"box_head"
    ]
}

# Create a function that finds a small number of matching code lines.
# Matching a keyword shows that code exists somewhere in the library, not necessarily that the Nano model executed it.
def find_source_evidence(patterns, maximum_matches=5):

    # Accumulate short source-code excerpts with their filenames and line numbers.
    # Keeping evidence compact makes it practical to review before adding screenshots to the report.
    matches = []

    for source_path in source_files:

        # Read the installed package's Python source as ordinary text.
        # Errors in optional package files are ignored rather than causing the full audit to fail.
        try:
            lines = source_path.read_text(
                encoding="utf-8"
            ).splitlines()
        except (UnicodeError, OSError):
            continue

        # Search each source line for the requested architectural or training terms.
        # The original line number is retained so the code can be located precisely.
        for line_number, line in enumerate(lines, start=1):
            if any(
                re.search(pattern, line, re.IGNORECASE)
                for pattern in patterns
            ):
                matches.append({
                    "File": str(source_path.relative_to(PACKAGE_DIR)),
                    "Line": line_number,
                    "Code": line.strip()[:180]
                })

                if len(matches) >= maximum_matches:
                    return matches

    return matches

# Collect short evidence excerpts for each required architectural component.
# These findings will help distinguish supported implementation details from unverified modifications.
source_evidence = {}
evidence_rows = []

for component, patterns in component_patterns.items():

    # Search the installed package for references to the selected component.
    # The results are evidence candidates rather than conclusions about the exact model execution path.
    matches = find_source_evidence(patterns)

    source_evidence[component] = matches

    evidence_rows.append({
        "Component": component,
        "Matching_Source_Lines": len(matches),
        "Evidence_Status": (
            "Source reference found; runtime usage not verified"
            if matches
            else "Not located by keyword search"
        ),
        "Example_Source_File": (
            matches[0]["File"] if matches else ""
        ),
        "Example_Line": (
            matches[0]["Line"] if matches else ""
        )
    })

# Convert the architectural evidence records into a structured table.
# This table intentionally avoids treating keyword matches as proof that the Nano architecture contains every component.
evidence_df = pd.DataFrame(evidence_rows)

print("\nIMPLEMENTATION SOURCE EVIDENCE")
print("=" * 75)
print(evidence_df.to_string(index=False))

# Inspect the RF-DETR Nano class definition without instantiating another model.
# This captures the constructor's implementation and can reveal the configured model family.
try:
    nano_source = inspect.getsource(RFDETRNano)
except (OSError, TypeError):
    nano_source = "RFDETRNano source could not be retrieved."

# Save the model class source as a research-only evidence file.
# This file is not a claim of authorship and must be attributed to the official RF-DETR package.
NANO_SOURCE_TXT = AUDIT_DIR / "cell10_official_rfdetr_nano_class_excerpt.txt"

NANO_SOURCE_TXT.write_text(
    nano_source[:20000],
    encoding="utf-8"
)

# Locate saved training configuration files from the previous fine-tuning activity.
# The actual experiment settings must be distinguished from package defaults.
configuration_candidates = [
    RESULTS_DIR / "cell5_rfdetr_training_configuration.json",
    MODEL_DIR / "rfdetr_nano_training" / "cell5b_run_configuration.json"
]

# Load the first available training configuration.
# Missing configuration information is documented instead of reconstructed from memory.
training_record = {}
configuration_source = None

for path in configuration_candidates:
    if path.is_file():
        with open(path, "r", encoding="utf-8") as file:
            training_record = json.load(file)

        configuration_source = str(path)
        break

# Extract the saved training arguments from the recorded experiment metadata.
# The two prior training cells used slightly different configuration record formats.
training_args = training_record.get(
    "training_parameters",
    training_record.get("training_config", {})
)

# Display the actual optimization settings when available.
# This distinguishes configured training parameters from confirmed epoch-by-epoch behavior.
print("\nTRAINING CONFIGURATION AUDIT")
print("=" * 75)
print("Configuration Record:", configuration_source or "NOT FOUND")

for name in [
    "epochs",
    "batch_size",
    "grad_accum_steps",
    "lr",
    "optimizer",
    "lr_scheduler",
    "early_stopping",
    "early_stopping_patience",
    "best_model_metric",
    "checkpoint_interval",
    "compute_val_loss"
]:
    print(
        f"{name}:",
        training_args.get(name, "NOT RECORDED")
    )

# Identify whether any experiment explicitly recorded validation-loss early stopping.
# The official RF-DETR mAP/mAR stopping approach must not be relabeled as validation-loss stopping.
early_stopping_metric = training_args.get(
    "best_model_metric",
    "not recorded"
)

print("\nEARLY STOPPING REQUIREMENT")
print("=" * 75)
print("Configured Best Model Metric:", early_stopping_metric)
print("Assignment-Requested Monitor:", "validation loss")
print(
    "Interpretation:",
    "Validation-mAP/mAR model selection is not equivalent "
    "to validation-loss early stopping."
)

# Inspect the available model checkpoints without loading or changing their contents.
# Their presence documents saved weights, but not necessarily the number of completed training epochs.
checkpoint_files = sorted([
    path
    for path in MODEL_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".pth", ".ckpt"}
])

checkpoint_df = pd.DataFrame([
    {
        "Checkpoint": path.name,
        "Path": str(path),
        "Size_MB": round(
            path.stat().st_size / (1024 ** 2),
            2
        )
    }
    for path in checkpoint_files
])

print("\nCHECKPOINT AUDIT")
print("=" * 75)
print("Saved Checkpoint Files:", len(checkpoint_files))
print("Best Model Exists:", CHECKPOINT_PATH.is_file())

# Search for existing training history files and alternative experiment directories.
# This helps establish whether actual hyperparameter comparisons or loss histories are available.
training_files = sorted([
    path
    for path in MODEL_DIR.rglob("*")
    if path.is_file()
    and (
        path.suffix.lower() in {".csv", ".json"}
        or "tfevents" in path.name.lower()
    )
])

run_directories = sorted([
    path.name
    for path in MODEL_DIR.iterdir()
    if path.is_dir()
    and "training" in path.name.lower()
])

print("Training Output Directories:", run_directories)
print("Training Metadata/Log Files:", len(training_files))

# Build a transparent project compliance table based on the current evidence.
# Missing architecture modifications and dataset-source differences are not marked complete.
compliance_rows = [
    {
        "Requirement": "Dataset acquired from Kaggle or Roboflow",
        "Status": "Not met as written",
        "Evidence": SOURCE_COMPLIANCE
    },
    {
        "Requirement": "Bounding-box annotation format",
        "Status": "Confirmed",
        "Evidence": "COCO annotations from verified pedestrian labels"
    },
    {
        "Requirement": "70/15/15 train-validation-test split",
        "Status": "Confirmed approximately",
        "Evidence": "Source-level split recorded in the Cell 3 manifest"
    },
    {
        "Requirement": "Resizing and normalization",
        "Status": "Needs transform-path verification",
        "Evidence": "Installed source inspected; training transform settings not yet traced"
    },
    {
        "Requirement": "Data augmentation",
        "Status": "Needs transform-path verification",
        "Evidence": "Augmentation use must be verified from the actual training pipeline"
    },
    {
        "Requirement": "Official RF-DETR Nano adaptation",
        "Status": "Confirmed",
        "Evidence": "Official rfdetr library and existing trained checkpoint"
    },
    {
        "Requirement": "CNN/ResNet backbone",
        "Status": "Architecture mismatch",
        "Evidence": "Official RF-DETR Nano uses a vision transformer backbone"
    },
    {
        "Requirement": "Custom receptive-field enhancement module",
        "Status": "Not demonstrated",
        "Evidence": "No separately implemented project module has been provided"
    },
    {
        "Requirement": "Transformer and prediction heads",
        "Status": "Source audit required",
        "Evidence": "See official installed package source excerpts"
    },
    {
        "Requirement": "Hungarian matching and DETR losses",
        "Status": "Source audit required",
        "Evidence": "See official installed package source excerpts"
    },
    {
        "Requirement": "AdamW and learning-rate scheduling",
        "Status": (
            "Configured"
            if training_args.get("optimizer") == "adamw"
            else "Not confirmed"
        ),
        "Evidence": "Saved fine-tuning configuration"
    },
    {
        "Requirement": "Validation-loss early stopping",
        "Status": "Mismatch",
        "Evidence": "Official RF-DETR stopping monitors validation mAP/mAR"
    },
    {
        "Requirement": "Hyperparameter comparison experiments",
        "Status": "Not yet established",
        "Evidence": "Alternative run logs and results must be reviewed"
    },
    {
        "Requirement": "Saved model checkpoints",
        "Status": "Confirmed",
        "Evidence": str(CHECKPOINT_PATH)
    },
    {
        "Requirement": "Test accuracy, precision, recall, IoU, mAP",
        "Status": (
            "Results file exists"
            if (
                RESULTS_DIR
                / "cell7_final_test_evaluation"
                / "cell7_rfdetr_final_test_metrics.csv"
            ).is_file()
            else "Results file not found"
        ),
        "Evidence": "Cell 7 final test metrics"
    },
    {
        "Requirement": "Prediction and error visualizations",
        "Status": (
            "Results file exists"
            if (
                RESULTS_DIR
                / "cell8_error_analysis"
                / "cell8_lighting_performance_comparison.csv"
            ).is_file()
            else "Results file not found"
        ),
        "Evidence": "Cell 7 and Cell 8 figures"
    },
    {
        "Requirement": "Screenshots of implementation code",
        "Status": "Not yet demonstrated",
        "Evidence": "Actual code screenshots still required"
    },
    {
        "Requirement": "GitHub repository and README",
        "Status": "Not yet demonstrated",
        "Evidence": "Final repository publication remains to be completed"
    }
]

# Convert the compliance audit into a saved CSV table.
# This will become a checklist for correcting the final PDF report and repository submission.
compliance_df = pd.DataFrame(compliance_rows)

print("\nFINAL REQUIREMENT COMPLIANCE AUDIT")
print("=" * 75)
print(
    compliance_df[
        ["Requirement", "Status"]
    ].to_string(index=False)
)

# Save the architectural evidence, checkpoint inventory, and compliance table.
# These files can be reviewed later without rerunning model initialization or training.
EVIDENCE_CSV = AUDIT_DIR / "cell10_official_source_evidence.csv"
CHECKPOINT_CSV = AUDIT_DIR / "cell10_checkpoint_inventory.csv"
COMPLIANCE_CSV = AUDIT_DIR / "cell10_requirement_compliance.csv"
SPLIT_CSV = AUDIT_DIR / "cell10_dataset_split_summary.csv"

evidence_df.to_csv(EVIDENCE_CSV, index=False)
checkpoint_df.to_csv(CHECKPOINT_CSV, index=False)
compliance_df.to_csv(COMPLIANCE_CSV, index=False)
split_summary.to_csv(SPLIT_CSV, index=False)

# Save short code snippets with source filenames and line numbers.
# The excerpts are official third-party package evidence, not student-authored model code.
SOURCE_EXCERPTS_TXT = (
    AUDIT_DIR / "cell10_official_rfdetr_source_excerpts.txt"
)

source_text_lines = [
    "OFFICIAL RF-DETR PACKAGE SOURCE EVIDENCE",
    f"Installed Package: {PACKAGE_DIR}",
    f"Version: {version('rfdetr')}",
    "Note: Keyword matches do not prove execution by RFDETRNano.",
    ""
]

for component, matches in source_evidence.items():
    source_text_lines.append(f"=== {component} ===")

    if not matches:
        source_text_lines.append("No matching source line located.")

    for match in matches:
        source_text_lines.append(
            f"{match['File']}:{match['Line']} | {match['Code']}"
        )

    source_text_lines.append("")

SOURCE_EXCERPTS_TXT.write_text(
    "\n".join(source_text_lines),
    encoding="utf-8"
)

# Save a compact JSON record describing the dataset origin and verified experiment artifacts.
# The record will help keep the GitHub README and PDF report consistent with the actual project.
AUDIT_JSON = AUDIT_DIR / "cell10_project_implementation_record.json"

audit_record = {
    "dataset_origin": DATASET_PROVENANCE,
    "dataset_source_requirement": SOURCE_COMPLIANCE,
    "object_category": "person",
    "model": "RFDETRNano",
    "rfdetr_version": version("rfdetr"),
    "installed_package_directory": str(PACKAGE_DIR),
    "best_checkpoint": str(CHECKPOINT_PATH),
    "training_configuration_file": configuration_source,
    "training_configuration": training_args,
    "training_output_directories": run_directories,
    "dataset_split_summary": split_summary.to_dict("records"),
    "architecture_note": (
        "Official RF-DETR Nano is a real-time detection transformer. "
        "A custom ResNet backbone or separate receptive-field "
        "enhancement module has not been demonstrated."
    ),
    "early_stopping_note": (
        "Official RF-DETR early stopping is based on validation "
        "mAP or mAR rather than the assignment's requested validation loss."
    )
}

with open(AUDIT_JSON, "w", encoding="utf-8") as file:
    json.dump(
        audit_record,
        file,
        indent=2,
        default=str
    )

# Display the completed implementation audit files.
# The next stage will use these findings to decide which project corrections require additional experiments.
print("\nCELL 10 OUTPUT FILES")
print("=" * 75)
print("Compliance Checklist:", COMPLIANCE_CSV)
print("Architecture Evidence:", EVIDENCE_CSV)
print("Official Source Excerpts:", SOURCE_EXCERPTS_TXT)
print("Dataset Split Summary:", SPLIT_CSV)
print("Checkpoint Inventory:", CHECKPOINT_CSV)
print("Implementation Record:", AUDIT_JSON)
print("RF-DETR Nano Class Excerpt:", NANO_SOURCE_TXT)

# Confirm completion of the read-only implementation and requirement audit.
# No dataset files, checkpoint weights, or original package source files were modified.
print("\nCELL 10 COMPLETED SUCCESSFULLY!")

### 11. RF-DETR Hyperparameter Optimization Through Learning-Rate Experimentation

This section investigates the effect of learning-rate adjustment on RF-DETR Nano fine-tuning. The original training configuration uses a learning rate of 0.0001, while a separate experimental configuration uses a reduced learning rate of 0.00005. Other principal settings, including the maximum number of training epochs, batch size, gradient accumulation, optimizer, and learning-rate scheduler, are held constant to support a controlled comparison.

The experiment uses the same COCO training and validation partitions as the original model. Validation loss and detection metrics are recorded where supported by the installed RF-DETR framework, and model checkpoints are saved independently to prevent the original experiment from being overwritten.

The resulting training artifacts will be used to compare model convergence and validation performance. The independent test dataset remains excluded from hyperparameter selection to preserve the integrity of the final evaluation.

In [ ]:

# Import libraries for configuring and recording the RF-DETR optimization experiment.
# These utilities manage reproducibility, training artifacts, and model checkpoint verification.
from pathlib import Path
from datetime import datetime
from importlib.metadata import version
from time import perf_counter
import json
import random
import gc
import numpy as np
import pandas as pd
import torch

# Import the official RF-DETR Nano model and its training dependencies.
# The experiment uses the installed PyTorch-based framework rather than introducing a different detection model.
from rfdetr import RFDETRNano
import pytorch_lightning as pl

# Define the existing RF-DETR project, dataset, and output folders.
# The secondary experiment will use the same dataset without modifying its annotations.
PROJECT_DIR = Path("/content/drive/MyDrive/Person_Detection")
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"
DATASET_DIR = RFDETR_DIR / "dataset" / "rfdetr_coco"
MODEL_DIR = RFDETR_DIR / "trained_models"
RESULTS_DIR = RFDETR_DIR / "evaluation_results"

# Define the original and secondary training output directories.
# Different directories prevent the new learning-rate experiment from overwriting the original model.
ORIGINAL_DIR = MODEL_DIR / "rfdetr_nano_training"
EXPERIMENT_DIR = MODEL_DIR / "rfdetr_nano_lr_00005"

# Identify the original best-model checkpoint saved in the previous training activity.
# Its existence is a prerequisite for conducting a meaningful experiment comparison.
ORIGINAL_CHECKPOINT = (
    ORIGINAL_DIR / "checkpoint_best_total.pth"
)

# Confirm that the original trained checkpoint is still available.
# This experiment will preserve the checkpoint rather than modify or resume from it.
assert ORIGINAL_CHECKPOINT.is_file(), (
    f"Original trained checkpoint is missing: {ORIGINAL_CHECKPOINT}"
)

# Confirm that CUDA is available for transformer fine-tuning.
# This experiment is configured for a Google Colab GPU rather than CPU-only training.
assert torch.cuda.is_available(), (
    "CUDA GPU unavailable. Enable a GPU in Colab Runtime settings."
)

# Verify the required COCO annotation files for all dataset partitions.
# The training experiment uses train and valid, while test remains reserved.
for split in ["train", "valid", "test"]:

    # Locate the annotation file corresponding to the current dataset partition.
    # An absent annotation file indicates that the COCO export requires investigation.
    annotation_path = (
        DATASET_DIR / split / "_annotations.coco.json"
    )

    assert annotation_path.is_file(), (
        f"Missing COCO annotation file: {annotation_path}"
    )

# Verify that the training and validation partitions contain annotated pedestrian images.
# An empty dataset must not be passed to an expensive training operation.
for split in ["train", "valid"]:

    # Read the saved COCO annotation structure without editing the original JSON.
    # The verification checks for image records, pedestrian objects, and a valid category definition.
    with open(
        DATASET_DIR / split / "_annotations.coco.json",
        "r",
        encoding="utf-8"
    ) as file:
        coco_data = json.load(file)

    assert len(coco_data["images"]) > 0
    assert len(coco_data["annotations"]) > 0

    assert len(coco_data["categories"]) == 1
    assert coco_data["categories"][0]["name"] == "person"

# Prevent accidental retraining into an output folder that already contains results.
# Existing experimental checkpoints or logs must be reviewed before restarting a run.
if EXPERIMENT_DIR.exists():

    # Search recursively for files that indicate a previous experimental attempt.
    # Even an incomplete training run may contain checkpoints worth preserving.
    previous_files = [
        path
        for path in EXPERIMENT_DIR.rglob("*")
        if path.is_file()
    ]

    if previous_files:

        # Display the existing files so the training state is transparent.
        # The code deliberately stops rather than deleting valuable model artifacts.
        print("EXISTING EXPERIMENT FILES DETECTED")
        print("=" * 70)

        for path in previous_files[:30]:
            print(path.relative_to(EXPERIMENT_DIR))

        raise RuntimeError(
            "Cell 11 experiment output already contains files. "
            "Do not overwrite the experiment. "
            "Inspect its checkpoints before rerunning."
        )

# Create a dedicated output directory for the new experiment.
# This folder is separate from the original checkpoint and all earlier evaluation results.
EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Define the controlled learning-rate comparison settings.
# The learning rate is the principal changed hyperparameter between the two experiments.
SEED = 42
MAX_EPOCHS = 15
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
EVAL_BATCH_SIZE = 1

ORIGINAL_LEARNING_RATE = 0.0001
EXPERIMENT_LEARNING_RATE = 0.00005

# Keep the optimizer, scheduler, and early-stopping configuration consistent.
# Holding these settings constant makes the observed learning-rate comparison easier to interpret.
OPTIMIZER = "adamw"
SCHEDULER = "cosine"
EARLY_STOPPING_PATIENCE = 5
CHECKPOINT_INTERVAL = 5

# Set random seeds for Python, NumPy, and PyTorch.
# Consistent seeds reduce avoidable variability without guaranteeing bit-for-bit identical training.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Record the original model checkpoint size and modification timestamp.
# These details allow us to verify that the experiment did not alter the existing weights.
original_checkpoint_size = ORIGINAL_CHECKPOINT.stat().st_size
original_checkpoint_mtime = ORIGINAL_CHECKPOINT.stat().st_mtime_ns

# Define the full secondary experiment training configuration.
# The official RF-DETR pipeline handles matching, classification and localization losses, and optimization.
experiment_config = {
    "dataset_dir": str(DATASET_DIR),
    "output_dir": str(EXPERIMENT_DIR),
    "epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "lr": EXPERIMENT_LEARNING_RATE,
    "optimizer": OPTIMIZER,
    "lr_scheduler": SCHEDULER,
    "early_stopping": True,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": 0.001,
    "skip_best_epochs": 1,
    "checkpoint_interval": CHECKPOINT_INTERVAL,
    "compute_val_loss": True,
    "tensorboard": True,
    "seed": SEED,
    "device": "cuda"
}

# Save the configuration before executing the training run.
# This ensures that the selected parameters remain documented even if Colab disconnects.
CONFIG_PATH = (
    EXPERIMENT_DIR
    / "cell11_experiment_configuration.json"
)

experiment_record = {
    "experiment": "RF-DETR Nano Reduced Learning Rate",
    "experiment_type": "Controlled learning-rate comparison",
    "baseline_learning_rate": ORIGINAL_LEARNING_RATE,
    "experimental_learning_rate": EXPERIMENT_LEARNING_RATE,
    "dataset_origin": "Researcher-recorded pedestrian video",
    "dataset_directory": str(DATASET_DIR),
    "model": "RFDETRNano",
    "rfdetr_version": version("rfdetr"),
    "pytorch_version": torch.__version__,
    "pytorch_lightning_version": pl.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "original_checkpoint": str(ORIGINAL_CHECKPOINT),
    "training_arguments": experiment_config,
    "validation_loss_logged": True,
    "early_stopping_monitor": "Validation detection metric, not validation loss",
    "status": "Configured"
}

# Write the experiment metadata to the separate training output directory.
# This JSON record provides a reproducible description of the second configuration.
with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_record,
        file,
        indent=2
    )

# Display both configurations before executing the experiment.
# The comparison makes the changed learning rate and unchanged training settings explicit.
print("CELL 11 - RF-DETR LEARNING-RATE EXPERIMENT")
print("=" * 70)

print("Model:", "RF-DETR Nano")
print("GPU:", torch.cuda.get_device_name(0))
print("Original Learning Rate:", ORIGINAL_LEARNING_RATE)
print("Experimental Learning Rate:", EXPERIMENT_LEARNING_RATE)
print("Maximum Epochs:", MAX_EPOCHS)
print("Batch Size:", BATCH_SIZE)
print("Gradient Accumulation:", GRAD_ACCUM_STEPS)
print("Effective Batch Size:", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Optimizer:", OPTIMIZER)
print("Scheduler:", SCHEDULER)
print("Validation Loss Logging:", True)
print("Early Stopping Patience:", EARLY_STOPPING_PATIENCE)
print("Original Checkpoint:", ORIGINAL_CHECKPOINT)
print("New Training Directory:", EXPERIMENT_DIR)

# Initialize a separate official pretrained RF-DETR Nano model.
# The reduced-learning-rate experiment does not load the previously fine-tuned checkpoint.
print("\nINITIALIZING SECOND EXPERIMENT MODEL")
print("=" * 70)

experiment_model = RFDETRNano()

# Clear unused CPU and GPU allocations before fine-tuning.
# This can help limit memory pressure caused by prior Colab evaluations.
gc.collect()
torch.cuda.empty_cache()

# Synchronize CUDA and record the experiment start time.
# Training duration includes data loading, optimization, validation, and checkpoint operations.
torch.cuda.synchronize()
start_time = perf_counter()

# Start the second RF-DETR fine-tuning run with the reduced learning rate.
# The framework performs real training and validation using the recorded experimental configuration.
print("\nSTARTING REDUCED LEARNING-RATE FINE-TUNING")
print("=" * 70)

experiment_model.train(
    **experiment_config
)

# Synchronize the GPU after training returns.
# This ensures the reported elapsed time includes outstanding device computations.
torch.cuda.synchronize()

elapsed_seconds = (
    perf_counter() - start_time
)

# Search for the actual checkpoints created by the secondary training run.
# Only files found in the dedicated experiment directory are considered experimental weights.
experimental_checkpoints = sorted([
    path
    for path in EXPERIMENT_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".pth", ".ckpt"}
])

# Identify the validation-selected model checkpoint for the secondary experiment.
# This checkpoint will be evaluated on validation images before comparing hyperparameters.
EXPERIMENT_CHECKPOINT = (
    EXPERIMENT_DIR / "checkpoint_best_total.pth"
)

# Confirm that the original training checkpoint was not modified.
# Both the file size and its recorded modification timestamp must remain unchanged.
assert ORIGINAL_CHECKPOINT.stat().st_size == original_checkpoint_size

assert (
    ORIGINAL_CHECKPOINT.stat().st_mtime_ns
    == original_checkpoint_mtime
), (
    "The original checkpoint modification timestamp changed."
)

# Record the completed experiment status and elapsed training time.
# These measurements will later support the optimization impact section of the report.
experiment_record["status"] = "Training completed"
experiment_record["training_seconds"] = round(
    elapsed_seconds,
    2
)
experiment_record["training_minutes"] = round(
    elapsed_seconds / 60,
    2
)
experiment_record["checkpoint_count"] = len(
    experimental_checkpoints
)
experiment_record["best_checkpoint"] = (
    str(EXPERIMENT_CHECKPOINT)
    if EXPERIMENT_CHECKPOINT.is_file()
    else None
)

# Update the experiment configuration with actual completion information.
# The experiment record does not claim a best checkpoint exists unless the file is found.
with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_record,
        file,
        indent=2
    )

# Create a checkpoint inventory for the secondary training run.
# The inventory records real model artifacts rather than assuming a specific number of saved epochs.
checkpoint_df = pd.DataFrame([
    {
        "Filename": path.name,
        "Path": str(path),
        "Size_MB": round(
            path.stat().st_size / (1024 ** 2),
            2
        )
    }
    for path in experimental_checkpoints
], columns=[
    "Filename",
    "Path",
    "Size_MB"
])

# Save the checkpoint inventory to the evaluation results folder.
# These records will be useful when performing validation-based hyperparameter comparison.
CHECKPOINT_CSV = (
    RESULTS_DIR / "cell11_experiment_checkpoint_inventory.csv"
)

checkpoint_df.to_csv(
    CHECKPOINT_CSV,
    index=False
)

# Display the actual training duration and saved checkpoint inventory.
# The output allows the experimental run to be verified before model comparison.
print("\nCELL 11 EXPERIMENT RESULTS")
print("=" * 70)

print(
    "Training Duration (minutes):",
    round(elapsed_seconds / 60, 2)
)

print(
    "Saved Checkpoint Files:",
    len(experimental_checkpoints)
)

print(
    "Best Experimental Checkpoint:",
    EXPERIMENT_CHECKPOINT
    if EXPERIMENT_CHECKPOINT.is_file()
    else "NOT FOUND"
)

if not checkpoint_df.empty:
    print(
        checkpoint_df.to_string(index=False)
    )

# Require a saved validation-selected checkpoint before proceeding to model comparison.
# Training completion alone does not guarantee that the expected best-model file exists.
assert EXPERIMENT_CHECKPOINT.is_file(), (
    "The secondary training run finished without the expected "
    "checkpoint_best_total.pth file. Inspect its logs and checkpoint directory."
)

# Display the final paths and successful completion status.
# The next cell will compare both checkpoints using the same validation dataset and evaluation settings.
print("\nCELL 11 OUTPUT FILES")
print("=" * 70)
print("Experiment Configuration:", CONFIG_PATH)
print("Experiment Checkpoint Inventory:", CHECKPOINT_CSV)
print("Original Model:", ORIGINAL_CHECKPOINT)
print("Experimental Model:", EXPERIMENT_CHECKPOINT)

# Confirm completion of the controlled second training experiment.
# The original model and independent test dataset have been preserved.
print("\nCELL 11 COMPLETED SUCCESSFULLY!")

### 12. RF-DETR Hyperparameter Comparison and Validation Performance Analysis

This section compares two fine-tuned RF-DETR Nano object detection models using the same independent validation partition. The baseline configuration uses a learning rate of 0.0001, whereas the experimental configuration uses a reduced learning rate of 0.00005. The purpose is to examine whether the learning-rate adjustment is associated with measurable differences in pedestrian detection performance.

Both checkpoints are evaluated using a consistent confidence threshold, IoU threshold, and COCO evaluation protocol. Precision, recall, F1-score, mAP@0.50, mAP@0.50:0.95, and inference time are calculated from actual model predictions. The original test dataset is excluded from the comparison to prevent hyperparameter selection using held-out test results.

The measured results are presented in comparative tables and visualizations. Interpretation considers the limitations of a small validation dataset, stochastic training variation, and any unverified differences between the recorded training configurations.

In [ ]:

# Import libraries for comparing two independently trained RF-DETR checkpoints.
# This cell uses the validation dataset only and does not perform additional training.
from pathlib import Path
from time import perf_counter
import json
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from rfdetr import RFDETRNano

# Define the RF-DETR project and dataset locations.
# All comparison outputs will remain separate from the original checkpoint directories.
PROJECT_DIR = Path("/content/drive/MyDrive/Person_Detection")
RFDETR_DIR = PROJECT_DIR / "RF_DETR_Final_Project"
COCO_DIR = RFDETR_DIR / "dataset" / "rfdetr_coco"
MODEL_DIR = RFDETR_DIR / "trained_models"
RESULTS_DIR = RFDETR_DIR / "evaluation_results"

# Create folders for validation results and comparison charts.
# These operations will not modify the original model weights or COCO annotations.
COMPARISON_DIR = RESULTS_DIR / "cell12_optimization_comparison"
FIGURES_DIR = RFDETR_DIR / "figures" / "cell12_optimization_comparison"

COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Identify the two actual fine-tuned RF-DETR Nano checkpoints.
# The first uses the baseline training experiment, while the second uses the reduced learning rate.
BASELINE_CHECKPOINT = (
    MODEL_DIR
    / "rfdetr_nano_training"
    / "checkpoint_best_total.pth"
)

EXPERIMENT_CHECKPOINT = (
    MODEL_DIR
    / "rfdetr_nano_lr_00005"
    / "checkpoint_best_total.pth"
)

# Define the independent validation annotations.
# The reserved test dataset must not be used to select the better learning rate.
VALID_JSON = COCO_DIR / "valid" / "_annotations.coco.json"

# Confirm that both training experiments produced usable checkpoint files.
# If the second checkpoint is missing, Cell 11 must be completed before proceeding.
for checkpoint in [BASELINE_CHECKPOINT, EXPERIMENT_CHECKPOINT]:
    assert checkpoint.is_file(), (
        f"Required trained checkpoint is missing: {checkpoint}. "
        "Complete the relevant training experiment first."
    )

    assert checkpoint.stat().st_size > 1024, (
        f"Checkpoint appears empty: {checkpoint}"
    )

# Verify that the validation annotation file exists.
# No performance comparison can be performed without its reference bounding boxes.
assert VALID_JSON.is_file(), (
    f"Missing validation COCO annotations: {VALID_JSON}"
)

# Define the same evaluation thresholds for both models.
# COCO mAP uses confidence-ranked predictions while precision and recall use a fixed operating threshold.
CONFIDENCE_THRESHOLD = 0.25
MAP_THRESHOLD = 0.001
IOU_THRESHOLD = 0.50

# Load the validation dataset using the official COCO API.
# Each image and person annotation will be shared across both checkpoint evaluations.
coco_gt = COCO(str(VALID_JSON))
image_ids = sorted(coco_gt.getImgIds())

# Confirm that the dataset contains images and exactly one person category.
# This prevents an unrelated annotation file from entering the optimization comparison.
assert len(image_ids) > 0
assert coco_gt.getCatIds() == [1]
assert coco_gt.cats[1]["name"] == "person"

# Define a function that converts COCO bounding boxes into XYXY pixel coordinates.
# The model's predicted coordinates use this representation during IoU calculations.
def get_ground_truth_boxes(image_id):
    annotations = coco_gt.loadAnns(
        coco_gt.getAnnIds(imgIds=[int(image_id)])
    )

    boxes = []

    for annotation in annotations:
        x, y, width, height = annotation["bbox"]

        boxes.append([
            float(x),
            float(y),
            float(x + width),
            float(y + height)
        ])

    return boxes

# Define the Intersection over Union calculation for two detection rectangles.
# IoU measures the overlap between a prediction and its reference pedestrian box.
def calculate_iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    intersection = (
        max(0.0, x2 - x1)
        * max(0.0, y2 - y1)
    )

    area_a = (
        max(0.0, box_a[2] - box_a[0])
        * max(0.0, box_a[3] - box_a[1])
    )

    area_b = (
        max(0.0, box_b[2] - box_b[0])
        * max(0.0, box_b[3] - box_b[1])
    )

    union = area_a + area_b - intersection

    return (
        intersection / union
        if union > 0
        else 0.0
    )

# Define confidence-ranked, one-to-one detection matching.
# Every ground-truth pedestrian can contribute to at most one true positive.
def match_predictions(predictions, ground_truth, threshold=0.50):
    ordered = sorted(
        predictions,
        key=lambda prediction: prediction["score"],
        reverse=True
    )

    matched = set()
    true_positives = 0
    false_positives = 0
    matched_ious = []

    for prediction in ordered:
        best_index = None
        best_iou = 0.0

        # Find the highest-overlap reference pedestrian not already matched.
        # Overlapping duplicate predictions remain false positives after the first valid match.
        for index, gt_box in enumerate(ground_truth):
            if index in matched:
                continue

            overlap = calculate_iou(
                prediction["box"],
                gt_box
            )

            if overlap > best_iou:
                best_iou = overlap
                best_index = index

        # Count a correct detection only when its class and bounding box satisfy the matching criteria.
        # Otherwise the unmatched confidence-filtered prediction is counted as a false positive.
        if (
            prediction["category_id"] == 1
            and best_index is not None
            and best_iou >= threshold
        ):
            true_positives += 1
            matched.add(best_index)
            matched_ious.append(best_iou)
        else:
            false_positives += 1

    # Count all remaining reference pedestrians as false negatives.
    # These represent missed objects at the selected operating confidence threshold.
    false_negatives = len(ground_truth) - len(matched)

    return (
        true_positives,
        false_positives,
        false_negatives,
        matched_ious
    )

# Define the checkpoint evaluation procedure used for both models.
# Identical preprocessing inputs and metric calculations support a consistent validation comparison.
def evaluate_checkpoint(model_name, checkpoint_path):

    # Initialize the trained Nano detector from its own saved checkpoint.
    # The checkpoint is restored independently rather than continuing training from the other model.
    model = RFDETRNano(
        pretrain_weights=str(checkpoint_path)
    )

    # Create collections for standard COCO predictions and operating-point results.
    # The same validation image list and threshold settings will be used for both experiments.
    coco_predictions = []
    per_image_records = []
    all_matched_ious = []

    print(f"\nEVALUATING: {model_name}")
    print("=" * 70)

    for image_number, image_id in enumerate(image_ids, start=1):

        # Locate the validation image using its COCO metadata.
        # Both checkpoints receive the exact same source image pixels.
        image_info = coco_gt.loadImgs([int(image_id)])[0]

        image_path = (
            COCO_DIR / "valid" / image_info["file_name"]
        )

        assert image_path.is_file(), (
            f"Missing validation image: {image_path}"
        )

        # Load the original image in RGB format.
        # The model's prediction interface performs its supported inference preprocessing.
        with Image.open(image_path) as image:
            image_rgb = image.convert("RGB").copy()

        # Synchronize CUDA before starting the prediction timer.
        # This helps prevent unfinished prior GPU operations from distorting measured inference time.
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = perf_counter()

        # Generate confidence-ranked RF-DETR predictions for the current validation image.
        # The low cutoff retains detections needed for standard COCO Average Precision evaluation.
        with torch.inference_mode():
            detections = model.predict(
                image_rgb,
                threshold=MAP_THRESHOLD
            )

        # Synchronize again before stopping the inference timer.
        # The measured duration should include GPU work initiated during the prediction call.
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        inference_seconds = (
            perf_counter() - start_time
        )

        # Convert the prediction arrays into consistent numerical types.
        # Each box should correspond to one confidence value and one predicted class ID.
        boxes = np.asarray(
            detections.xyxy,
            dtype=float
        ).reshape(-1, 4)

        scores = np.asarray(
            detections.confidence,
            dtype=float
        ).reshape(-1)

        class_ids = np.asarray(
            detections.class_id,
            dtype=int
        ).reshape(-1)

        assert len(boxes) == len(scores) == len(class_ids)

        # Require the supported single-class label convention used by the fine-tuned detector.
        # Class index zero or COCO category one is mapped to the sole annotated person category.
        if len(class_ids):
            assert set(class_ids.tolist()).issubset({0, 1}), (
                f"Unexpected prediction classes: {set(class_ids.tolist())}"
            )

        current_predictions = []

        for box, score, class_id in zip(
            boxes,
            scores,
            class_ids
        ):

            # Reject invalid model outputs instead of silently accepting corrupted coordinates.
            # Nonfinite scores or boxes would invalidate both operating-point and COCO metrics.
            assert np.isfinite(box).all()
            assert np.isfinite(score)
            assert 0 <= score <= 1

            # Clip predictions to the actual validation image boundaries.
            # Positive-area boxes are required by the COCO detection evaluation format.
            x1 = float(np.clip(box[0], 0, image_rgb.width))
            y1 = float(np.clip(box[1], 0, image_rgb.height))
            x2 = float(np.clip(box[2], 0, image_rgb.width))
            y2 = float(np.clip(box[3], 0, image_rgb.height))

            if x2 <= x1 or y2 <= y1:
                continue

            # Store the one-class prediction with its confidence and image-space coordinates.
            # The model's single trainable category corresponds to the person label.
            prediction = {
                "box": [x1, y1, x2, y2],
                "score": float(score),
                "category_id": 1
            }

            current_predictions.append(prediction)

            # Add every retained prediction to the COCO evaluation records.
            # COCO Average Precision uses these scores to construct precision-recall curves.
            coco_predictions.append({
                "image_id": int(image_id),
                "category_id": 1,
                "bbox": [
                    x1,
                    y1,
                    x2 - x1,
                    y2 - y1
                ],
                "score": float(score)
            })

        # Retrieve the actual pedestrian reference boxes for this validation image.
        # These annotations remain identical for the two learning-rate experiments.
        gt_boxes = get_ground_truth_boxes(image_id)

        # Apply the common operating confidence threshold to calculate detection counts.
        # This threshold is not used to truncate the confidence-ranked COCO mAP inputs.
        selected_predictions = [
            prediction
            for prediction in current_predictions
            if prediction["score"] >= CONFIDENCE_THRESHOLD
        ]

        # Match the selected predictions to their reference pedestrians.
        # True positives require an IoU of at least 0.50 under one-to-one matching.
        tp, fp, fn, matched_ious = match_predictions(
            selected_predictions,
            gt_boxes,
            IOU_THRESHOLD
        )

        all_matched_ious.extend(matched_ious)

        # Record the result for each individual validation image.
        # These records make it possible to inspect changes in false positives and missed detections.
        per_image_records.append({
            "Model": model_name,
            "Image_ID": int(image_id),
            "Image": image_info["file_name"],
            "Ground_Truth_Objects": len(gt_boxes),
            "Predicted_Objects": len(selected_predictions),
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "Inference_Seconds": inference_seconds
        })

        # Display progress periodically during the validation run.
        # This provides a visible record of how many shared images were processed.
        if image_number % 15 == 0 or image_number == len(image_ids):
            print(
                f"Processed {image_number}/{len(image_ids)} validation images"
            )

    # Convert the measured image-level detection counts into a table.
    # All operating-point metrics will be derived from these actual predictions.
    per_image_df = pd.DataFrame(per_image_records)

    tp = int(per_image_df["TP"].sum())
    fp = int(per_image_df["FP"].sum())
    fn = int(per_image_df["FN"].sum())

    # Calculate precision, recall, F1-score, and detection-focused accuracy.
    # Undefined denominators return zero consistently with the earlier test-evaluation convention.
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    accuracy = (
        tp / (tp + fp + fn)
        if tp + fp + fn
        else 0.0
    )

    # Construct the detection results object for standard COCO evaluation.
    # The empty-prediction case is handled without inventing detections.
    if coco_predictions:
        coco_dt = coco_gt.loadRes(coco_predictions)
    else:
        coco_dt = COCO()
        coco_dt.dataset = {
            "images": list(coco_gt.dataset["images"]),
            "categories": list(coco_gt.dataset["categories"]),
            "annotations": []
        }
        coco_dt.createIndex()

    # Evaluate the predictions using standard COCO IoU thresholds and confidence ranking.
    # This produces mAP@0.50 and mAP@0.50:0.95 for the same validation dataset.
    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox"
    )

    evaluator.params.imgIds = image_ids
    evaluator.params.catIds = [1]

    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    # Extract the standard COCO Average Precision measurements.
    # Negative-one values indicate undefined COCO statistics and are preserved as missing measurements.
    map_50_95 = float(evaluator.stats[0])
    map_50 = float(evaluator.stats[1])

    if map_50_95 < 0:
        map_50_95 = np.nan

    if map_50 < 0:
        map_50 = np.nan

    # Store the complete validation summary for the current checkpoint.
    # A matched-IoU average is computed over actual true-positive bounding-box pairs only.
    result = {
        "Model": model_name,
        "Checkpoint": str(checkpoint_path),
        "Validation_Images": len(image_ids),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Detection_Accuracy_Percent": accuracy * 100,
        "Precision_Percent": precision * 100,
        "Recall_Percent": recall * 100,
        "F1_Percent": f1 * 100,
        "mAP50_Percent": map_50 * 100,
        "mAP50_95_Percent": map_50_95 * 100,
        "Mean_Matched_IoU": (
            float(np.mean(all_matched_ious))
            if all_matched_ious
            else np.nan
        ),
        "Mean_Inference_Seconds": float(
            per_image_df["Inference_Seconds"].mean()
        )
    }

    # Save the confidence-ranked predictions for reproducing this evaluation.
    # Each checkpoint has its own JSON file so the predictions cannot be confused.
    prediction_path = (
        COMPARISON_DIR
        / f"cell12_{model_name}_validation_predictions.json"
    )

    with open(prediction_path, "w", encoding="utf-8") as file:
        json.dump(coco_predictions, file, indent=2)

    # Save the image-level evaluation results for inspection and error analysis.
    # The original checkpoint and validation annotation files remain unchanged.
    per_image_path = (
        COMPARISON_DIR
        / f"cell12_{model_name}_per_image_results.csv"
    )

    per_image_df.to_csv(
        per_image_path,
        index=False
    )

    # Release unused model resources before loading the other checkpoint.
    # This reduces peak GPU memory use in a notebook that has previously run training.
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

# Verify that the saved training configurations document the intended learning-rate difference.
# If either record is unavailable, the model comparison is still measurable but its causal interpretation is limited.
BASELINE_CONFIG = (
    RESULTS_DIR
    / "cell5_rfdetr_training_configuration.json"
)

EXPERIMENT_CONFIG = (
    MODEL_DIR
    / "rfdetr_nano_lr_00005"
    / "cell11_experiment_configuration.json"
)

def read_training_arguments(path):

    # Load the saved training settings only when the specified file exists.
    # Missing metadata must not be replaced by an assumed configuration.
    if not path.is_file():
        return {}

    with open(path, "r", encoding="utf-8") as file:
        record = json.load(file)

    return record.get(
        "training_parameters",
        record.get(
            "training_config",
            record.get("training_arguments", {})
        )
    )

# Read the recorded baseline and experimental training settings.
# These records can identify confounding differences beyond the intended learning-rate adjustment.
baseline_settings = read_training_arguments(BASELINE_CONFIG)
experiment_settings = read_training_arguments(EXPERIMENT_CONFIG)

# Identify whether key optimization settings were documented consistently.
# Differences other than the learning rate limit a strict single-variable experimental interpretation.
comparison_keys = [
    "epochs",
    "batch_size",
    "grad_accum_steps",
    "optimizer",
    "lr_scheduler",
    "early_stopping",
    "early_stopping_patience",
    "seed"
]

configuration_differences = []

for key in comparison_keys:
    original_value = baseline_settings.get(key, "NOT RECORDED")
    experiment_value = experiment_settings.get(key, "NOT RECORDED")

    if original_value != experiment_value:
        configuration_differences.append(
            f"{key}: baseline={original_value}; experimental={experiment_value}"
        )

# Display configuration differences before the numerical comparison.
# This prevents attributing performance changes entirely to learning rate when other settings differ.
print("RF-DETR TRAINING CONFIGURATION COMPARISON")
print("=" * 75)
print("Baseline Learning Rate:", baseline_settings.get("lr", "NOT RECORDED"))
print("Experiment Learning Rate:", experiment_settings.get("lr", "NOT RECORDED"))
print("Other Configuration Differences:", configuration_differences or "None found")

# Evaluate both saved checkpoints on the same validation images.
# The test dataset is intentionally excluded from this optimization experiment.
baseline_result = evaluate_checkpoint(
    "baseline_lr_0001",
    BASELINE_CHECKPOINT
)

experiment_result = evaluate_checkpoint(
    "experiment_lr_00005",
    EXPERIMENT_CHECKPOINT
)

# Combine the measured checkpoint results into one comparison table.
# Every numerical score comes from actual validation predictions.
comparison_df = pd.DataFrame([
    baseline_result,
    experiment_result
])

# Save the complete validation comparison as a CSV file.
# The table is suitable for direct use in the revised project report.
COMPARISON_CSV = (
    COMPARISON_DIR
    / "cell12_rfdetr_validation_comparison.csv"
)

comparison_df.to_csv(
    COMPARISON_CSV,
    index=False
)

# Calculate the experimental-minus-baseline difference for each performance metric.
# Positive differences represent higher experimental scores, while negative differences represent lower scores.
metric_columns = [
    "Detection_Accuracy_Percent",
    "Precision_Percent",
    "Recall_Percent",
    "F1_Percent",
    "mAP50_Percent",
    "mAP50_95_Percent"
]

difference_records = []

for metric in metric_columns:
    difference_records.append({
        "Metric": metric,
        "Baseline": float(comparison_df.iloc[0][metric]),
        "Experiment": float(comparison_df.iloc[1][metric]),
        "Difference_Percentage_Points": (
            float(comparison_df.iloc[1][metric])
            - float(comparison_df.iloc[0][metric])
        )
    })

# Save a separate metric-difference table for the optimization analysis.
# This makes improvements and regressions easy to identify in the academic report.
difference_df = pd.DataFrame(difference_records)

DIFFERENCE_CSV = (
    COMPARISON_DIR
    / "cell12_metric_differences.csv"
)

difference_df.to_csv(
    DIFFERENCE_CSV,
    index=False
)

# Display the actual validation comparison and metric differences.
# These tables support the required discussion of the impact of optimization choices.
print("\nRF-DETR HYPERPARAMETER VALIDATION COMPARISON")
print("=" * 75)

display_columns = [
    "Model",
    "Precision_Percent",
    "Recall_Percent",
    "F1_Percent",
    "mAP50_Percent",
    "mAP50_95_Percent"
]

print(
    comparison_df[display_columns]
    .round(3)
    .to_string(index=False)
)

print("\nEXPERIMENTAL MINUS BASELINE DIFFERENCES")
print("=" * 75)
print(difference_df.round(3).to_string(index=False))

# Create a comparison chart for precision, recall, and F1-score.
# The shared percentage axis allows the two checkpoints to be compared directly.
labels = ["Precision", "Recall", "F1-score"]
columns = [
    "Precision_Percent",
    "Recall_Percent",
    "F1_Percent"
]

x = np.arange(len(labels))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    x - bar_width / 2,
    comparison_df.iloc[0][columns].astype(float).values,
    width=bar_width,
    label="Original LR 0.0001"
)

ax.bar(
    x + bar_width / 2,
    comparison_df.iloc[1][columns].astype(float).values,
    width=bar_width,
    label="Experimental LR 0.00005"
)

# Format the performance comparison using a fixed zero-to-100 percentage scale.
# This avoids visually exaggerating small differences between the two checkpoints.
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 100)
ax.set_ylabel("Validation Performance (%)")
ax.set_title("RF-DETR Learning-Rate Experiment: Precision, Recall, and F1")
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

PERFORMANCE_FIGURE = (
    FIGURES_DIR
    / "cell12_precision_recall_f1_comparison.png"
)

plt.savefig(
    PERFORMANCE_FIGURE,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# Create a separate comparison chart for COCO mAP values.
# mAP@0.50 and mAP@0.50:0.95 measure detection quality under different localization strictness.
fig, ax = plt.subplots(figsize=(9, 6))

map_labels = ["mAP@0.50", "mAP@0.50:0.95"]
map_columns = ["mAP50_Percent", "mAP50_95_Percent"]
map_x = np.arange(len(map_labels))

ax.bar(
    map_x - bar_width / 2,
    comparison_df.iloc[0][map_columns].astype(float).values,
    width=bar_width,
    label="Original LR 0.0001"
)

ax.bar(
    map_x + bar_width / 2,
    comparison_df.iloc[1][map_columns].astype(float).values,
    width=bar_width,
    label="Experimental LR 0.00005"
)

# Label the mAP chart and preserve a meaningful percentage scale.
# The chart shows measured validation AP, not held-out test results.
ax.set_xticks(map_x)
ax.set_xticklabels(map_labels)
ax.set_ylim(0, 100)
ax.set_ylabel("COCO Average Precision (%)")
ax.set_title("RF-DETR Validation mAP by Learning Rate")
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

MAP_FIGURE = (
    FIGURES_DIR
    / "cell12_validation_map_comparison.png"
)

plt.savefig(
    MAP_FIGURE,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# Identify which checkpoint has the higher observed validation mAP@0.50:0.95.
# This selection is based exclusively on validation results and does not use held-out test outcomes.
selection_metric = "mAP50_95_Percent"
valid_scores = pd.to_numeric(
    comparison_df[selection_metric],
    errors="coerce"
)

assert valid_scores.notna().all(), (
    "Validation mAP values are unavailable. "
    "Cannot select the preferred checkpoint."
)

selected_index = int(valid_scores.idxmax())
selected_row = comparison_df.iloc[selected_index]

# Calculate the observed change in validation mAP.
# This comparison is descriptive and does not establish statistical significance from one run per configuration.
map_difference = (
    float(experiment_result["mAP50_95_Percent"])
    - float(baseline_result["mAP50_95_Percent"])
)

# Construct a report-ready optimization discussion using the measured results.
# Any differences in saved training settings are explicitly disclosed as experimental limitations.
discussion = f"""
RF-DETR NANO - HYPERPARAMETER OPTIMIZATION ANALYSIS

The original model and reduced-learning-rate model were compared using
{len(image_ids)} validation images and the same detection evaluation
procedure.

The original model achieved:
Precision: {baseline_result['Precision_Percent']:.2f}%
Recall: {baseline_result['Recall_Percent']:.2f}%
F1-score: {baseline_result['F1_Percent']:.2f}%
mAP@0.50: {baseline_result['mAP50_Percent']:.2f}%
mAP@0.50:0.95: {baseline_result['mAP50_95_Percent']:.2f}%

The reduced-learning-rate model achieved:
Precision: {experiment_result['Precision_Percent']:.2f}%
Recall: {experiment_result['Recall_Percent']:.2f}%
F1-score: {experiment_result['F1_Percent']:.2f}%
mAP@0.50: {experiment_result['mAP50_Percent']:.2f}%
mAP@0.50:0.95: {experiment_result['mAP50_95_Percent']:.2f}%

The experimental model differed from the original model by
{map_difference:+.2f} percentage points in validation mAP@0.50:0.95.

The checkpoint with the higher measured validation mAP@0.50:0.95
was {selected_row['Model']}.

Recorded differences in other training settings:
{configuration_differences if configuration_differences else 'None identified in the available configuration records.'}

This is a comparison of two actual training runs. Because each
configuration was trained only once, observed differences should not
be interpreted as proof of a statistically significant or generally
optimal learning rate.

The experiment used validation data for comparison. The held-out
test dataset was not evaluated by this cell.
""".strip()

# Save and display the measured optimization discussion.
# This text can be incorporated into the revised PDF after checking the recorded training configurations.
DISCUSSION_PATH = (
    COMPARISON_DIR
    / "cell12_optimization_analysis.txt"
)

DISCUSSION_PATH.write_text(
    discussion,
    encoding="utf-8"
)

print("\nFINAL RF-DETR OPTIMIZATION DISCUSSION")
print("=" * 75)
print(discussion)

# Save the selected checkpoint as a metadata record without changing either trained model.
# Subsequent work can reference this decision while preserving both original and experimental weights.
SELECTION_JSON = (
    COMPARISON_DIR
    / "cell12_validation_model_selection.json"
)

with open(SELECTION_JSON, "w", encoding="utf-8") as file:
    json.dump(
        {
            "selection_dataset": "validation",
            "selection_metric": selection_metric,
            "selected_model": str(selected_row["Model"]),
            "selected_checkpoint": str(selected_row["Checkpoint"]),
            "baseline_checkpoint": str(BASELINE_CHECKPOINT),
            "experimental_checkpoint": str(EXPERIMENT_CHECKPOINT),
            "configuration_differences": configuration_differences
        },
        file,
        indent=2
    )

# Confirm that all required comparison artifacts were saved successfully.
# The original model and experimental checkpoints remain unchanged.
for output_path in [
    COMPARISON_CSV,
    DIFFERENCE_CSV,
    PERFORMANCE_FIGURE,
    MAP_FIGURE,
    DISCUSSION_PATH,
    SELECTION_JSON
]:
    assert output_path.is_file(), (
        f"Missing saved comparison output: {output_path}"
    )

# Display the final file locations for the report revision.
# The next stage will integrate these genuine validation findings into the project documentation.
print("\nCELL 12 OUTPUT FILES")
print("=" * 75)
print("Validation Comparison:", COMPARISON_CSV)
print("Metric Differences:", DIFFERENCE_CSV)
print("Performance Figure:", PERFORMANCE_FIGURE)
print("mAP Figure:", MAP_FIGURE)
print("Optimization Discussion:", DISCUSSION_PATH)
print("Validation Model Selection:", SELECTION_JSON)
print("Selected Model:", selected_row["Model"])
print("Selected Checkpoint:", selected_row["Checkpoint"])

# Confirm successful completion of the two-model validation comparison.
# No additional training or held-out test evaluation was performed in this cell.
print("\nCELL 12 COMPLETED SUCCESSFULLY!")